# ESAP RSSD v2.9 - SBAD-constrained scalable sensor-directed response surface sampling

This notebook implements a clean, scalable modernization of the four-stage spatial response surface sampling (SRS/RSSD) workflow in Lesch (2005):

1. standardize sensor variables and form standardized, decorrelated PCA scores;
2. construct a central composite response-surface design in coded PCA space;
3. identify observations close to each theoretical target while retaining geographically separated alternatives; and
4. use reproducible coordinate exchange to maximize the exact geometric mean minimum separation distance (geoMSD), with PCA mismatch used only as a constraint and tie-breaker.

The implementation never constructs a full survey-by-survey distance matrix and never enumerates the Cartesian product of candidate pools. It is intended for projected coordinates in linear units. The default configuration runs a synthetic demonstration so that **Runtime > Run all** succeeds in a fresh Google Colab runtime. For real data, follow the numbered cell-by-cell workflow: upload, optional existing locations, preview, coordinate conversion, ID/X/Y assignment, feature/scaling selection, PCA/design controls, and optimization settings. The plain configuration cell remains available for scripted runs.

Reference: Lesch, S. M. (2005). Sensor-directed response surface sampling designs for characterizing spatial variation in soil properties. *Computers and Electronics in Agriculture*, 46, 153-179. https://doi.org/10.1016/j.compag.2004.11.004


In [ ]:
# @title Initialize ESAP RSSD v2.9 engine { display-mode: "form" }
# Core dependencies available in a standard Google Colab runtime.
from __future__ import annotations

import gc
import json
import math
import os
import platform
import sys
import time
import warnings
from collections import OrderedDict
from dataclasses import asdict, dataclass, field
from importlib import metadata as importlib_metadata
import json
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from scipy.spatial.distance import pdist, squareform
from scipy.stats import chi2, skew, spearmanr
from sklearn.decomposition import IncrementalPCA, PCA
from sklearn.preprocessing import PowerTransformer, StandardScaler

NOTEBOOK_VERSION = "2.9.0"
np.set_printoptions(precision=4, suppress=True)


# v2.9 default configuration
@dataclass
class RSSDConfig:
    # Input. The defaults make the notebook executable without external files.
    INPUT_FILE: Optional[str] = None
    EXCEL_SHEET_NAME: Any = 0
    USE_SYNTHETIC_DEMO: bool = True
    SYNTHETIC_ROWS: int = 6000

    # Required column roles. Change these for a real file.
    ID_COLUMN: str = "sample_id"
    X_COLUMN: str = "x"
    Y_COLUMN: str = "y"
    FEATURE_COLUMNS: List[str] = field(
        default_factory=lambda: ["sensor_1", "sensor_2", "sensor_3", "sensor_4"]
    )
    FEATURE_TRANSFORMS: Dict[str, str] = field(default_factory=dict)

    # Response-surface design and sample budget.
    N_SAMPLES: int = 20
    N_DESIGN_COMPONENTS: int = 2
    DESIGN_COVERAGE: float = 0.80
    CENTER_REPLICATES: int = 2
    SAMPLE_BUDGET_MODE: str = "rssd_with_support"  # rssd_with_support, ccd_exact, or balanced_target_replication
    SUPPORT_SITE_MODE: str = "legacy_inspired_auto"  # legacy_inspired_auto, manual, or none
    N_SUPPORT_SITES: int = 0

    # Statistical screening and candidate discovery.
    OUTLIER_COVERAGE: float = 0.999
    CANDIDATE_EXPLORATION_MODE: str = "adaptive"
    CANDIDATES_PER_TARGET: int = 3
    CANDIDATE_SATURATION_START_K: int = 3
    CANDIDATE_SATURATION_GROWTH_FACTOR: int = 2
    CANDIDATE_SATURATION_MAX_K: int = 64
    PC_CANDIDATE_TOLERANCE: float = 0.15
    MAX_PC_CANDIDATE_TOLERANCE: float = 1.50
    CANDIDATE_TOLERANCE_GROWTH: float = 1.5

    # Optimizer. N_OPTIMIZER_STARTS is a maximum when stable-search early stopping is enabled.
    N_OPTIMIZER_STARTS: int = 50
    MIN_OPTIMIZER_STARTS: int = 12
    OPTIMIZER_NO_IMPROVEMENT_PATIENCE: int = 12
    EARLY_STOP_ON_STABLE_SEARCH: bool = True
    VERIFY_REPRODUCIBILITY_BY_RERUN: bool = False
    OPTIMIZER_INITIALIZATION_MODE: str = "adaptive"
    MAX_OPTIMIZER_SWEEPS: int = 100
    AD_SUPPORT_MODE: str = "adaptive"
    AD_SUPPORT_START_SIZE: int = 5000
    AD_SUPPORT_GROWTH_FACTOR: int = 2
    AD_SUPPORT_MAX_SIZE: int = 40000
    AD_SUPPORT_FIXED_SIZE: int = 20000
    AD_SUPPORT_REL_TOL: float = 0.005
    AD_SUPPORT_RANK_CORRELATION_MIN: float = 0.995
    AD_SUPPORT_STABLE_STAGES_REQUIRED: int = 2
    AD_SUPPORT_RANK_PANEL_SIZE: int = 20
    AD_COVERAGE_ENVELOPE_REL_TOL: float = 0.05
    AD_DISTANCE_CACHE_MAX_MIB: int = 512
    RUN_REFERENCE_DESIGNS: bool = False
    RANDOM_SEED: int = 20250717
    GEOMSD_TIE_REL_TOL: float = 1e-6
    PCA_TIE_TOL: float = 1e-12

    # Memory behavior.
    MEMORY_MODE: str = "auto"  # full, auto, or incremental
    MAX_WORKING_MEMORY_FRACTION: float = 0.45
    INCREMENTAL_BATCH_SIZE: int = 50000
    NUMERIC_DTYPE: str = "float32"

    # Optional approximation for extreme candidate-discovery workloads.
    ALLOW_APPROXIMATE_PREFILTER: bool = False
    APPROX_PREFILTER_TRIGGER_ROWS: int = 750000
    APPROX_PREFILTER_MAX_ROWS: int = 250000
    APPROX_PREFILTER_BINS_PER_PC: int = 8

    # Diagnostics and testing.
    PLOT_MAX_POINTS: int = 50000
    SURVEY_COLOR: str = "C0"
    TARGET_COLOR: str = "C3"
    OUTLIER_COLOR: str = "C1"
    CANDIDATE_COLOR: str = "C2"
    SELECTED_COLOR: str = "C3"
    FOOTPRINT_COLOR: str = "0.5"
    SUPPORT_COLOR: str = "C4"
    DIAGNOSTIC_CHUNK_SIZE: int = 100000
    RUN_SPACING_DIAGNOSTIC: bool = True
    SPACING_DIAGNOSTIC_MAX_POINTS: int = 6000
    SPACING_DIAGNOSTIC_NEIGHBORS: int = 24
    SPACING_DIAGNOSTIC_RANDOM_PAIRS: int = 120000
    SPACING_DIAGNOSTIC_BINS: int = 16
    SPACING_TARGET_SEMIVARIANCE_FRACTION: float = 0.90
    SPACING_CAUTION_RATIO: float = 1.00
    SPACING_WARNING_RATIO: float = 0.75


cfg = RSSDConfig()

# Explicit transform examples for a real run:
# cfg.FEATURE_TRANSFORMS = {"ECa_vertical": "log", "NDVI": "yeo-johnson"}
# cfg.USE_SYNTHETIC_DEMO = False
# cfg.INPUT_FILE = "/content/my_survey.csv"  # or None for Colab upload




# @title Load internal data and PCA functions (run; no editing required) { display-mode: "form" }
def make_synthetic_survey(n: int, seed: int) -> pd.DataFrame:
    """Create correlated, spatially structured proxy variables for a runnable demonstration."""
    rng = np.random.default_rng(seed)
    x = rng.uniform(0.0, 2400.0, n)
    y = rng.uniform(0.0, 1600.0, n)
    z1 = 0.9 * np.sin(x / 310.0) + 0.7 * np.cos(y / 240.0) + rng.normal(0, 0.35, n)
    z2 = 0.75 * z1 + 0.5 * np.sin((x + y) / 420.0) + rng.normal(0, 0.3, n)
    z3 = -0.35 * z1 + 0.9 * np.cos(x / 520.0) + rng.normal(0, 0.45, n)
    z4 = 0.45 * z2 + 0.5 * z3 + rng.normal(0, 0.4, n)
    return pd.DataFrame(
        {
            "sample_id": np.arange(1, n + 1, dtype=np.int64),
            "x": x,
            "y": y,
            "sensor_1": z1,
            "sensor_2": z2,
            "sensor_3": z3,
            "sensor_4": z4,
        }
    )


def load_survey(config: RSSDConfig) -> Tuple[pd.DataFrame, str]:
    """Load CSV/XLS/XLSX input or the deterministic synthetic demonstration."""
    if config.USE_SYNTHETIC_DEMO:
        return make_synthetic_survey(config.SYNTHETIC_ROWS, config.RANDOM_SEED), "synthetic_demo"

    input_path = config.INPUT_FILE
    if input_path is None:
        try:
            from google.colab import files  # type: ignore
        except ImportError as exc:
            raise ValueError(
                "Set cfg.INPUT_FILE to a CSV/XLS/XLSX path outside Colab, or enable the synthetic demo."
            ) from exc
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one CSV or Excel survey file.")
        input_path = next(iter(uploaded))

    path = Path(input_path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        df = pd.read_csv(path)
    elif suffix in {".xls", ".xlsx", ".xlsm"}:
        try:
            df = pd.read_excel(path, sheet_name=config.EXCEL_SHEET_NAME)
        except ImportError as exc:
            raise ImportError("Excel input requires openpyxl (xlsx/xlsm) or xlrd (xls).") from exc
    else:
        raise ValueError(f"Unsupported input type {suffix!r}; use CSV, XLS, XLSX, or XLSM.")
    return df, str(path)


def validate_config(config: RSSDConfig) -> None:
    """Validate settings that determine statistical and computational behavior."""
    if config.MEMORY_MODE not in {"full", "auto", "incremental"}:
        raise ValueError("MEMORY_MODE must be 'full', 'auto', or 'incremental'.")
    if config.SAMPLE_BUDGET_MODE not in {"ccd_exact", "balanced_target_replication"}:
        raise ValueError("SAMPLE_BUDGET_MODE must be 'ccd_exact' or 'balanced_target_replication'.")
    if not (1 <= config.N_DESIGN_COMPONENTS <= 4):
        raise ValueError(
            "This version supports 1-4 design PCs. For >4 PCs, specify a hybrid or small composite design explicitly."
        )
    if config.N_DESIGN_COMPONENTS == 4:
        warnings.warn(
            "A full 4-D central composite design contains many targets; Lesch notes that hybrid or small composite designs may be preferable for 4-6 variables."
        )
    for name, value in {
        "OUTLIER_COVERAGE": config.OUTLIER_COVERAGE,
        "DESIGN_COVERAGE": config.DESIGN_COVERAGE,
    }.items():
        if not 0.0 < value < 1.0:
            raise ValueError(f"{name} must lie strictly between 0 and 1.")
    if config.CANDIDATES_PER_TARGET < 2:
        raise ValueError("CANDIDATES_PER_TARGET must be at least 2.")
    if config.N_OPTIMIZER_STARTS < 1:
        raise ValueError("N_OPTIMIZER_STARTS must be positive.")
    if not (1 <= config.MIN_OPTIMIZER_STARTS <= config.N_OPTIMIZER_STARTS):
        raise ValueError("MIN_OPTIMIZER_STARTS must be between 1 and N_OPTIMIZER_STARTS.")
    if config.OPTIMIZER_NO_IMPROVEMENT_PATIENCE < 1:
        raise ValueError("OPTIMIZER_NO_IMPROVEMENT_PATIENCE must be positive.")
    if config.SPACING_DIAGNOSTIC_MAX_POINTS < 100:
        raise ValueError("SPACING_DIAGNOSTIC_MAX_POINTS must be at least 100.")
    if config.SPACING_DIAGNOSTIC_NEIGHBORS < 1:
        raise ValueError("SPACING_DIAGNOSTIC_NEIGHBORS must be positive.")
    if config.SPACING_DIAGNOSTIC_RANDOM_PAIRS < 1000:
        raise ValueError("SPACING_DIAGNOSTIC_RANDOM_PAIRS must be at least 1000.")
    if config.SPACING_DIAGNOSTIC_BINS < 4:
        raise ValueError("SPACING_DIAGNOSTIC_BINS must be at least 4.")
    if not (0.0 < config.SPACING_TARGET_SEMIVARIANCE_FRACTION < 1.0):
        raise ValueError("SPACING_TARGET_SEMIVARIANCE_FRACTION must lie strictly between 0 and 1.")


def coordinates_look_geographic(x: np.ndarray, y: np.ndarray, x_name: str, y_name: str) -> bool:
    """Conservatively detect decimal-degree longitude/latitude coordinates."""
    xn, yn = x_name.lower(), y_name.lower()
    name_signal = ("lon" in xn or "longitude" in xn) and ("lat" in yn or "latitude" in yn)
    bounded = np.mean((x >= -180) & (x <= 180) & (y >= -90) & (y <= 90)) >= 0.99
    finite = np.isfinite(x) & np.isfinite(y)
    if not finite.any():
        return False
    xf, yf = x[finite], y[finite]
    decimal_signal = np.mean(np.abs(xf - np.round(xf)) > 1e-6) > 0.8 and np.mean(
        np.abs(yf - np.round(yf)) > 1e-6
    ) > 0.8
    typical_lonlat = (
        np.nanmedian(np.abs(xf)) >= 20
        and np.nanmedian(np.abs(yf)) >= 10
        and np.ptp(xf) <= 20
        and np.ptp(yf) <= 20
    )
    return bool(name_signal or (bounded and decimal_signal and typical_lonlat))


def validate_and_index_data(
    df: pd.DataFrame, config: RSSDConfig
) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray, Dict[str, Any]]:
    """Validate required data and return valid-row and coordinate-eligibility masks."""
    required = [config.ID_COLUMN, config.X_COLUMN, config.Y_COLUMN, *config.FEATURE_COLUMNS]
    missing_columns = [c for c in required if c not in df.columns]
    if missing_columns:
        raise KeyError(f"Missing required columns: {missing_columns}")
    if len(set(config.FEATURE_COLUMNS)) != len(config.FEATURE_COLUMNS):
        raise ValueError("FEATURE_COLUMNS contains duplicates.")
    if config.N_DESIGN_COMPONENTS > len(config.FEATURE_COLUMNS):
        raise ValueError("N_DESIGN_COMPONENTS cannot exceed the number of feature columns.")

    duplicate_id_count = int(df[config.ID_COLUMN].duplicated(keep=False).sum())
    if duplicate_id_count:
        raise ValueError(
            f"Found {duplicate_id_count} rows with duplicated IDs. IDs must be unique before RSSD selection."
        )

    numeric = df[[config.X_COLUMN, config.Y_COLUMN, *config.FEATURE_COLUMNS]].apply(
        pd.to_numeric, errors="coerce"
    )
    arr = numeric.to_numpy(dtype=np.float64, copy=False)
    finite_mask = np.isfinite(arr).all(axis=1)
    missing_nonfinite_counts = {
        c: int((~np.isfinite(numeric[c].to_numpy(dtype=np.float64))).sum())
        for c in numeric.columns
    }
    n_invalid = int((~finite_mask).sum())
    if finite_mask.sum() < max(config.N_SAMPLES, 20):
        raise ValueError("Too few complete finite rows remain for the requested design.")

    valid_features = numeric.loc[finite_mask, config.FEATURE_COLUMNS]
    zero_variance = [c for c in config.FEATURE_COLUMNS if float(valid_features[c].var(ddof=0)) <= 0.0]
    if zero_variance:
        raise ValueError(f"Zero-variance feature columns are not usable: {zero_variance}")

    x = numeric.loc[finite_mask, config.X_COLUMN].to_numpy(dtype=np.float64)
    y = numeric.loc[finite_mask, config.Y_COLUMN].to_numpy(dtype=np.float64)
    if coordinates_look_geographic(x, y, config.X_COLUMN, config.Y_COLUMN):
        raise ValueError(
            "Coordinates strongly resemble longitude/latitude in decimal degrees. Reproject to an appropriate linear-unit CRS (for example, the relevant UTM zone) before computing geoMSD."
        )

    duplicated_coordinates_all = df.duplicated([config.X_COLUMN, config.Y_COLUMN], keep=False).to_numpy()
    later_coordinate_duplicate = df.duplicated([config.X_COLUMN, config.Y_COLUMN], keep="first").to_numpy()
    coordinate_eligible = finite_mask & ~later_coordinate_duplicate

    out = df.copy()
    out["rssd_complete_finite"] = finite_mask
    out["rssd_duplicate_coordinate"] = duplicated_coordinates_all
    out["rssd_coordinate_candidate_eligible"] = coordinate_eligible
    report = {
        "input_rows": int(len(df)),
        "complete_finite_rows": int(finite_mask.sum()),
        "invalid_rows_excluded": n_invalid,
        "duplicated_id_rows": duplicate_id_count,
        "duplicated_coordinate_rows": int(duplicated_coordinates_all.sum()),
        "later_duplicate_coordinate_rows_excluded_from_candidates": int(later_coordinate_duplicate.sum()),
        "missing_or_nonfinite_by_column": missing_nonfinite_counts,
        "duplicate_coordinate_handling": (
            "All records are retained; among exact coordinate duplicates, only the first valid record is candidate-eligible."
        ),
    }
    return out, finite_mask, coordinate_eligible, report


def transform_features(
    values: np.ndarray, feature_names: Sequence[str], transforms: Mapping[str, str]
) -> Tuple[np.ndarray, Dict[str, Dict[str, Any]]]:
    """Apply explicit per-feature transformations without automatic choices."""
    transformed = np.asarray(values, dtype=np.float64).copy()
    details: Dict[str, Dict[str, Any]] = {}
    allowed = {"none", "log", "yeo-johnson"}
    unknown_keys = sorted(set(transforms) - set(feature_names))
    if unknown_keys:
        raise KeyError(f"FEATURE_TRANSFORMS contains unknown features: {unknown_keys}")
    for j, name in enumerate(feature_names):
        method = transforms.get(name, "none").lower()
        if method not in allowed:
            raise ValueError(f"Unsupported transform {method!r} for {name}; choose {sorted(allowed)}.")
        if method == "none":
            details[name] = {"method": "none"}
        elif method == "log":
            if np.any(transformed[:, j] <= 0):
                raise ValueError(f"Log transformation requires strictly positive values in {name!r}.")
            transformed[:, j] = np.log(transformed[:, j])
            details[name] = {"method": "log", "base": "natural"}
        else:
            pt = PowerTransformer(method="yeo-johnson", standardize=False)
            transformed[:, [j]] = pt.fit_transform(transformed[:, [j]])
            details[name] = {"method": "yeo-johnson", "lambda": float(pt.lambdas_[0])}
    return transformed.astype(np.float32, copy=False), details


def available_memory_bytes() -> int:
    """Return available RAM when psutil is present, otherwise use a conservative Colab fallback."""
    try:
        import psutil

        return int(psutil.virtual_memory().available)
    except ImportError:
        return 8 * 1024**3


def choose_pca_mode(n_rows: int, n_features: int, config: RSSDConfig) -> Tuple[str, Dict[str, float]]:
    """Choose full or incremental PCA from an explicit working-memory estimate."""
    # Approximate transformed, standardized, work, and score arrays plus estimator overhead.
    estimated = int(n_rows * n_features * (4 + 4 + 8 + 4) + n_rows * config.N_DESIGN_COMPONENTS * 4)
    available = available_memory_bytes()
    safe = int(config.MAX_WORKING_MEMORY_FRACTION * available)
    if config.MEMORY_MODE == "auto":
        mode = "full" if estimated <= safe else "incremental"
    else:
        mode = config.MEMORY_MODE
    return mode, {
        "estimated_working_bytes": estimated,
        "available_memory_bytes": available,
        "safe_memory_bytes": safe,
        "estimated_working_mib": estimated / 1024**2,
    }


def fit_standardized_pca(
    transformed: np.ndarray, config: RSSDConfig
) -> Tuple[np.ndarray, StandardScaler, Any, str, Dict[str, float]]:
    """Standardize features, fit PCA, and divide retained scores by sqrt(explained variance)."""
    n, f = transformed.shape
    mode, memory_report = choose_pca_mode(n, f, config)
    scaler = StandardScaler(copy=True)
    batch = max(config.INCREMENTAL_BATCH_SIZE, f + 1)

    def batch_slices() -> Iterable[Tuple[int, int]]:
        """Yield batches while merging a final batch smaller than the feature count."""
        start = 0
        while start < n:
            end = min(start + batch, n)
            if 0 < n - end < f:
                end = n
            yield start, end
            start = end

    if mode == "full":
        standardized = scaler.fit_transform(transformed).astype(np.float32, copy=False)
        estimator: Any = PCA(n_components=f, svd_solver="full", random_state=config.RANDOM_SEED)
        estimator.fit(standardized)
        retained_raw = (standardized - estimator.mean_) @ estimator.components_[: config.N_DESIGN_COMPONENTS].T
    else:
        for start, end in batch_slices():
            scaler.partial_fit(transformed[start:end])
        estimator = IncrementalPCA(n_components=f, batch_size=batch)
        for start, end in batch_slices():
            xb = scaler.transform(transformed[start:end]).astype(np.float32, copy=False)
            estimator.partial_fit(xb)
        retained_raw = np.empty((n, config.N_DESIGN_COMPONENTS), dtype=np.float32)
        for start, end in batch_slices():
            xb = scaler.transform(transformed[start:end]).astype(np.float32, copy=False)
            retained_raw[start:end] = (
                (xb - estimator.mean_) @ estimator.components_[: config.N_DESIGN_COMPONENTS].T
            )

    scales = np.sqrt(np.asarray(estimator.explained_variance_[: config.N_DESIGN_COMPONENTS]))
    if np.any(scales <= 0):
        raise ValueError("PCA produced a nonpositive explained variance in a retained component.")
    design_scores = (retained_raw / scales).astype(np.float32, copy=False)
    return design_scores, scaler, estimator, mode, memory_report


def pca_validation_table(scores: np.ndarray) -> Tuple[pd.DataFrame, np.ndarray]:
    """Return means, sample standard deviations, and the correlation matrix."""
    names = [f"PC{i + 1}" for i in range(scores.shape[1])]
    table = pd.DataFrame(
        {"mean": np.mean(scores, axis=0), "sample_sd": np.std(scores, axis=0, ddof=1)}, index=names
    )
    corr = np.corrcoef(scores, rowvar=False)
    corr = np.atleast_2d(corr)
    return table, corr


# @title Load internal spatial-design functions (run; no editing required) { display-mode: "form" }
def selected_nearest_neighbor_distances(coords: np.ndarray) -> np.ndarray:
    """Calculate each selected site's exact minimum separation distance."""
    coords = np.asarray(coords, dtype=np.float64)
    if coords.ndim != 2 or coords.shape[0] < 2:
        raise ValueError("At least two selected coordinate pairs are required.")
    distances, _ = cKDTree(coords).query(coords, k=2, workers=-1)
    return np.asarray(distances[:, 1], dtype=np.float64)


def exact_geo_msd(coords: np.ndarray) -> float:
    """Compute exp(mean(log(d_j))) for exact nearest-selected-neighbor distances d_j."""
    d = selected_nearest_neighbor_distances(coords)
    if np.any(d <= 0):
        return 0.0
    return float(np.exp(np.mean(np.log(d))))


def make_base_ccd(p: int, radius: float, center_replicates: int) -> pd.DataFrame:
    """Construct cube, axial, and replicated-center targets on common outer radius R."""
    if p > 4:
        raise ValueError("p > 4 requires a separately specified hybrid or small composite design.")
    rows: List[Dict[str, Any]] = []
    cube_level = radius / math.sqrt(p)
    for number in range(2**p):
        signs = np.array([1.0 if (number >> j) & 1 else -1.0 for j in range(p)])
        row = {"base_target_id": f"cube_{number + 1:02d}", "target_type": "cube"}
        row.update({f"target_PC{j + 1}": float(signs[j] * cube_level) for j in range(p)})
        rows.append(row)
    for axis in range(p):
        for sign, label in [(-1.0, "minus"), (1.0, "plus")]:
            values = np.zeros(p)
            values[axis] = sign * radius
            row = {
                "base_target_id": f"axial_PC{axis + 1}_{label}",
                "target_type": "axial",
            }
            row.update({f"target_PC{j + 1}": float(values[j]) for j in range(p)})
            rows.append(row)
    for c in range(center_replicates):
        row = {"base_target_id": f"center_{c + 1:02d}", "target_type": "center"}
        row.update({f"target_PC{j + 1}": 0.0 for j in range(p)})
        rows.append(row)
    return pd.DataFrame(rows)


def allocate_target_instances(
    base_targets: pd.DataFrame, n_samples: int, mode: str, p: int
) -> Tuple[pd.DataFrame, pd.Series]:
    """Create exact-CCD or deterministically balanced replicated target instances."""
    base_n = len(base_targets)
    if mode == "ccd_exact":
        counts = np.ones(base_n, dtype=int)
        if n_samples != base_n:
            print(f"ccd_exact mode uses {base_n} samples; configured N_SAMPLES={n_samples} is ignored.")
    else:
        if n_samples < base_n:
            raise ValueError(
                f"balanced_target_replication requires N_SAMPLES >= {base_n}, the base CCD size."
            )
        counts = np.full(base_n, n_samples // base_n, dtype=int)
        counts[: n_samples % base_n] += 1

    rows: List[Dict[str, Any]] = []
    target_cols = [f"target_PC{j + 1}" for j in range(p)]
    for i, base in base_targets.iterrows():
        for replicate in range(1, int(counts[i]) + 1):
            row = {
                "target_instance_id": f"T{len(rows) + 1:03d}",
                "base_target_id": base["base_target_id"],
                "target_type": base["target_type"],
                "target_replicate_number": replicate,
            }
            row.update({c: float(base[c]) for c in target_cols})
            rows.append(row)
    instances = pd.DataFrame(rows)
    replication = instances.groupby("base_target_id", sort=False).size().rename("replication_count")
    return instances, replication


def approximate_pca_prefilter(
    eligible_indices: np.ndarray,
    scores: np.ndarray,
    targets: np.ndarray,
    config: RSSDConfig,
) -> Tuple[np.ndarray, bool, Dict[str, Any]]:
    """Optional reproducible occupancy prefilter that explicitly preserves tails and target neighborhoods."""
    eligible_indices = np.asarray(eligible_indices, dtype=int)
    if (
        not config.ALLOW_APPROXIMATE_PREFILTER
        or len(eligible_indices) <= config.APPROX_PREFILTER_TRIGGER_ROWS
    ):
        return eligible_indices, False, {"reason": "disabled_or_below_trigger"}

    rng = np.random.default_rng(config.RANDOM_SEED)
    z = scores[eligible_indices]
    p = z.shape[1]
    bins = config.APPROX_PREFILTER_BINS_PER_PC
    bin_ids = np.zeros((len(z), p), dtype=np.int16)
    for j in range(p):
        edges = np.quantile(z[:, j], np.linspace(0, 1, bins + 1)[1:-1])
        bin_ids[:, j] = np.digitize(z[:, j], np.unique(edges), right=False)
    multipliers = (bins ** np.arange(p)).astype(np.int64)
    codes = (bin_ids.astype(np.int64) * multipliers).sum(axis=1)
    occupied = np.unique(codes)
    per_bin_cap = max(1, math.ceil(config.APPROX_PREFILTER_MAX_ROWS / len(occupied)))
    kept_local: List[np.ndarray] = []
    for code_value in occupied:
        members = np.flatnonzero(codes == code_value)
        if len(members) > per_bin_cap:
            members = np.sort(rng.choice(members, size=per_bin_cap, replace=False))
        kept_local.append(members)

    # Explicitly preserve univariate tails.
    tail_n = min(1000, len(z))
    tail_local = []
    for j in range(p):
        order = np.argsort(z[:, j], kind="stable")
        tail_local.extend([order[:tail_n], order[-tail_n:]])

    # Explicitly preserve every observation in each initial target neighborhood.
    tree = cKDTree(z)
    target_local: List[int] = []
    for target in np.unique(targets, axis=0):
        target_local.extend(tree.query_ball_point(target, r=config.PC_CANDIDATE_TOLERANCE))

    local = np.unique(
        np.concatenate([*kept_local, *tail_local, np.asarray(target_local, dtype=int)])
    )
    kept = eligible_indices[local]
    report = {
        "input_eligible_rows": int(len(eligible_indices)),
        "prefilter_rows": int(len(kept)),
        "occupied_pca_strata": int(len(occupied)),
        "per_stratum_cap": int(per_bin_cap),
        "target_neighborhood_rows_explicitly_preserved": int(len(np.unique(target_local))),
    }
    return kept, True, report


def candidate_pool_for_target(
    target: np.ndarray,
    pc_tree: cKDTree,
    search_scores: np.ndarray,
    search_coords: np.ndarray,
    search_global_indices: np.ndarray,
    desired: int,
    config: RSSDConfig,
) -> Tuple[np.ndarray, np.ndarray, float, int]:
    """Retain the closest site, then geographically separated sites within an expanding PC tolerance."""
    tolerance = config.PC_CANDIDATE_TOLERANCE
    local = pc_tree.query_ball_point(target, r=tolerance)
    expansions = 0
    while len(local) < desired and tolerance < config.MAX_PC_CANDIDATE_TOLERANCE:
        tolerance = min(
            config.MAX_PC_CANDIDATE_TOLERANCE,
            tolerance * config.CANDIDATE_TOLERANCE_GROWTH,
        )
        local = pc_tree.query_ball_point(target, r=tolerance)
        expansions += 1

    if len(local) < desired:
        k = min(max(desired, 1), len(search_scores))
        nearest_d, nearest_i = pc_tree.query(target, k=k)
        local = np.atleast_1d(nearest_i).astype(int).tolist()
        tolerance = max(tolerance, float(np.max(np.atleast_1d(nearest_d))))
        warnings.warn(
            f"Target required nearest-k fallback; final PCA tolerance expanded to {tolerance:.4f}."
        )
    if len(local) == 0:
        raise RuntimeError("No candidate observations were found for a response-surface target.")

    local_arr = np.asarray(local, dtype=int)
    pc_dist = np.linalg.norm(search_scores[local_arr] - target, axis=1)
    stable = np.lexsort((search_global_indices[local_arr], pc_dist))
    local_arr = local_arr[stable]
    pc_dist = pc_dist[stable]

    selected_positions = [0]
    remaining = np.ones(len(local_arr), dtype=bool)
    remaining[0] = False
    while len(selected_positions) < min(desired, len(local_arr)):
        rem_pos = np.flatnonzero(remaining)
        retained_coords = search_coords[local_arr[np.asarray(selected_positions)]]
        candidate_coords = search_coords[local_arr[rem_pos]]
        min_geo = np.full(len(rem_pos), np.inf)
        for retained_coord in retained_coords:
            min_geo = np.minimum(min_geo, np.linalg.norm(candidate_coords - retained_coord, axis=1))
        # Max geographic separation, then lower PCA distance, then lower original row index.
        order = np.lexsort(
            (
                search_global_indices[local_arr[rem_pos]],
                pc_dist[rem_pos],
                -min_geo,
            )
        )
        chosen = int(rem_pos[order[0]])
        selected_positions.append(chosen)
        remaining[chosen] = False

    positions = np.asarray(selected_positions, dtype=int)
    return (
        search_global_indices[local_arr[positions]],
        pc_dist[positions],
        float(tolerance),
        expansions,
    )


def discover_candidate_pools(
    target_instances: pd.DataFrame,
    search_indices: np.ndarray,
    scores: np.ndarray,
    coords: np.ndarray,
    config: RSSDConfig,
    pool_multiplier: int = 1,
) -> Tuple[List[np.ndarray], List[np.ndarray], pd.DataFrame, List[float]]:
    """Build target-specific candidate pools without survey-wide geographic pairwise distances."""
    p = config.N_DESIGN_COMPONENTS
    target_cols = [f"target_PC{j + 1}" for j in range(p)]
    targets = target_instances[target_cols].to_numpy(dtype=np.float64)
    search_scores = scores[search_indices]
    search_coords = coords[search_indices]
    pc_tree = cKDTree(search_scores)

    # Identical theoretical targets need enough shared alternatives to support unique field sites.
    target_keys = [tuple(row) for row in np.round(targets, 12)]
    multiplicities = {key: target_keys.count(key) for key in set(target_keys)}
    pools: List[np.ndarray] = []
    pool_distances: List[np.ndarray] = []
    tolerances: List[float] = []
    records: List[Dict[str, Any]] = []
    for t, target in enumerate(targets):
        desired = max(
            config.CANDIDATES_PER_TARGET * pool_multiplier,
            multiplicities[target_keys[t]],
        )
        indices, distances, tolerance, expansions = candidate_pool_for_target(
            target,
            pc_tree,
            search_scores,
            search_coords,
            search_indices,
            desired,
            config,
        )
        pools.append(indices)
        pool_distances.append(distances)
        tolerances.append(tolerance)
        for rank, (index, distance) in enumerate(zip(indices, distances), start=1):
            record = {
                "target_position": t,
                "target_instance_id": target_instances.iloc[t]["target_instance_id"],
                "base_target_id": target_instances.iloc[t]["base_target_id"],
                "target_type": target_instances.iloc[t]["target_type"],
                "candidate_analysis_index": int(index),
                "candidate_rank": rank,
                "pca_target_distance": float(distance),
                "final_tolerance_used": tolerance,
                "tolerance_expansions": expansions,
            }
            record.update({c: float(target_instances.iloc[t][c]) for c in target_cols})
            records.append(record)
    return pools, pool_distances, pd.DataFrame(records), tolerances


def assignment_from_costs(
    pools: Sequence[np.ndarray],
    pool_distances: Sequence[np.ndarray],
    rng: Optional[np.random.Generator] = None,
) -> Optional[np.ndarray]:
    """Use a minimum-cost bipartite assignment; random costs produce reproducible alternative starts."""
    union = np.unique(np.concatenate(pools))
    if len(union) < len(pools):
        return None
    column = {int(index): j for j, index in enumerate(union)}
    prohibitive = 1e12
    cost = np.full((len(pools), len(union)), prohibitive, dtype=np.float64)
    for i, (pool, distances) in enumerate(zip(pools, pool_distances)):
        values = distances if rng is None else rng.random(len(pool))
        for index, value in zip(pool, values):
            cost[i, column[int(index)]] = float(value)
    rows, cols = linear_sum_assignment(cost)
    if len(rows) != len(pools) or np.any(cost[rows, cols] >= prohibitive):
        return None
    selected = np.empty(len(pools), dtype=int)
    selected[rows] = union[cols]
    return selected


def matching_metrics(selected: np.ndarray, targets: np.ndarray, scores: np.ndarray) -> Tuple[float, float]:
    """Return mean and maximum Euclidean target mismatch in standardized PC units."""
    distances = np.linalg.norm(scores[selected] - targets, axis=1)
    return float(np.mean(distances)), float(np.max(distances))


def design_rank(selected: np.ndarray, targets: np.ndarray, scores: np.ndarray, coords: np.ndarray) -> Tuple[float, float, float]:
    """Return geoMSD, mean mismatch, and maximum mismatch for lexicographic comparison."""
    mean_pc, max_pc = matching_metrics(selected, targets, scores)
    return exact_geo_msd(coords[selected]), mean_pc, max_pc


def rank_is_better(
    candidate: Tuple[float, float, float],
    incumbent: Tuple[float, float, float],
    config: RSSDConfig,
) -> bool:
    """Compare valid designs: geoMSD first, then mean and maximum PCA mismatch only on ties."""
    geo_tol = config.GEOMSD_TIE_REL_TOL * max(1.0, abs(candidate[0]), abs(incumbent[0]))
    if candidate[0] > incumbent[0] + geo_tol:
        return True
    if abs(candidate[0] - incumbent[0]) <= geo_tol:
        if candidate[1] < incumbent[1] - config.PCA_TIE_TOL:
            return True
        if abs(candidate[1] - incumbent[1]) <= config.PCA_TIE_TOL:
            return candidate[2] < incumbent[2] - config.PCA_TIE_TOL
    return False



def candidate_pool_summary_table(
    target_instances: pd.DataFrame,
    pools: Sequence[np.ndarray],
    pool_distances: Sequence[np.ndarray],
    target_tolerances: Sequence[float],
    config: RSSDConfig,
) -> pd.DataFrame:
    """Summarize how broad each target-specific candidate search actually was."""
    records: List[Dict[str, Any]] = []
    for position, (pool, distances, tolerance) in enumerate(
        zip(pools, pool_distances, target_tolerances)
    ):
        distances = np.asarray(distances, dtype=float)
        row = target_instances.iloc[position]
        records.append(
            {
                "target_position": int(position),
                "target_instance_id": row["target_instance_id"],
                "base_target_id": row["base_target_id"],
                "target_type": row["target_type"],
                "candidate_pool_size": int(len(pool)),
                "unique_candidate_pool_size": int(len(np.unique(pool))),
                "minimum_pca_target_distance": float(np.min(distances)) if len(distances) else None,
                "median_pca_target_distance": float(np.median(distances)) if len(distances) else None,
                "maximum_pca_target_distance": float(np.max(distances)) if len(distances) else None,
                "final_tolerance_used": float(tolerance),
                "tolerance_ratio_to_configured_max": float(tolerance / config.MAX_PC_CANDIDATE_TOLERANCE)
                if config.MAX_PC_CANDIDATE_TOLERANCE > 0
                else None,
                "nearest_k_fallback_implied": bool(tolerance > config.MAX_PC_CANDIDATE_TOLERANCE * (1.0 + 1e-12)),
            }
        )
    return pd.DataFrame(records)


def search_sufficiency_diagnostics(
    optimizer_summary: pd.DataFrame,
    candidate_pool_report: pd.DataFrame,
    assignment_attempts: int,
    config: RSSDConfig,
) -> Dict[str, Any]:
    """Describe whether the bounded candidate/exchange search appears saturated."""
    if optimizer_summary.empty:
        return {"status": "not_evaluated", "recommendation": "No optimizer starts were recorded."}

    best_geo = float(optimizer_summary["final_geoMSD"].max())
    geo_tol = config.GEOMSD_TIE_REL_TOL * max(1.0, abs(best_geo))
    near_best = optimizer_summary["final_geoMSD"] >= best_geo - geo_tol
    best_rows = optimizer_summary.loc[near_best]
    starts_run = int(optimizer_summary["start_number"].max())
    first_best_start = int(best_rows["start_number"].iloc[0])
    last_best_start = int(best_rows["start_number"].iloc[-1])
    starts_after_last_best = starts_run - last_best_start
    stop_values = optimizer_summary.get("stop_after_this_start", pd.Series(dtype=object)).replace("", np.nan).dropna()
    stop_reason = str(stop_values.iloc[-1]) if len(stop_values) else "configured_max_starts_completed"

    min_pool = int(candidate_pool_report["candidate_pool_size"].min()) if len(candidate_pool_report) else 0
    median_pool = float(candidate_pool_report["candidate_pool_size"].median()) if len(candidate_pool_report) else 0.0
    maxed_tolerance_targets = int(
        (candidate_pool_report["tolerance_ratio_to_configured_max"] >= 0.999).sum()
    ) if len(candidate_pool_report) else 0
    fallback_targets = int(candidate_pool_report["nearest_k_fallback_implied"].sum()) if len(candidate_pool_report) else 0
    low_pool_targets = int(
        (candidate_pool_report["candidate_pool_size"] < max(3, config.CANDIDATES_PER_TARGET)).sum()
    ) if len(candidate_pool_report) else 0

    flags: List[str] = []
    if assignment_attempts > 1:
        flags.append("candidate pools had to be enlarged before a unique assignment existed")
    if fallback_targets:
        flags.append("one or more targets required nearest-k fallback beyond the configured PCA tolerance")
    if maxed_tolerance_targets:
        flags.append("one or more targets reached the configured maximum PCA tolerance")
    if low_pool_targets:
        flags.append("one or more targets retained fewer candidates than the requested pool size")

    if flags:
        status = "candidate_space_constrained"
        recommendation = "Review target tolerances, candidate counts, forced locations, and PCA feature choices before treating the search as saturated."
    elif stop_reason == "stable_no_improvement" or starts_after_last_best >= config.OPTIMIZER_NO_IMPROVEMENT_PATIENCE:
        status = "plausibly_saturated_local_search"
        recommendation = "The best design survived the configured no-improvement patience; additional starts are optional unless the field plan is high stakes."
    elif starts_run >= config.N_OPTIMIZER_STARTS and starts_after_last_best < max(3, config.OPTIMIZER_NO_IMPROVEMENT_PATIENCE // 2):
        status = "late_improvement_possible"
        recommendation = "The best design appeared late; increase the maximum starts or candidate pools if runtime allows."
    else:
        status = "review_optimizer_trace"
        recommendation = "Inspect the convergence table; more starts may be useful if geoMSD is still improving materially."

    return {
        "status": status,
        "recommendation": recommendation,
        "best_geoMSD": best_geo,
        "starts_run": starts_run,
        "configured_max_starts": int(config.N_OPTIMIZER_STARTS),
        "first_best_start": first_best_start,
        "last_best_start": last_best_start,
        "starts_after_last_best": int(starts_after_last_best),
        "near_best_start_count": int(near_best.sum()),
        "optimizer_stop_reason": stop_reason,
        "assignment_attempts": int(assignment_attempts),
        "minimum_candidate_pool_size": min_pool,
        "median_candidate_pool_size": median_pool,
        "targets_at_maximum_tolerance": maxed_tolerance_targets,
        "targets_with_nearest_k_fallback": fallback_targets,
        "targets_below_requested_pool_size": low_pool_targets,
        "constraint_flags": flags,
        "interpretation": "This is a bounded-search diagnostic, not a mathematical proof of global optimality.",
    }


def coordinate_exchange(
    start: np.ndarray,
    pools: Sequence[np.ndarray],
    targets: np.ndarray,
    scores: np.ndarray,
    coords: np.ndarray,
    config: RSSDConfig,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    """Optimize one site at a time; no Cartesian product of candidate pools is generated."""
    selected = np.asarray(start, dtype=int).copy()
    if len(np.unique(selected)) != len(selected):
        raise ValueError("Coordinate exchange requires a unique starting assignment.")
    initial_rank = design_rank(selected, targets, scores, coords)
    current_rank = initial_rank
    accepted = 0
    sweeps = 0
    for sweep in range(1, config.MAX_OPTIMIZER_SWEEPS + 1):
        accepted_this_sweep = 0
        for position in rng.permutation(len(selected)):
            used_elsewhere = set(np.delete(selected, position).tolist())
            best_index = int(selected[position])
            best_rank = current_rank
            for candidate_index in pools[position]:
                candidate_index = int(candidate_index)
                if candidate_index == selected[position] or candidate_index in used_elsewhere:
                    continue
                proposal = selected.copy()
                proposal[position] = candidate_index
                proposal_rank = design_rank(proposal, targets, scores, coords)
                if rank_is_better(proposal_rank, best_rank, config):
                    best_index, best_rank = candidate_index, proposal_rank
            if best_index != selected[position]:
                selected[position] = best_index
                current_rank = best_rank
                accepted += 1
                accepted_this_sweep += 1
        sweeps = sweep
        if accepted_this_sweep == 0:
            break
    return selected, {
        "initial_geoMSD": initial_rank[0],
        "final_geoMSD": current_rank[0],
        "accepted_swaps": accepted,
        "sweeps": sweeps,
        "final_mean_pca_target_distance": current_rank[1],
        "final_max_pca_target_distance": current_rank[2],
    }


def optimize_multiple_starts(
    minimum_distance_start: np.ndarray,
    pools: Sequence[np.ndarray],
    pool_distances: Sequence[np.ndarray],
    targets: np.ndarray,
    scores: np.ndarray,
    coords: np.ndarray,
    config: RSSDConfig,
) -> Tuple[np.ndarray, pd.DataFrame]:
    """Run minimum-mismatch and reproducible random starts with optional stable-search stopping."""
    rng = np.random.default_rng(config.RANDOM_SEED)
    summaries: List[Dict[str, Any]] = []
    best: Optional[np.ndarray] = None
    best_rank: Optional[Tuple[float, float, float]] = None
    starts_since_best = 0
    stop_reason = "configured_max_starts_completed"
    for start_number in range(1, config.N_OPTIMIZER_STARTS + 1):
        start = minimum_distance_start.copy() if start_number == 1 else assignment_from_costs(
            pools, pool_distances, rng
        )
        if start is None:
            raise RuntimeError("Unable to generate a unique random assignment from candidate pools.")
        optimized, summary = coordinate_exchange(start, pools, targets, scores, coords, config, rng)
        summary["start_number"] = start_number
        summary["start_type"] = "minimum_pca_distance" if start_number == 1 else "random_unique_assignment"
        rank = design_rank(optimized, targets, scores, coords)
        new_best = best_rank is None or rank_is_better(rank, best_rank, config)
        if new_best:
            best, best_rank = optimized.copy(), rank
            starts_since_best = 0
        else:
            starts_since_best += 1
        summary["new_best_design"] = bool(new_best)
        summary["best_so_far_geoMSD"] = float(best_rank[0]) if best_rank is not None else None
        summary["starts_since_best"] = int(starts_since_best)
        summary["stop_after_this_start"] = ""
        summaries.append(summary)
        if (
            config.EARLY_STOP_ON_STABLE_SEARCH
            and start_number >= config.MIN_OPTIMIZER_STARTS
            and starts_since_best >= config.OPTIMIZER_NO_IMPROVEMENT_PATIENCE
        ):
            stop_reason = "stable_no_improvement"
            summaries[-1]["stop_after_this_start"] = stop_reason
            break
    if best is None:
        raise RuntimeError("Optimizer produced no design.")
    summary_df = pd.DataFrame(summaries)
    summary_df["optimizer_stop_reason"] = stop_reason
    return best, summary_df


# @title Load diagnostics and acceptance tests (run; no editing required) { display-mode: "form" }
def second_order_matrix(scores: np.ndarray) -> Tuple[np.ndarray, List[str]]:
    """Build intercept, linear, squared, and pairwise-interaction columns."""
    z = np.asarray(scores, dtype=np.float64)
    n, p = z.shape
    columns = [np.ones(n)]
    names = ["Intercept"]
    for j in range(p):
        columns.append(z[:, j])
        names.append(f"PC{j + 1}")
    for j in range(p):
        columns.append(z[:, j] ** 2)
        names.append(f"PC{j + 1}^2")
    for j in range(p):
        for k in range(j + 1, p):
            columns.append(z[:, j] * z[:, k])
            names.append(f"PC{j + 1}:PC{k + 1}")
    return np.column_stack(columns), names


def regression_design_diagnostics(
    selected_scores: np.ndarray,
    population_scores: np.ndarray,
    chunk_size: int,
) -> Tuple[Dict[str, Any], np.ndarray]:
    """Calculate rank, condition, leverage, and chunked average relative prediction variance."""
    x, names = second_order_matrix(selected_scores)
    rank = int(np.linalg.matrix_rank(x))
    n_columns = x.shape[1]
    condition = float(np.linalg.cond(x))
    result: Dict[str, Any] = {
        "model_terms": names,
        "matrix_rows": int(x.shape[0]),
        "matrix_columns": int(n_columns),
        "matrix_rank": rank,
        "full_rank": rank == n_columns,
        "condition_number": condition,
    }
    if rank != n_columns:
        result.update(
            {
                "maximum_leverage": None,
                "average_relative_prediction_variance": None,
                "warning": "Rank deficient: inverse-based leverage and avePVar were not calculated.",
            }
        )
        return result, np.full(len(selected_scores), np.nan)

    xtx_inv = np.linalg.inv(x.T @ x)
    leverage = np.einsum("ij,jk,ik->i", x, xtx_inv, x)
    total = 0.0
    count = 0
    for start in range(0, len(population_scores), chunk_size):
        xp, _ = second_order_matrix(population_scores[start : start + chunk_size])
        h = np.einsum("ij,jk,ik->i", xp, xtx_inv, xp)
        total += float(np.sum(1.0 + h))
        count += len(h)
    result.update(
        {
            "maximum_leverage": float(np.max(leverage)),
            "average_relative_prediction_variance": total / count,
        }
    )
    return result, leverage



def proxy_spacing_diagnostic(
    eligible_coords: np.ndarray,
    eligible_scores: np.ndarray,
    selected_nearest_distances: np.ndarray,
    selected_geo_msd: float,
    config: RSSDConfig,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Estimate whether selected spacing is large relative to proxy spatial structure.

    The calculation samples coordinate pairs directly from supplied projected X/Y values.
    It is a pre-calibration predictor-structure screen only; it does not estimate future residuals.
    """
    if not config.RUN_SPACING_DIAGNOSTIC:
        return pd.DataFrame(), {
            "status": "disabled",
            "interpretation": "Spacing diagnostic disabled by configuration.",
        }

    coords_arr = np.asarray(eligible_coords, dtype=np.float64)
    scores_arr = np.asarray(eligible_scores, dtype=np.float64)
    selected_nn = np.asarray(selected_nearest_distances, dtype=np.float64)
    n = len(coords_arr)
    if n < 3 or len(selected_nn) < 2:
        return pd.DataFrame(), {
            "status": "not_estimable",
            "interpretation": "Too few eligible or selected points for a spacing diagnostic.",
        }

    rng = np.random.default_rng(config.RANDOM_SEED + 280)
    sample_n = min(int(config.SPACING_DIAGNOSTIC_MAX_POINTS), n)
    sample_idx = np.sort(rng.choice(n, size=sample_n, replace=False))
    sample_coords = coords_arr[sample_idx]
    sample_scores = scores_arr[sample_idx]

    left_parts: List[np.ndarray] = []
    right_parts: List[np.ndarray] = []
    if sample_n >= 2 and config.SPACING_DIAGNOSTIC_NEIGHBORS > 0:
        k = min(sample_n, int(config.SPACING_DIAGNOSTIC_NEIGHBORS) + 1)
        _, neighbor_idx = cKDTree(sample_coords).query(sample_coords, k=k, workers=-1)
        neighbor_idx = np.atleast_2d(neighbor_idx)
        anchors = np.repeat(np.arange(sample_n), neighbor_idx.shape[1] - 1)
        neighbors = neighbor_idx[:, 1:].reshape(-1)
        keep = anchors < neighbors
        left_parts.append(anchors[keep])
        right_parts.append(neighbors[keep])

    random_pairs = int(config.SPACING_DIAGNOSTIC_RANDOM_PAIRS)
    a = rng.integers(0, sample_n, size=random_pairs, endpoint=False)
    b = rng.integers(0, sample_n, size=random_pairs, endpoint=False)
    same = a == b
    while np.any(same):
        b[same] = rng.integers(0, sample_n, size=int(np.sum(same)), endpoint=False)
        same = a == b
    left_parts.append(a)
    right_parts.append(b)

    left = np.concatenate(left_parts)
    right = np.concatenate(right_parts)
    deltas_xy = sample_coords[left] - sample_coords[right]
    distances = np.sqrt(np.einsum("ij,ij->i", deltas_xy, deltas_xy))
    deltas_score = sample_scores[left] - sample_scores[right]
    semivariance = 0.5 * np.mean(deltas_score * deltas_score, axis=1)
    valid = np.isfinite(distances) & np.isfinite(semivariance) & (distances > 0)
    distances = distances[valid]
    semivariance = semivariance[valid]
    if len(distances) < config.SPACING_DIAGNOSTIC_BINS:
        return pd.DataFrame(), {
            "status": "not_estimable",
            "sampled_pairs": int(len(distances)),
            "interpretation": "Too few finite coordinate pairs for the spacing diagnostic.",
        }

    far_cutoff = float(np.quantile(distances, 0.80))
    sill = float(np.mean(semivariance[distances >= far_cutoff]))
    if not np.isfinite(sill) or sill <= 0:
        return pd.DataFrame(), {
            "status": "not_estimable",
            "sampled_pairs": int(len(distances)),
            "interpretation": "The sampled proxy semivariance did not produce a positive far-distance reference value.",
        }

    quantiles = np.linspace(0.0, 1.0, int(config.SPACING_DIAGNOSTIC_BINS) + 1)
    edges = np.unique(np.quantile(distances, quantiles))
    if len(edges) < 3:
        edges = np.linspace(float(np.min(distances)), float(np.max(distances)), int(config.SPACING_DIAGNOSTIC_BINS) + 1)
        edges = np.unique(edges)
    bin_id = np.digitize(distances, edges[1:-1], right=False)
    records: List[Dict[str, Any]] = []
    for b_id in range(len(edges) - 1):
        mask = bin_id == b_id
        if not np.any(mask):
            continue
        gamma = float(np.mean(semivariance[mask]))
        records.append(
            {
                "bin": int(b_id + 1),
                "pair_count": int(np.sum(mask)),
                "distance_min": float(np.min(distances[mask])),
                "distance_median": float(np.median(distances[mask])),
                "distance_max": float(np.max(distances[mask])),
                "mean_proxy_semivariance": gamma,
                "fraction_of_far_distance_semivariance": float(gamma / sill),
            }
        )
    table = pd.DataFrame(records)
    threshold = float(config.SPACING_TARGET_SEMIVARIANCE_FRACTION * sill)
    decorrelation_distance: Optional[float] = None
    reached = table[table["mean_proxy_semivariance"] >= threshold]
    if len(reached):
        row = reached.iloc[0]
        current_distance = float(row["distance_median"])
        current_gamma = float(row["mean_proxy_semivariance"])
        previous = table[table["distance_median"] < current_distance].tail(1)
        if len(previous) and current_gamma > float(previous.iloc[0]["mean_proxy_semivariance"]):
            prev_distance = float(previous.iloc[0]["distance_median"])
            prev_gamma = float(previous.iloc[0]["mean_proxy_semivariance"])
            weight = (threshold - prev_gamma) / (current_gamma - prev_gamma)
            decorrelation_distance = float(prev_distance + np.clip(weight, 0.0, 1.0) * (current_distance - prev_distance))
        else:
            decorrelation_distance = current_distance

    absolute_min = float(np.min(selected_nn))
    median_nn = float(np.median(selected_nn))
    mean_nn = float(np.mean(selected_nn))
    ratios = {
        "minimum_spacing_to_proxy_range_ratio": None,
        "geoMSD_to_proxy_range_ratio": None,
        "median_spacing_to_proxy_range_ratio": None,
    }
    if decorrelation_distance is not None and decorrelation_distance > 0:
        ratios = {
            "minimum_spacing_to_proxy_range_ratio": float(absolute_min / decorrelation_distance),
            "geoMSD_to_proxy_range_ratio": float(selected_geo_msd / decorrelation_distance),
            "median_spacing_to_proxy_range_ratio": float(median_nn / decorrelation_distance),
        }
        if absolute_min >= decorrelation_distance:
            status = "minimum_spacing_exceeds_proxy_range"
            recommendation = "Selected-site nearest-neighbor spacing is large relative to the sampled proxy decorrelation distance."
        elif median_nn >= decorrelation_distance and selected_geo_msd >= decorrelation_distance:
            status = "typical_spacing_exceeds_proxy_range_with_close_pair"
            recommendation = "Most selected spacing is large relative to proxy structure, but inspect the closest selected pair."
        elif selected_geo_msd >= config.SPACING_WARNING_RATIO * decorrelation_distance:
            status = "caution_spacing_near_proxy_range"
            recommendation = "Selected spacing is near, but does not clearly exceed, the sampled proxy decorrelation distance."
        else:
            status = "warning_spacing_shorter_than_proxy_range"
            recommendation = "Selected spacing is short relative to sampled proxy structure; consider larger pools, more starts, fewer targets, or a wider survey design if feasible."
    else:
        status = "proxy_range_not_reached"
        recommendation = "The sampled proxy semivariogram did not reach the configured fraction of far-distance semivariance; inspect the plot and residuals after calibration."

    summary = {
        "status": status,
        "recommendation": recommendation,
        "coordinate_topology_assumption": "none; sampled pairs use supplied projected X/Y coordinates directly",
        "sampled_points": int(sample_n),
        "sampled_pairs": int(len(distances)),
        "far_distance_semivariance_reference": sill,
        "target_semivariance_fraction": float(config.SPACING_TARGET_SEMIVARIANCE_FRACTION),
        "proxy_decorrelation_distance": decorrelation_distance,
        "selected_absolute_minimum_spacing": absolute_min,
        "selected_geoMSD": float(selected_geo_msd),
        "selected_mean_nearest_neighbor_spacing": mean_nn,
        "selected_median_nearest_neighbor_spacing": median_nn,
        **ratios,
        "interpretation": "This pre-calibration screen uses sensor-PC proxy structure to judge spacing plausibility. It does not prove future calibration residual independence; Moran's I and residual plots remain post-calibration checks.",
    }
    return table, summary


def json_ready(value: Any) -> Any:
    """Recursively convert NumPy/Pandas values to JSON-safe Python values."""
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (pd.Timestamp,)):
        return value.isoformat()
    return value


def package_versions() -> Dict[str, str]:
    """Record software versions needed to reproduce a run."""
    packages = ["numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "openpyxl"]
    versions = {"python": platform.python_version()}
    for package in packages:
        try:
            versions[package] = importlib_metadata.version(package)
        except importlib_metadata.PackageNotFoundError:
            versions[package] = "not_installed"
    return versions



def export_selected_sites_kmz(
    selected_sites: pd.DataFrame,
    config: RSSDConfig,
    source_epsg: int,
    output_path: str = "ESAP_RSSD_selected_sites.kmz",
) -> Dict[str, Any]:
    """Transform selected projected coordinates to WGS84 and write a labeled Google Earth KMZ."""
    import html
    import zipfile

    try:
        from pyproj import CRS, Transformer
    except ImportError:
        import subprocess

        print("pyproj is not present; installing the coordinate-export dependency once...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyproj"])
        from pyproj import CRS, Transformer

    if not source_epsg or int(source_epsg) <= 0:
        raise ValueError("A valid projected source EPSG code is required for KMZ export.")
    if config.X_COLUMN not in selected_sites or config.Y_COLUMN not in selected_sites:
        raise KeyError("Selected-site export does not contain the configured X/Y columns.")
    source_crs = CRS.from_epsg(int(source_epsg))
    if not source_crs.is_projected:
        raise ValueError("KMZ source EPSG must describe a projected CRS; use the decimal-degree conversion option for longitude/latitude.")
    transformer = Transformer.from_crs(source_crs, CRS.from_epsg(4326), always_xy=True)
    x = pd.to_numeric(selected_sites[config.X_COLUMN], errors="raise").to_numpy(dtype=float)
    y = pd.to_numeric(selected_sites[config.Y_COLUMN], errors="raise").to_numpy(dtype=float)
    longitude, latitude = transformer.transform(x, y)
    longitude = np.asarray(longitude, dtype=float)
    latitude = np.asarray(latitude, dtype=float)
    if not np.isfinite(longitude).all() or not np.isfinite(latitude).all():
        raise ValueError("Coordinate transformation produced nonfinite longitude/latitude values.")
    if np.any((longitude < -180) | (longitude > 180) | (latitude < -90) | (latitude > 90)):
        raise ValueError("Coordinate transformation produced values outside WGS84 bounds; verify the EPSG code.")

    id_column = config.ID_COLUMN
    placemarks: List[str] = []
    for position, (_, row) in enumerate(selected_sites.iterrows(), start=1):
        site_id = str(row.get(id_column, position))
        target_id = str(row.get("target_instance_id", ""))
        target_type = str(row.get("target_type", ""))
        label = f"{position:02d} - {site_id}"
        details = {
            "Sample order": position,
            "Sample ID": site_id,
            "Target instance": target_id,
            "Target type": target_type,
            "Target replicate": row.get("target_replicate_number", ""),
            "PCA target distance": row.get("pca_target_distance", ""),
            "Nearest selected neighbor": row.get(
                "nearest_selected_geographic_neighbor_distance", ""
            ),
            "Projected X": row.get(config.X_COLUMN, ""),
            "Projected Y": row.get(config.Y_COLUMN, ""),
            "Source EPSG": int(source_epsg),
        }
        description_rows = "".join(
            f"<tr><th align='left'>{html.escape(str(key))}</th>"
            f"<td>{html.escape(str(detail))}</td></tr>"
            for key, detail in details.items()
        )
        placemarks.append(
            "<Placemark>"
            f"<name>{html.escape(label)}</name>"
            "<styleUrl>#rssdSample</styleUrl>"
            f"<description><![CDATA[<table>{description_rows}</table>]]></description>"
            f"<ExtendedData><Data name='sample_id'><value>{html.escape(site_id)}</value></Data>"
            f"<Data name='target_instance_id'><value>{html.escape(target_id)}</value></Data></ExtendedData>"
            f"<Point><coordinates>{longitude[position - 1]:.10f},{latitude[position - 1]:.10f},0</coordinates></Point>"
            "</Placemark>"
        )

    document_name = html.escape(f"ESAP RSSD selected sites - {Path(str(output_path)).stem}")
    kml = (
        "<?xml version='1.0' encoding='UTF-8'?>"
        "<kml xmlns='http://www.opengis.net/kml/2.2'><Document>"
        f"<name>{document_name}</name>"
        "<Style id='rssdSample'><IconStyle><color>ff0000ff</color><scale>1.1</scale>"
        "<Icon><href>http://maps.google.com/mapfiles/kml/pushpin/red-pushpin.png</href></Icon>"
        "</IconStyle><LabelStyle><color>ff000000</color><scale>0.9</scale></LabelStyle></Style>"
        + "".join(placemarks)
        + "</Document></kml>"
    )
    destination = Path(output_path)
    with zipfile.ZipFile(destination, mode="w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("doc.kml", kml.encode("utf-8"))
    return {
        "created": True,
        "file": str(destination),
        "source_epsg": int(source_epsg),
        "output_crs": "EPSG:4326",
        "placemark_count": int(len(selected_sites)),
        "label_format": "sample order - sample ID",
        "longitude_min": float(np.min(longitude)),
        "longitude_max": float(np.max(longitude)),
        "latitude_min": float(np.min(latitude)),
        "latitude_max": float(np.max(latitude)),
    }



def _save_or_show_figure(fig: Any, name: str, output_dir: Optional[Path], show: bool, saved: List[str]) -> None:
    fig.tight_layout()
    if output_dir is not None:
        output_dir.mkdir(parents=True, exist_ok=True)
        path = output_dir / name
        fig.savefig(path, dpi=220, bbox_inches="tight", facecolor="white")
        saved.append(str(path))
    if show:
        plt.show()
    else:
        plt.close(fig)


def create_esap_rssd_figures(result: RSSDRunResult, output_dir: Optional[Any] = None, show: bool = True) -> List[str]:
    """Create live and bundle figures for v2.9 maps, legacy diagnostics, and SBAD diagnostics."""
    cfg_local = result.config
    output_path = Path(output_dir) if output_dir is not None else None
    saved: List[str] = []
    selected = result.selected_sites.copy()
    candidates = result.candidate_sites.copy()
    support = result.support_sequence.copy()
    coverage = result.field_coverage_distances.copy()
    diag = result.diagnostic_data or {}
    coords_all = diag.get("coords")
    scores_all = diag.get("design_scores")
    eligible = diag.get("eligible_indices")
    original_features = diag.get("original_features")
    xcol, ycol = cfg_local.X_COLUMN, cfg_local.Y_COLUMN
    p = int(cfg_local.N_DESIGN_COMPONENTS)

    # 01: full survey/support footprint and final selected sites.
    fig, ax = plt.subplots(figsize=(9, 7))
    if coords_all is not None:
        ax.scatter(coords_all[:, 0], coords_all[:, 1], s=2, alpha=0.10, color=cfg_local.FOOTPRINT_COLOR, rasterized=True, label="Survey footprint")
    if len(support):
        final_support = support[support.get("used_in_final_support_prefix", False).astype(bool)] if "used_in_final_support_prefix" in support else support
        ax.scatter(final_support["X"], final_support["Y"], s=8, alpha=0.20, color=cfg_local.SUPPORT_COLOR, label="SBAD support prefix")
    for role, marker, color, label in [("response_surface", "x", cfg_local.SELECTED_COLOR, "Response-surface sites"), ("spatial_support", "^", cfg_local.SUPPORT_COLOR, "Spatial support sites")]:
        part = selected[selected["sample_role"] == role]
        if len(part):
            ax.scatter(part[xcol], part[ycol], marker=marker, s=90 if marker == "x" else 70, color=color, edgecolor="black" if marker != "x" else None, label=label)
    if len(selected) <= 60:
        for _, row in selected.iterrows():
            ax.annotate(str(row["sample_order"]), (row[xcol], row[ycol]), xytext=(3, 3), textcoords="offset points", fontsize=7)
    ax.set_xlabel(f"{xcol} (projected units)"); ax.set_ylabel(f"{ycol} (projected units)"); ax.set_title("Final ESAP RSSD v2.9 field map")
    ax.set_aspect("equal", adjustable="box"); ax.legend(loc="best")
    _save_or_show_figure(fig, "01_final_field_map.png", output_path, show, saved)

    # 02: candidates and selected sites.
    fig, ax = plt.subplots(figsize=(9, 7))
    if len(candidates):
        unique_candidates = candidates.drop_duplicates("candidate_analysis_index")
        ax.scatter(unique_candidates[xcol], unique_candidates[ycol], s=12, alpha=0.35, color=cfg_local.CANDIDATE_COLOR, label="Candidate observations")
    ax.scatter(selected[xcol], selected[ycol], marker="x", s=80, color=cfg_local.SELECTED_COLOR, label="Final selected")
    ax.set_xlabel(f"{xcol} (projected units)"); ax.set_ylabel(f"{ycol} (projected units)"); ax.set_title("Candidate pool and final selected locations")
    ax.set_aspect("equal", adjustable="box"); ax.legend(loc="best")
    _save_or_show_figure(fig, "02_candidates_and_selected_map.png", output_path, show, saved)

    # 03: response-surface target matches in PC space.
    response = selected[selected["sample_role"] == "response_surface"].copy()
    target_cols = [f"target_PC{j + 1}" for j in range(p)]
    selected_pc_cols = [f"selected_PC{j + 1}" for j in range(p)]
    for col in target_cols + selected_pc_cols:
        if col in response:
            response[col] = pd.to_numeric(response[col], errors="coerce")
    fig, ax = plt.subplots(figsize=(8, 6 if p > 1 else 4.5))
    if p == 1:
        ax.scatter(response["target_PC1"], np.zeros(len(response)), marker="x", s=80, color=cfg_local.TARGET_COLOR, label="Targets")
        ax.scatter(response["selected_PC1"], np.zeros(len(response)), s=40, color=cfg_local.SELECTED_COLOR, label="Selected")
        for _, row in response.iterrows():
            ax.plot([row["target_PC1"], row["selected_PC1"]], [0, 0], linewidth=0.8, alpha=0.55)
        ax.set_yticks([]); ax.set_xlabel("Standardized PC1")
    else:
        ax.scatter(response["target_PC1"], response["target_PC2"], marker="x", s=80, color=cfg_local.TARGET_COLOR, label="Targets")
        ax.scatter(response["selected_PC1"], response["selected_PC2"], s=42, color=cfg_local.SELECTED_COLOR, label="Selected response-surface sites")
        for _, row in response.iterrows():
            ax.plot([row["target_PC1"], row["selected_PC1"]], [row["target_PC2"], row["selected_PC2"]], linewidth=0.8, alpha=0.55)
        ax.set_xlabel("Standardized PC1"); ax.set_ylabel("Standardized PC2"); ax.set_aspect("equal", adjustable="box")
    ax.set_title("Response-surface target matches"); ax.legend(loc="best")
    _save_or_show_figure(fig, "03_response_surface_target_matches.png", output_path, show, saved)

    # 04: selected versus whole eligible PC boxplots, restored from v2.8 style.
    if scores_all is not None and eligible is not None:
        fig, axes = plt.subplots(1, p, figsize=(4.8 * p, 5), squeeze=False)
        selected_indices = np.asarray(diag.get("selected_indices"), dtype=int)
        for j in range(p):
            axes[0, j].boxplot([scores_all[eligible, j], scores_all[selected_indices, j]], patch_artist=True)
            axes[0, j].set_xticks([1, 2], ["Whole eligible", "Selected"])
            axes[0, j].set_title(f"Standardized PC{j + 1}")
        fig.suptitle("PC distributions: whole eligible survey vs selected")
        _save_or_show_figure(fig, "04_pc_boxplots_whole_vs_selected.png", output_path, show, saved)

    # 05: original feature boxplots, restored from v2.8 style.
    if original_features is not None and eligible is not None:
        feature_names = list(diag.get("feature_columns", cfg_local.FEATURE_COLUMNS))
        n_features = len(feature_names); ncols = min(3, n_features); nrows = math.ceil(n_features / ncols)
        selected_indices = np.asarray(diag.get("selected_indices"), dtype=int)
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.2 * nrows), squeeze=False)
        for j, feature in enumerate(feature_names):
            axes.flat[j].boxplot([original_features[eligible, j], original_features[selected_indices, j]], patch_artist=True)
            axes.flat[j].set_xticks([1, 2], ["Whole eligible", "Selected"])
            axes.flat[j].set_title(str(feature))
        for ax in axes.flat[n_features:]:
            ax.set_visible(False)
        fig.suptitle("Sensor features: whole eligible survey vs selected")
        _save_or_show_figure(fig, "05_feature_boxplots_whole_vs_selected.png", output_path, show, saved)

    # 06: SBAD support-resolution diagnostics.
    ad = result.ad_support_resolution.copy()
    if len(ad):
        fig, ax1 = plt.subplots(figsize=(8.5, 5.2))
        ax1.plot(ad["support_size"], ad["SBAD_star"], marker="o", label="AD-reference SBAD*")
        ax1.plot(ad["support_size"], ad["hybrid_core_SBAD"], marker="o", label="Hybrid core SBAD")
        ax1.set_xlabel("Nested support prefix size"); ax1.set_ylabel("SBAD (projected units)")
        ax2 = ax1.twinx()
        ax2.plot(ad["support_size"], ad["core_coverage_ratio"], marker="s", linestyle="--", color="black", label="Core coverage ratio")
        ax2.axhline(1.0 + cfg_local.AD_COVERAGE_ENVELOPE_REL_TOL, linestyle=":", color="black", linewidth=1)
        ax2.set_ylabel("SBAD / SBAD*")
        lines1, labels1 = ax1.get_legend_handles_labels(); lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")
        ax1.set_title("Adaptive SBAD support-resolution diagnostics")
        _save_or_show_figure(fig, "06_sbad_support_resolution.png", output_path, show, saved)

    # 07: final field coverage distance distribution.
    if len(coverage):
        d = pd.to_numeric(coverage["nearest_final_site_distance"], errors="coerce").dropna().to_numpy(float)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.8))
        ax1.hist(d, bins=30, color=cfg_local.SUPPORT_COLOR, alpha=0.8)
        ax1.axvline(np.mean(d), linestyle="--", color="black", label=f"SBAD={np.mean(d):.3g}")
        ax1.set_xlabel("Nearest final-site distance"); ax1.set_ylabel("Support coordinates"); ax1.set_title("Field coverage distances"); ax1.legend()
        sorted_d = np.sort(d); ax2.plot(sorted_d, np.linspace(0, 1, len(sorted_d), endpoint=True))
        ax2.set_xlabel("Nearest final-site distance"); ax2.set_ylabel("Cumulative fraction"); ax2.set_title("Coverage-distance CDF")
        _save_or_show_figure(fig, "07_field_coverage_distance_distribution.png", output_path, show, saved)

    # 08: spatial support additions.
    additions = result.spatial_support_sites.copy()
    if len(additions):
        fig, ax = plt.subplots(figsize=(7.5, 4.8))
        x = additions["support_addition_order"].to_numpy(int)
        ax.plot(x - 1, additions["SBAD_before_addition"], marker="o", label="Before addition")
        ax.plot(x, additions["SBAD_after_addition"], marker="o", label="After addition")
        ax.set_xlabel("Spatial support addition step"); ax.set_ylabel("Final-design SBAD")
        ax.set_title("Sequential spatial support-site SBAD reduction"); ax.legend()
        _save_or_show_figure(fig, "08_spatial_support_sbad_reduction.png", output_path, show, saved)

    # 09: nearest selected-neighbor distances.
    if "nearest_selected_geographic_neighbor_distance" in selected:
        nn = pd.to_numeric(selected["nearest_selected_geographic_neighbor_distance"], errors="coerce").dropna().to_numpy(float)
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.hist(nn, bins=min(15, max(5, len(nn) // 2)), color=cfg_local.SELECTED_COLOR, alpha=0.85)
        ax.axvline(float(np.exp(np.mean(np.log(nn[nn > 0])))), linestyle="--", color="black", label="geoMSD")
        ax.set_xlabel("Nearest selected-neighbor distance"); ax.set_ylabel("Selected sites"); ax.set_title("Selected-site nearest-neighbor distances"); ax.legend()
        _save_or_show_figure(fig, "09_nearest_neighbor_distances.png", output_path, show, saved)

    # 10: optimizer stability traces.
    opt = result.optimizer_stability.copy()
    if len(opt):
        final_support = opt["support_size"].max()
        opt_final = opt[opt["support_size"] == final_support]
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))
        for objective, part in opt_final.groupby("objective_type"):
            ax1.plot(part["start_number"], part["final_SBAD"], marker=".", label=objective)
            ax2.plot(part["start_number"], part["final_geoMSD"], marker=".", label=objective)
        ax1.set_xlabel("Optimizer start"); ax1.set_ylabel("Final SBAD"); ax1.set_title("Optimizer SBAD trace")
        ax2.set_xlabel("Optimizer start"); ax2.set_ylabel("Final geoMSD"); ax2.set_title("Optimizer geoMSD trace")
        ax1.legend(); ax2.legend()
        _save_or_show_figure(fig, "10_optimizer_stability_traces.png", output_path, show, saved)

    # 11: proxy spatial-scale diagnostic.
    proxy = result.proxy_spatial_scale.copy()
    if len(proxy):
        fig, ax = plt.subplots(figsize=(8.5, 5.2))
        for pc_name, part in proxy.groupby("PC"):
            ax.plot(part["distance_median"], part["fraction_of_far_distance_semivariance"], marker="o", label=pc_name)
        ax.axhline(cfg_local.SPACING_TARGET_SEMIVARIANCE_FRACTION, linestyle="--", color="black", linewidth=1, label="Target fraction")
        ax.set_xlabel("Pair distance (projected units)"); ax.set_ylabel("Fraction of far-distance semivariance")
        ax.set_title("Per-PC proxy spatial-scale diagnostic")
        ax.legend(loc="best")
        _save_or_show_figure(fig, "11_proxy_spatial_scale.png", output_path, show, saved)

    return saved


def run_unit_validations(seed: int) -> None:
    """Validate exact geoMSD and standardized PCA scores on small synthetic data."""
    coords = np.array([[0.0, 0.0], [3.0, 0.0], [0.0, 4.0], [3.0, 4.0], [8.0, 1.0]])
    kd_value = exact_geo_msd(coords)
    dense = squareform(pdist(coords))
    np.fill_diagonal(dense, np.inf)
    brute_value = float(np.exp(np.mean(np.log(np.min(dense, axis=1)))))
    np.testing.assert_allclose(kd_value, brute_value, rtol=1e-13, atol=1e-13)

    rng = np.random.default_rng(seed)
    latent = rng.normal(size=(4000, 3))
    mixed = latent @ np.array([[1.0, 0.8, -0.4], [0.2, 1.4, 0.7], [-0.5, 0.1, 1.1]])
    xs = StandardScaler().fit_transform(mixed)
    estimator = PCA(n_components=3, svd_solver="full").fit(xs)
    whitened = estimator.transform(xs) / np.sqrt(estimator.explained_variance_)
    np.testing.assert_allclose(np.mean(whitened, axis=0), 0.0, atol=1e-12)
    np.testing.assert_allclose(np.std(whitened, axis=0, ddof=1), 1.0, atol=1e-12)
    np.testing.assert_allclose(np.corrcoef(whitened, rowvar=False), np.eye(3), atol=1e-12)

    diag_coords = rng.uniform(0.0, 100.0, size=(250, 2))
    diag_scores = np.column_stack([
        np.sin(diag_coords[:, 0] / 25.0),
        np.cos(diag_coords[:, 1] / 30.0),
    ])
    diag_table, diag_summary = proxy_spacing_diagnostic(
        diag_coords, diag_scores, np.array([20.0, 25.0, 30.0, 35.0]), 27.0, cfg
    )
    assert diag_summary["coordinate_topology_assumption"].startswith("none")
    assert len(diag_table) > 0
    print(f"Unit validations passed: exact geoMSD={kd_value:.12g}; PCA scores standardized/decorrelated; irregular-coordinate spacing diagnostic ran.")




# --- ESAP RSSD v2.9 overrides inserted below ---


@dataclass
class RSSDRunResult:
    config: RSSDConfig
    selected_sites: pd.DataFrame
    candidate_sites: pd.DataFrame
    target_instances: pd.DataFrame
    support_sequence: pd.DataFrame
    ad_support_resolution: pd.DataFrame
    candidate_saturation: pd.DataFrame
    optimizer_stability: pd.DataFrame
    spatial_support_sites: pd.DataFrame
    field_coverage_distances: pd.DataFrame
    proxy_spatial_scale: pd.DataFrame
    pca_validation: pd.DataFrame
    feature_skewness: pd.DataFrame
    metadata: Dict[str, Any]
    run_summary: str
    diagnostic_data: Dict[str, Any] = field(default_factory=dict)


def validate_config(config: RSSDConfig) -> None:
    if config.MEMORY_MODE not in {"full", "auto", "incremental"}:
        raise ValueError("MEMORY_MODE must be 'full', 'auto', or 'incremental'.")
    if config.SAMPLE_BUDGET_MODE not in {"rssd_with_support", "ccd_exact", "balanced_target_replication"}:
        raise ValueError("SAMPLE_BUDGET_MODE must be rssd_with_support, ccd_exact, or balanced_target_replication.")
    if config.SUPPORT_SITE_MODE not in {"legacy_inspired_auto", "manual", "none"}:
        raise ValueError("SUPPORT_SITE_MODE must be legacy_inspired_auto, manual, or none.")
    if config.AD_SUPPORT_MODE not in {"adaptive", "fixed"}:
        raise ValueError("AD_SUPPORT_MODE must be adaptive or fixed.")
    if not (1 <= config.N_DESIGN_COMPONENTS <= 4):
        raise ValueError("This version supports 1-4 design PCs.")
    if config.N_DESIGN_COMPONENTS == 4:
        warnings.warn("A full 4-D central composite design contains many targets; Lesch notes that hybrid or small composite designs may be preferable.")
    for name, value in {"OUTLIER_COVERAGE": config.OUTLIER_COVERAGE, "DESIGN_COVERAGE": config.DESIGN_COVERAGE}.items():
        if not 0.0 < value < 1.0:
            raise ValueError(f"{name} must lie strictly between 0 and 1.")
    if config.CANDIDATES_PER_TARGET < 2:
        raise ValueError("CANDIDATES_PER_TARGET must be at least 2.")
    if config.N_OPTIMIZER_STARTS < 1 or not (1 <= config.MIN_OPTIMIZER_STARTS <= config.N_OPTIMIZER_STARTS):
        raise ValueError("Optimizer start settings are inconsistent.")
    if config.AD_SUPPORT_START_SIZE < 1 or config.AD_SUPPORT_MAX_SIZE < 1 or config.AD_SUPPORT_GROWTH_FACTOR < 2:
        raise ValueError("AD support-size settings are inconsistent.")
    if not (0.0 <= config.AD_COVERAGE_ENVELOPE_REL_TOL < 1.0):
        raise ValueError("AD_COVERAGE_ENVELOPE_REL_TOL must be nonnegative and less than 1.")


def allocate_response_surface_instances(base_targets: pd.DataFrame, response_surface_count: int, p: int) -> Tuple[pd.DataFrame, pd.Series]:
    base_n = len(base_targets)
    if response_surface_count < base_n:
        raise ValueError(f"The full base CCD requires {base_n} response-surface sites; requested {response_surface_count}.")
    counts = np.ones(base_n, dtype=int)
    extras = response_surface_count - base_n
    if extras > 0:
        counts += extras // base_n
        counts[: extras % base_n] += 1
    rows: List[Dict[str, Any]] = []
    target_cols = [f"target_PC{j + 1}" for j in range(p)]
    for i, base in base_targets.iterrows():
        for replicate in range(1, int(counts[i]) + 1):
            row = {"target_instance_id": f"T{len(rows) + 1:03d}", "base_target_id": base["base_target_id"], "target_type": base["target_type"], "target_replicate_number": replicate, "sample_role": "response_surface"}
            row.update({c: float(base[c]) for c in target_cols})
            rows.append(row)
    instances = pd.DataFrame(rows)
    return instances, instances.groupby("base_target_id", sort=False).size().rename("replication_count")


def _legacy_inspired_desired_support_count(n_samples: int) -> int:
    if n_samples <= 6:
        return 1
    if n_samples <= 20:
        return 2
    return max(3, int(math.ceil(n_samples / 10.0)))


def allocate_sample_budget(base_targets: pd.DataFrame, n_samples: int, config: RSSDConfig, p: int) -> Tuple[pd.DataFrame, pd.Series, Dict[str, Any]]:
    base_n = len(base_targets)
    if config.SAMPLE_BUDGET_MODE == "ccd_exact":
        response_count, support_count = base_n, 0
        if n_samples != base_n:
            print(f"ccd_exact mode uses {base_n} response-surface samples; configured N_SAMPLES={n_samples} is ignored.")
    elif config.SAMPLE_BUDGET_MODE == "balanced_target_replication":
        response_count, support_count = n_samples, 0
    else:
        if n_samples < base_n:
            raise ValueError(f"rssd_with_support requires N_SAMPLES >= {base_n}, the base CCD size.")
        desired = 0 if config.SUPPORT_SITE_MODE == "none" else (max(0, int(config.N_SUPPORT_SITES)) if config.SUPPORT_SITE_MODE == "manual" else _legacy_inspired_desired_support_count(n_samples))
        available_extra_sites = n_samples - base_n
        support_count = min(desired, max(0, available_extra_sites))
        response_count = n_samples - support_count
        if desired > support_count:
            print(f"Spatial support sites capped at {support_count} because the full {base_n}-site response-surface core must be preserved.")
    target_instances, replication = allocate_response_surface_instances(base_targets, response_count, p)
    report = {"sample_budget_mode": config.SAMPLE_BUDGET_MODE, "support_site_mode": config.SUPPORT_SITE_MODE, "requested_total_samples": int(n_samples), "base_CCD_target_count": int(base_n), "response_surface_sites": int(response_count), "spatial_support_sites": int(support_count), "full_response_surface_core_preserved": bool(response_count >= base_n), "support_allocation_label": "legacy-inspired automatic support allocation" if config.SUPPORT_SITE_MODE == "legacy_inspired_auto" else config.SUPPORT_SITE_MODE}
    return target_instances, replication, report


def allocate_target_instances(base_targets: pd.DataFrame, n_samples: int, mode: str, p: int) -> Tuple[pd.DataFrame, pd.Series]:
    temp = RSSDConfig(N_SAMPLES=n_samples, SAMPLE_BUDGET_MODE=mode)
    if mode == "rssd_with_support":
        temp.SUPPORT_SITE_MODE = "none"
    instances, replication, _ = allocate_sample_budget(base_targets, n_samples, temp, p)
    return instances, replication


def _nearest_observed_coordinate_to_center(coords: np.ndarray, ids: np.ndarray, indices: np.ndarray, center: np.ndarray) -> int:
    d2 = np.einsum("ij,ij->i", coords[indices] - center, coords[indices] - center)
    best = np.flatnonzero(d2 == np.min(d2))
    return int(indices[int(best[np.argmin(ids[indices[best]])])])


def _leaf_bbox(coords: np.ndarray, indices: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    subset = coords[indices]
    lo = np.min(subset, axis=0); hi = np.max(subset, axis=0); span = hi - lo
    return lo, hi, span, float(np.sqrt(np.sum(span * span)))


def build_spatially_balanced_support_sequence(domain_coords: np.ndarray, stable_ids: Optional[np.ndarray] = None, max_size: int = 40000) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    import heapq
    coords_in = np.asarray(domain_coords, dtype=np.float64)
    finite = np.isfinite(coords_in).all(axis=1)
    coords_in = coords_in[finite]
    ids_in = (np.arange(len(domain_coords), dtype=np.int64) if stable_ids is None else np.asarray(stable_ids, dtype=np.int64))[finite]
    if len(coords_in) == 0:
        raise ValueError("No finite coordinates are available for the spatial support sequence.")
    order = np.lexsort((ids_in, coords_in[:, 1], coords_in[:, 0]))
    sorted_coords = coords_in[order]; sorted_ids = ids_in[order]
    unique_mask = np.ones(len(sorted_coords), dtype=bool)
    if len(sorted_coords) > 1:
        unique_mask[1:] = np.any(np.diff(sorted_coords, axis=0) != 0.0, axis=1)
    coords = sorted_coords[unique_mask]; ids = sorted_ids[unique_mask]
    max_len = min(int(max_size), len(coords))
    all_indices = np.arange(len(coords), dtype=np.int64)
    lo, hi, span, diagonal = _leaf_bbox(coords, all_indices)
    root_rep = _nearest_observed_coordinate_to_center(coords, ids, all_indices, 0.5 * (lo + hi))
    rows = [{"support_rank": 1, "analysis_index": int(ids[root_rep]), "X": float(coords[root_rep, 0]), "Y": float(coords[root_rep, 1])}]
    leaves = {0: {"indices": all_indices, "representative": int(root_rep), "lo": lo, "hi": hi, "span": span, "diagonal": diagonal}}
    heap: List[Tuple[float, int]] = []
    if len(all_indices) > 1 and diagonal > 0:
        heapq.heappush(heap, (-diagonal, 0))
    leaf_counter = 0
    while len(rows) < max_len and heap:
        _, leaf_id = heapq.heappop(heap)
        leaf = leaves.pop(leaf_id, None)
        if leaf is None:
            continue
        indices = leaf["indices"]; lo = leaf["lo"]; hi = leaf["hi"]; span = leaf["span"]; diagonal = leaf["diagonal"]
        if len(indices) <= 1 or diagonal <= 0 or np.max(span) <= 0:
            continue
        axis = int(np.argmax(span)); midpoint = float(0.5 * (lo[axis] + hi[axis]))
        left_indices = indices[coords[indices, axis] <= midpoint]; right_indices = indices[coords[indices, axis] > midpoint]
        if len(left_indices) == 0 or len(right_indices) == 0:
            continue
        rep = int(leaf["representative"])
        for child_indices in (left_indices, right_indices):
            child_lo, child_hi, child_span, child_diag = _leaf_bbox(coords, child_indices)
            append_rep = not np.any(child_indices == rep)
            child_rep = rep if not append_rep else _nearest_observed_coordinate_to_center(coords, ids, child_indices, 0.5 * (child_lo + child_hi))
            leaf_counter += 1
            leaves[leaf_counter] = {"indices": child_indices, "representative": int(child_rep), "lo": child_lo, "hi": child_hi, "span": child_span, "diagonal": child_diag}
            if len(child_indices) > 1 and child_diag > 0 and np.max(child_span) > 0:
                heapq.heappush(heap, (-float(child_diag), leaf_counter))
            if append_rep and len(rows) < max_len:
                rows.append({"support_rank": len(rows) + 1, "analysis_index": int(ids[child_rep]), "X": float(coords[child_rep, 0]), "Y": float(coords[child_rep, 1])})
    table = pd.DataFrame(rows)
    report = {"spatial_domain_rows": int(len(domain_coords)), "finite_spatial_domain_rows": int(np.sum(finite)), "unique_spatial_domain_coordinates": int(len(coords)), "support_sequence_maximum_length": int(len(table)), "algorithm": "deterministic recursive occupied-space bisection", "uses_only_xy": True, "uses_pc_scores": False, "uses_row_count_as_geographic_weight": False}
    return table, report


def exact_sbad(support_coords: np.ndarray, selected_coords: np.ndarray) -> float:
    support = np.asarray(support_coords, dtype=np.float64); selected = np.asarray(selected_coords, dtype=np.float64)
    if len(support) == 0 or len(selected) == 0:
        return float("nan")
    d, _ = cKDTree(selected).query(support, k=1, workers=-1)
    return float(np.mean(np.asarray(d, dtype=np.float64)))


class SupportDistanceCache:
    def __init__(self, support_coords: np.ndarray, coords: np.ndarray, max_mib: int = 512):
        self.support_coords = np.asarray(support_coords, dtype=np.float64); self.coords = np.asarray(coords, dtype=np.float64)
        self.max_bytes = int(max_mib) * 1024 * 1024; self.cache: "OrderedDict[int, np.ndarray]" = OrderedDict(); self.bytes = 0
    def get(self, candidate_index: int) -> np.ndarray:
        key = int(candidate_index)
        if key in self.cache:
            value = self.cache.pop(key); self.cache[key] = value; return value
        dist = np.sqrt(np.einsum("ij,ij->i", self.support_coords - self.coords[key], self.support_coords - self.coords[key])).astype(np.float32, copy=False)
        self.cache[key] = dist; self.bytes += dist.nbytes
        while self.bytes > self.max_bytes and len(self.cache) > 1:
            _, old = self.cache.popitem(last=False); self.bytes -= old.nbytes
        return dist


class SBADNearestState:
    def __init__(self, support_coords: np.ndarray, coords: np.ndarray, selected: np.ndarray, cache: SupportDistanceCache):
        self.support_coords = np.asarray(support_coords, dtype=np.float64); self.coords = np.asarray(coords, dtype=np.float64); self.selected = np.asarray(selected, dtype=int).copy(); self.cache = cache; self.recompute()
    def recompute(self) -> None:
        selected_coords = self.coords[self.selected]
        if len(selected_coords) == 1:
            d, _ = cKDTree(selected_coords).query(self.support_coords, k=1, workers=-1)
            self.nearest_distance = np.asarray(d, dtype=np.float64); self.second_distance = np.full(len(self.support_coords), np.inf); self.nearest_position = np.zeros(len(self.support_coords), dtype=int)
        else:
            d, nearest = cKDTree(selected_coords).query(self.support_coords, k=2, workers=-1)
            self.nearest_distance = np.asarray(d[:, 0], dtype=np.float64); self.second_distance = np.asarray(d[:, 1], dtype=np.float64); self.nearest_position = np.asarray(nearest[:, 0], dtype=int)
        self.sbad = float(np.mean(self.nearest_distance))
    def evaluate_swap(self, position: int, candidate_index: int) -> float:
        candidate_distance = self.cache.get(int(candidate_index))[:len(self.support_coords)].astype(np.float64, copy=False)
        baseline = np.where(self.nearest_position == int(position), self.second_distance, self.nearest_distance)
        return float(np.mean(np.minimum(baseline, candidate_distance), dtype=np.float64))



def minimum_selected_separation(coords: np.ndarray) -> float:
    return float(np.min(selected_nearest_neighbor_distances(coords))) if len(coords) >= 2 else float("inf")


def design_metrics(selected: np.ndarray, targets: np.ndarray, scores: np.ndarray, coords: np.ndarray, sbad: Optional[float] = None) -> Dict[str, float]:
    mean_pc, max_pc = matching_metrics(selected, targets, scores)
    selected_coords = coords[selected]
    return {"SBAD": float(sbad) if sbad is not None else float("nan"), "geoMSD": exact_geo_msd(selected_coords), "minimum_separation": minimum_selected_separation(selected_coords), "mean_pca_target_distance": float(mean_pc), "max_pca_target_distance": float(max_pc)}


def _tol_equal(a: float, b: float, rel_tol: float = 1e-9, abs_tol: float = 1e-12) -> bool:
    return abs(a - b) <= max(abs_tol, rel_tol * max(1.0, abs(a), abs(b)))


def ad_reference_is_better(candidate: Dict[str, float], incumbent: Dict[str, float], config: RSSDConfig) -> bool:
    if candidate["SBAD"] < incumbent["SBAD"] - config.PCA_TIE_TOL:
        return True
    if _tol_equal(candidate["SBAD"], incumbent["SBAD"], abs_tol=config.PCA_TIE_TOL):
        geo_tol = config.GEOMSD_TIE_REL_TOL * max(1.0, abs(candidate["geoMSD"]), abs(incumbent["geoMSD"]))
        if candidate["geoMSD"] > incumbent["geoMSD"] + geo_tol:
            return True
        if abs(candidate["geoMSD"] - incumbent["geoMSD"]) <= geo_tol:
            if candidate["minimum_separation"] > incumbent["minimum_separation"] + config.PCA_TIE_TOL:
                return True
            if _tol_equal(candidate["minimum_separation"], incumbent["minimum_separation"], abs_tol=config.PCA_TIE_TOL):
                if candidate["mean_pca_target_distance"] < incumbent["mean_pca_target_distance"] - config.PCA_TIE_TOL:
                    return True
                if _tol_equal(candidate["mean_pca_target_distance"], incumbent["mean_pca_target_distance"], abs_tol=config.PCA_TIE_TOL):
                    return candidate["max_pca_target_distance"] < incumbent["max_pca_target_distance"] - config.PCA_TIE_TOL
    return False


def hybrid_is_better(candidate: Dict[str, float], incumbent: Dict[str, float], config: RSSDConfig, sbad_limit: float) -> bool:
    cand_feasible = candidate["SBAD"] <= sbad_limit + config.PCA_TIE_TOL
    inc_feasible = incumbent["SBAD"] <= sbad_limit + config.PCA_TIE_TOL
    if cand_feasible and not inc_feasible:
        return True
    if inc_feasible and not cand_feasible:
        return False
    if not cand_feasible and not inc_feasible:
        return ad_reference_is_better(candidate, incumbent, config)
    geo_tol = config.GEOMSD_TIE_REL_TOL * max(1.0, abs(candidate["geoMSD"]), abs(incumbent["geoMSD"]))
    if candidate["geoMSD"] > incumbent["geoMSD"] + geo_tol:
        return True
    if abs(candidate["geoMSD"] - incumbent["geoMSD"]) <= geo_tol:
        if candidate["minimum_separation"] > incumbent["minimum_separation"] + config.PCA_TIE_TOL:
            return True
        if _tol_equal(candidate["minimum_separation"], incumbent["minimum_separation"], abs_tol=config.PCA_TIE_TOL):
            if candidate["mean_pca_target_distance"] < incumbent["mean_pca_target_distance"] - config.PCA_TIE_TOL:
                return True
            if _tol_equal(candidate["mean_pca_target_distance"], incumbent["mean_pca_target_distance"], abs_tol=config.PCA_TIE_TOL):
                return candidate["max_pca_target_distance"] < incumbent["max_pca_target_distance"] - config.PCA_TIE_TOL
    return False


def candidate_pool_for_target_incremental(target: np.ndarray, pc_tree: cKDTree, search_scores: np.ndarray, search_coords: np.ndarray, search_global_indices: np.ndarray, desired: int, config: RSSDConfig) -> Tuple[np.ndarray, np.ndarray, float, int, bool]:
    tolerance = float(config.PC_CANDIDATE_TOLERANCE)
    local = pc_tree.query_ball_point(target, r=tolerance)
    expansions = 0
    while len(local) < desired and tolerance < config.MAX_PC_CANDIDATE_TOLERANCE:
        tolerance = min(config.MAX_PC_CANDIDATE_TOLERANCE, tolerance * config.CANDIDATE_TOLERANCE_GROWTH)
        local = pc_tree.query_ball_point(target, r=tolerance); expansions += 1
    nearest_k_fallback = False
    if len(local) < desired:
        k = min(max(desired, 1), len(search_scores))
        nearest_d, nearest_i = pc_tree.query(target, k=k)
        local = np.atleast_1d(nearest_i).astype(int).tolist()
        tolerance = max(tolerance, float(np.max(np.atleast_1d(nearest_d))))
        nearest_k_fallback = True
    if len(local) == 0:
        raise RuntimeError("No candidate observations were found for a response-surface target.")
    local_arr = np.asarray(local, dtype=int)
    pc_dist = np.linalg.norm(search_scores[local_arr] - target, axis=1)
    stable = np.lexsort((search_global_indices[local_arr], pc_dist))
    local_arr = local_arr[stable]; pc_dist = pc_dist[stable]
    selected_positions = [0]
    remaining = np.ones(len(local_arr), dtype=bool); remaining[0] = False
    min_geo = np.linalg.norm(search_coords[local_arr] - search_coords[local_arr[0]], axis=1); min_geo[0] = -np.inf
    while len(selected_positions) < min(desired, len(local_arr)):
        rem_pos = np.flatnonzero(remaining)
        order = np.lexsort((search_global_indices[local_arr[rem_pos]], pc_dist[rem_pos], -min_geo[rem_pos]))
        chosen = int(rem_pos[order[0]])
        selected_positions.append(chosen); remaining[chosen] = False
        new_dist = np.linalg.norm(search_coords[local_arr] - search_coords[local_arr[chosen]], axis=1)
        min_geo = np.minimum(min_geo, new_dist); min_geo[~remaining] = -np.inf
    positions = np.asarray(selected_positions, dtype=int)
    return search_global_indices[local_arr[positions]], pc_dist[positions], float(tolerance), int(expansions), bool(nearest_k_fallback)


def candidate_pool_for_target(target, pc_tree, search_scores, search_coords, search_global_indices, desired, config):
    indices, distances, tolerance, expansions, _ = candidate_pool_for_target_incremental(target, pc_tree, search_scores, search_coords, search_global_indices, desired, config)
    return indices, distances, tolerance, expansions


def discover_candidate_pools(target_instances: pd.DataFrame, search_indices: np.ndarray, scores: np.ndarray, coords: np.ndarray, config: RSSDConfig, pool_multiplier: int = 1, requested_k: Optional[int] = None) -> Tuple[List[np.ndarray], List[np.ndarray], pd.DataFrame, List[float]]:
    p = config.N_DESIGN_COMPONENTS
    target_cols = [f"target_PC{j + 1}" for j in range(p)]
    targets = target_instances[target_cols].to_numpy(dtype=np.float64)
    search_scores = scores[search_indices]; search_coords = coords[search_indices]; pc_tree = cKDTree(search_scores)
    target_keys = [tuple(row) for row in np.round(targets, 12)]
    multiplicities = {key: target_keys.count(key) for key in set(target_keys)}
    unique_sequences: Dict[Tuple[float, ...], Tuple[np.ndarray, np.ndarray, float, int, bool]] = {}
    pools: List[np.ndarray] = []; pool_distances: List[np.ndarray] = []; tolerances: List[float] = []; records: List[Dict[str, Any]] = []
    base_k = int(requested_k if requested_k is not None else config.CANDIDATES_PER_TARGET * pool_multiplier)
    for t, target in enumerate(targets):
        key = target_keys[t]; desired = max(base_k, multiplicities[key])
        if key not in unique_sequences or len(unique_sequences[key][0]) < desired:
            unique_sequences[key] = candidate_pool_for_target_incremental(target, pc_tree, search_scores, search_coords, search_indices, desired, config)
        indices, distances, tolerance, expansions, fallback = unique_sequences[key]
        pools.append(indices[:desired]); pool_distances.append(distances[:desired]); tolerances.append(float(tolerance))
        for rank, (index, distance) in enumerate(zip(indices[:desired], distances[:desired]), start=1):
            row = {"target_position": t, "target_instance_id": target_instances.iloc[t]["target_instance_id"], "base_target_id": target_instances.iloc[t]["base_target_id"], "target_type": target_instances.iloc[t]["target_type"], "sample_role": target_instances.iloc[t].get("sample_role", "response_surface"), "candidate_analysis_index": int(index), "candidate_rank": rank, "pca_target_distance": float(distance), "final_tolerance_used": float(tolerance), "tolerance_expansions": int(expansions), "nearest_k_fallback_implied": bool(fallback), "candidate_sequence_shared_for_replicated_target": bool(multiplicities[key] > 1)}
            row.update({c: float(target_instances.iloc[t][c]) for c in target_cols})
            records.append(row)
    return pools, pool_distances, pd.DataFrame(records), tolerances


def discover_candidate_pools_saturated(target_instances: pd.DataFrame, search_indices: np.ndarray, scores: np.ndarray, coords: np.ndarray, config: RSSDConfig) -> Tuple[List[np.ndarray], List[np.ndarray], pd.DataFrame, List[float], pd.DataFrame, int, int]:
    if config.CANDIDATE_EXPLORATION_MODE == "fixed":
        k_values = [int(config.CANDIDATES_PER_TARGET)]
    else:
        k = max(int(config.CANDIDATE_SATURATION_START_K), int(config.CANDIDATES_PER_TARGET)); k_values = []
        while k <= int(config.CANDIDATE_SATURATION_MAX_K):
            k_values.append(k); k *= int(config.CANDIDATE_SATURATION_GROWTH_FACTOR)
        if k_values[-1] != int(config.CANDIDATE_SATURATION_MAX_K):
            k_values = sorted(set(k_values + [int(config.CANDIDATE_SATURATION_MAX_K)]))
    previous_unique = None; final = None; records = []; assignment_attempts = 0
    for stage, k in enumerate(k_values, start=1):
        pools, distances, associations, tolerances = discover_candidate_pools(target_instances, search_indices, scores, coords, config, requested_k=int(k))
        assignment_possible = assignment_from_costs(pools, distances) is not None
        if assignment_possible and assignment_attempts == 0:
            assignment_attempts = stage
        unique_candidates = int(len(np.unique(np.concatenate(pools)))) if pools else 0
        added = None if previous_unique is None else unique_candidates - previous_unique
        stable = bool(previous_unique is not None and added == 0 and assignment_possible)
        records.append({"stage": int(stage), "requested_K": int(k), "candidate_pools_nested": True, "unique_candidate_observations": unique_candidates, "assignment_possible": bool(assignment_possible), "targets_below_requested_K": int(sum(len(pool) < k for pool in pools)), "added_unique_candidates_from_previous": added, "stable_stage": stable, "confirmation_stage": stable, "maximum_K_reached": bool(k == max(k_values))})
        final = (pools, distances, associations, tolerances); previous_unique = unique_candidates
        if stable and stage < len(k_values):
            break
    if final is None or assignment_from_costs(final[0], final[1]) is None:
        raise RuntimeError("No unique target-to-site assignment was possible after adaptive candidate expansion.")
    table = pd.DataFrame(records)
    return final[0], final[1], final[2], final[3], table, max(1, assignment_attempts), int(table["requested_K"].iloc[-1])



def _start_designs(minimum_distance_start: np.ndarray, pools: Sequence[np.ndarray], pool_distances: Sequence[np.ndarray], rng: np.random.Generator, warm_starts: Optional[Sequence[np.ndarray]] = None) -> Iterable[Tuple[str, np.ndarray]]:
    seen: set[Tuple[int, ...]] = set()
    for label, design in [("minimum_pca_distance", minimum_distance_start), *[("warm_start", w) for w in (warm_starts or [])]]:
        if design is None:
            continue
        arr = np.asarray(design, dtype=int); key = tuple(arr.tolist())
        if len(np.unique(arr)) == len(arr) and key not in seen:
            seen.add(key); yield label, arr.copy()
    while True:
        arr = assignment_from_costs(pools, pool_distances, rng)
        if arr is None:
            raise RuntimeError("Unable to generate a unique random assignment from candidate pools.")
        key = tuple(arr.tolist())
        if key not in seen:
            seen.add(key); yield "random_unique_assignment", arr.copy()


def coordinate_exchange_sbad(start: np.ndarray, pools: Sequence[np.ndarray], targets: np.ndarray, scores: np.ndarray, coords: np.ndarray, support_coords: np.ndarray, config: RSSDConfig, rng: np.random.Generator, objective_type: str, sbad_limit: Optional[float] = None) -> Tuple[np.ndarray, Dict[str, Any]]:
    selected = np.asarray(start, dtype=int).copy()
    if len(np.unique(selected)) != len(selected):
        raise ValueError("Coordinate exchange requires a unique starting assignment.")
    cache = SupportDistanceCache(support_coords, coords, config.AD_DISTANCE_CACHE_MAX_MIB)
    state = SBADNearestState(support_coords, coords, selected, cache)
    current = design_metrics(selected, targets, scores, coords, state.sbad); initial = dict(current)
    accepted = 0; sweeps = 0
    for sweep in range(1, int(config.MAX_OPTIMIZER_SWEEPS) + 1):
        accepted_this_sweep = 0
        for position in rng.permutation(len(selected)):
            used_elsewhere = set(np.delete(selected, position).tolist())
            best_index = int(selected[position]); best_metrics = current
            for candidate_index in pools[position]:
                candidate_index = int(candidate_index)
                if candidate_index == selected[position] or candidate_index in used_elsewhere:
                    continue
                proposal = selected.copy(); proposal[position] = candidate_index
                proposal_sbad = state.evaluate_swap(int(position), candidate_index)
                proposal_metrics = design_metrics(proposal, targets, scores, coords, proposal_sbad)
                better = ad_reference_is_better(proposal_metrics, best_metrics, config) if objective_type == "ad_reference" else hybrid_is_better(proposal_metrics, best_metrics, config, float(sbad_limit))
                if better:
                    best_index = candidate_index; best_metrics = proposal_metrics
            if best_index != selected[position]:
                selected[position] = best_index
                state = SBADNearestState(support_coords, coords, selected, cache)
                current = design_metrics(selected, targets, scores, coords, state.sbad)
                accepted += 1; accepted_this_sweep += 1
        sweeps = sweep
        if accepted_this_sweep == 0:
            break
    summary = {"objective_type": objective_type, "initial_SBAD": initial["SBAD"], "final_SBAD": current["SBAD"], "initial_geoMSD": initial["geoMSD"], "final_geoMSD": current["geoMSD"], "initial_minimum_separation": initial["minimum_separation"], "final_minimum_separation": current["minimum_separation"], "accepted_swaps": int(accepted), "sweeps": int(sweeps), "converged_complete_sweep_zero_swaps": bool(sweeps < int(config.MAX_OPTIMIZER_SWEEPS)), "final_mean_pca_target_distance": current["mean_pca_target_distance"], "final_max_pca_target_distance": current["max_pca_target_distance"], "distance_cache_vectors": int(len(cache.cache)), "distance_cache_mib": float(cache.bytes / (1024 * 1024))}
    return selected, summary


def _panel_sort_key(record: Dict[str, Any], objective_type: str) -> Tuple[Any, ...]:
    m = record["metrics"]
    return (m["SBAD"], -m["geoMSD"], -m["minimum_separation"], m["mean_pca_target_distance"], m["max_pca_target_distance"]) if objective_type == "ad_reference" else (-m["geoMSD"], -m["minimum_separation"], m["mean_pca_target_distance"], m["max_pca_target_distance"], m["SBAD"])


def optimize_multiple_starts_sbad(minimum_distance_start: np.ndarray, pools: Sequence[np.ndarray], pool_distances: Sequence[np.ndarray], targets: np.ndarray, scores: np.ndarray, coords: np.ndarray, support_coords: np.ndarray, config: RSSDConfig, objective_type: str, sbad_limit: Optional[float] = None, warm_starts: Optional[Sequence[np.ndarray]] = None) -> Tuple[np.ndarray, pd.DataFrame, List[Dict[str, Any]]]:
    rng = np.random.default_rng(config.RANDOM_SEED + (17 if objective_type == "hybrid" else 0) + len(support_coords))
    summaries: List[Dict[str, Any]] = []; best = None; best_metrics = None; starts_since_best = 0; panel_by_key: Dict[Tuple[int, ...], Dict[str, Any]] = {}; stop_reason = "configured_max_starts_completed"
    starts = _start_designs(minimum_distance_start, pools, pool_distances, rng, warm_starts)
    for start_number in range(1, int(config.N_OPTIMIZER_STARTS) + 1):
        start_type, start = next(starts)
        optimized, summary = coordinate_exchange_sbad(start, pools, targets, scores, coords, support_coords, config, rng, objective_type, sbad_limit)
        metrics = design_metrics(optimized, targets, scores, coords, summary["final_SBAD"])
        key = tuple(int(x) for x in optimized.tolist())
        panel_by_key[key] = {"design_key": key, "selected": optimized.copy(), "metrics": metrics, "objective_type": objective_type}
        new_best = True if best_metrics is None else (ad_reference_is_better(metrics, best_metrics, config) if objective_type == "ad_reference" else hybrid_is_better(metrics, best_metrics, config, float(sbad_limit)))
        if new_best:
            best = optimized.copy(); best_metrics = metrics; starts_since_best = 0
        else:
            starts_since_best += 1
        summary.update({"start_number": int(start_number), "start_type": start_type, "new_best_design": bool(new_best), "best_so_far_SBAD": float(best_metrics["SBAD"]), "best_so_far_geoMSD": float(best_metrics["geoMSD"]), "starts_since_best": int(starts_since_best), "stop_after_this_start": ""})
        summaries.append(summary)
        if config.EARLY_STOP_ON_STABLE_SEARCH and start_number >= int(config.MIN_OPTIMIZER_STARTS) and starts_since_best >= int(config.OPTIMIZER_NO_IMPROVEMENT_PATIENCE):
            stop_reason = "stable_no_improvement"; summaries[-1]["stop_after_this_start"] = stop_reason; break
    if best is None:
        raise RuntimeError("Optimizer produced no design.")
    panel = sorted(panel_by_key.values(), key=lambda r: _panel_sort_key(r, objective_type))[: int(config.AD_SUPPORT_RANK_PANEL_SIZE)]
    summary_df = pd.DataFrame(summaries); summary_df["optimizer_stop_reason"] = stop_reason
    return best, summary_df, panel


def _support_sizes(config: RSSDConfig, max_available: int) -> List[int]:
    if config.AD_SUPPORT_MODE == "fixed":
        return [min(int(config.AD_SUPPORT_FIXED_SIZE), max_available)]
    sizes = []; size = min(int(config.AD_SUPPORT_START_SIZE), max_available); limit = min(int(config.AD_SUPPORT_MAX_SIZE), max_available)
    while True:
        sizes.append(size)
        if size >= limit:
            break
        size = min(size * int(config.AD_SUPPORT_GROWTH_FACTOR), limit)
        if size == sizes[-1]:
            break
    return sorted(set(int(s) for s in sizes))


def _evaluate_panel_sbad(panel_designs: Sequence[np.ndarray], support_coords: np.ndarray, coords: np.ndarray) -> List[float]:
    return [exact_sbad(support_coords, coords[np.asarray(d, dtype=int)]) for d in panel_designs]


def adaptive_support_resolution_optimization(minimum_assignment: np.ndarray, candidate_pools: Sequence[np.ndarray], candidate_pool_distances: Sequence[np.ndarray], targets: np.ndarray, scores: np.ndarray, coords: np.ndarray, support_sequence: pd.DataFrame, config: RSSDConfig) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    support_all = support_sequence[["X", "Y"]].to_numpy(dtype=np.float64)
    sizes = _support_sizes(config, len(support_all))
    previous_panel_designs: List[np.ndarray] = []; previous_size = None; previous_sbad_star = None; previous_hybrid_key = None; stable_count = 0
    records: List[Dict[str, Any]] = []; all_optimizer_rows: List[pd.DataFrame] = []; warm_ad: List[np.ndarray] = []; warm_hybrid: List[np.ndarray] = []
    final_ad = None; final_hybrid = None; final_meta: Dict[str, Any] = {}
    for size in sizes:
        stage_start = time.perf_counter(); prefix = support_all[:size]
        ad_best, ad_summary, ad_panel = optimize_multiple_starts_sbad(minimum_assignment, candidate_pools, candidate_pool_distances, targets, scores, coords, prefix, config, "ad_reference", warm_starts=warm_ad)
        sbad_star = exact_sbad(prefix, coords[ad_best]); sbad_limit = (1.0 + float(config.AD_COVERAGE_ENVELOPE_REL_TOL)) * sbad_star
        hybrid_best, hybrid_summary, hybrid_panel = optimize_multiple_starts_sbad(minimum_assignment, candidate_pools, candidate_pool_distances, targets, scores, coords, prefix, config, "hybrid", sbad_limit=sbad_limit, warm_starts=[ad_best, *warm_hybrid])
        hybrid_sbad = exact_sbad(prefix, coords[hybrid_best]); hybrid_metrics = design_metrics(hybrid_best, targets, scores, coords, hybrid_sbad)
        if hybrid_metrics["SBAD"] > sbad_limit + max(config.PCA_TIE_TOL, 1e-9 * max(1.0, sbad_limit)):
            warnings.warn("Hybrid optimizer did not find a coverage-feasible design; using AD-reference as fallback.")
            hybrid_best = ad_best.copy(); hybrid_sbad = sbad_star; hybrid_metrics = design_metrics(hybrid_best, targets, scores, coords, hybrid_sbad)
        all_optimizer_rows.extend([ad_summary.assign(support_size=int(size)), hybrid_summary.assign(support_size=int(size))])
        panel_by_key: Dict[Tuple[int, ...], np.ndarray] = {}
        for record in ad_panel + hybrid_panel:
            panel_by_key[tuple(record["selected"].tolist())] = record["selected"]
        current_panel_designs = list(panel_by_key.values())[: int(config.AD_SUPPORT_RANK_PANEL_SIZE)]
        rank_corr = None; rel_change = None; best_changed = None; stable_stage = False
        if previous_size is not None and previous_panel_designs:
            union = {tuple(d.tolist()): d for d in previous_panel_designs}; union.update({tuple(d.tolist()): d for d in current_panel_designs}); union_designs = list(union.values())
            prev_vals = _evaluate_panel_sbad(union_designs, support_all[:previous_size], coords); curr_vals = _evaluate_panel_sbad(union_designs, prefix, coords)
            rank_corr = 1.0 if len(union_designs) < 2 or np.std(prev_vals) == 0 or np.std(curr_vals) == 0 else float(spearmanr(prev_vals, curr_vals).correlation)
            rel_change = abs(sbad_star - float(previous_sbad_star)) / max(1e-12, abs(float(previous_sbad_star)))
            best_changed = bool(previous_hybrid_key is not None and tuple(hybrid_best.tolist()) != previous_hybrid_key)
            stable_stage = bool(rel_change <= float(config.AD_SUPPORT_REL_TOL) and rank_corr >= float(config.AD_SUPPORT_RANK_CORRELATION_MIN) and not best_changed)
            stable_count = stable_count + 1 if stable_stage else 0
        confirmation_stage = bool(stable_count >= int(config.AD_SUPPORT_STABLE_STAGES_REQUIRED))
        records.append({"support_size": int(size), "SBAD_star": float(sbad_star), "hybrid_core_SBAD": float(hybrid_metrics["SBAD"]), "core_coverage_ratio": float(hybrid_metrics["SBAD"] / sbad_star) if sbad_star > 0 else np.nan, "hybrid_geoMSD": float(hybrid_metrics["geoMSD"]), "hybrid_minimum_separation": float(hybrid_metrics["minimum_separation"]), "design_panel_size": int(len(current_panel_designs)), "SBAD_rank_correlation_to_previous": rank_corr, "relative_change_in_best_AD_reference_SBAD": rel_change, "best_hybrid_materially_changed": best_changed, "stage_runtime_seconds": float(time.perf_counter() - stage_start), "stable_stage": bool(stable_stage), "confirmation_stage": bool(confirmation_stage)})
        final_ad = ad_best.copy(); final_hybrid = hybrid_best.copy(); final_meta = {"final_support_size": int(size), "SBAD_star": float(sbad_star), "SBAD_limit": float(sbad_limit), "ad_support_resolution_stable": bool(confirmation_stage or config.AD_SUPPORT_MODE == "fixed"), "max_support_resolution_reached": bool(size == sizes[-1])}
        warm_ad = [ad_best]; warm_hybrid = [hybrid_best]; previous_panel_designs = current_panel_designs; previous_size = int(size); previous_sbad_star = float(sbad_star); previous_hybrid_key = tuple(hybrid_best.tolist())
        if confirmation_stage:
            break
    if final_hybrid is None or final_ad is None:
        raise RuntimeError("Adaptive support optimization did not produce a design.")
    if not final_meta.get("ad_support_resolution_stable", False) and config.AD_SUPPORT_MODE == "adaptive":
        warnings.warn("SBAD support resolution reached the configured maximum without meeting decision-stability criteria.")
    return final_hybrid, final_ad, pd.DataFrame(records), pd.concat(all_optimizer_rows, ignore_index=True), final_meta


def add_spatial_support_sites(core_selected: np.ndarray, n_support_sites: int, support_coords: np.ndarray, candidate_indices: np.ndarray, coords: np.ndarray, config: RSSDConfig) -> Tuple[np.ndarray, pd.DataFrame, pd.DataFrame]:
    selected = list(map(int, core_selected.tolist())); records: List[Dict[str, Any]] = []
    candidate_indices = np.asarray(candidate_indices, dtype=int); candidate_tree = cKDTree(coords[candidate_indices])
    before_sbad = exact_sbad(support_coords, coords[np.asarray(selected, dtype=int)])
    for step in range(1, int(n_support_sites) + 1):
        nearest_dist, _ = cKDTree(coords[np.asarray(selected, dtype=int)]).query(support_coords, k=1, workers=-1)
        chosen = None; chosen_gap_rank = None
        for gap_rank, support_pos in enumerate(np.argsort(-nearest_dist, kind="stable"), start=1):
            k = min(len(candidate_indices), max(10, step * 5))
            while True:
                _, local = candidate_tree.query(support_coords[int(support_pos)], k=k)
                for local_pos in np.atleast_1d(local).astype(int):
                    candidate = int(candidate_indices[int(local_pos)])
                    if candidate not in selected:
                        chosen = candidate; chosen_gap_rank = gap_rank; break
                if chosen is not None or k >= len(candidate_indices):
                    break
                k = min(len(candidate_indices), k * 2)
            if chosen is not None:
                break
        if chosen is None:
            warnings.warn("No unused candidate remained for requested spatial support sites."); break
        selected.append(chosen); after_sbad = exact_sbad(support_coords, coords[np.asarray(selected, dtype=int)])
        records.append({"support_addition_order": int(step), "analysis_index": int(chosen), "gap_rank_used": int(chosen_gap_rank), "SBAD_before_addition": float(before_sbad), "SBAD_after_addition": float(after_sbad), "SBAD_reduction": float(before_sbad - after_sbad), "sample_role": "spatial_support"})
        before_sbad = after_sbad
    final_selected = np.asarray(selected, dtype=int)
    d, nearest = cKDTree(coords[final_selected]).query(support_coords, k=1, workers=-1)
    coverage = pd.DataFrame({"support_rank": np.arange(1, len(support_coords) + 1, dtype=int), "support_X": support_coords[:, 0], "support_Y": support_coords[:, 1], "nearest_final_site_position": nearest.astype(int) + 1, "nearest_final_site_analysis_index": final_selected[nearest.astype(int)], "nearest_final_site_distance": d.astype(float)})
    return final_selected, pd.DataFrame(records), coverage



def proxy_spacing_diagnostic(eligible_coords: np.ndarray, eligible_scores: np.ndarray, selected_nearest_distances: np.ndarray, selected_geo_msd: float, config: RSSDConfig, support_coords: Optional[np.ndarray] = None, support_scores: Optional[np.ndarray] = None) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    if not config.RUN_SPACING_DIAGNOSTIC:
        return pd.DataFrame(), {"status": "disabled", "interpretation": "Proxy spatial-scale diagnostic disabled by configuration."}
    coords_arr = np.asarray(support_coords if support_coords is not None else eligible_coords, dtype=np.float64)
    scores_arr = np.asarray(support_scores if support_scores is not None else eligible_scores, dtype=np.float64)
    selected_nn = np.asarray(selected_nearest_distances, dtype=np.float64)
    n = len(coords_arr)
    if n < 3 or len(selected_nn) < 2:
        return pd.DataFrame(), {"status": "not_estimable", "interpretation": "Too few support or selected points for a proxy spatial-scale diagnostic."}
    rng = np.random.default_rng(config.RANDOM_SEED + 280)
    sample_n = min(int(config.SPACING_DIAGNOSTIC_MAX_POINTS), n)
    sample_idx = np.arange(sample_n, dtype=int)
    sample_coords = coords_arr[sample_idx]; sample_scores = scores_arr[sample_idx]
    left_parts: List[np.ndarray] = []; right_parts: List[np.ndarray] = []
    k = min(sample_n, int(config.SPACING_DIAGNOSTIC_NEIGHBORS) + 1)
    _, neighbor_idx = cKDTree(sample_coords).query(sample_coords, k=k, workers=-1)
    neighbor_idx = np.atleast_2d(neighbor_idx)
    anchors = np.repeat(np.arange(sample_n), neighbor_idx.shape[1] - 1); neighbors = neighbor_idx[:, 1:].reshape(-1); keep = anchors < neighbors
    left_parts.append(anchors[keep]); right_parts.append(neighbors[keep])
    random_pairs = min(int(config.SPACING_DIAGNOSTIC_RANDOM_PAIRS), max(1000, sample_n * 25))
    a = rng.integers(0, sample_n, size=random_pairs, endpoint=False); b = rng.integers(0, sample_n, size=random_pairs, endpoint=False)
    same = a == b
    while np.any(same):
        b[same] = rng.integers(0, sample_n, size=int(np.sum(same)), endpoint=False); same = a == b
    left_parts.append(a); right_parts.append(b)
    left = np.concatenate(left_parts); right = np.concatenate(right_parts)
    delta = sample_coords[left] - sample_coords[right]
    distances = np.sqrt(np.einsum("ij,ij->i", delta, delta))
    valid = np.isfinite(distances) & (distances > 0); distances = distances[valid]; left = left[valid]; right = right[valid]
    if len(distances) < int(config.SPACING_DIAGNOSTIC_BINS):
        return pd.DataFrame(), {"status": "not_estimable", "sampled_pairs": int(len(distances)), "interpretation": "Too few finite support-coordinate pairs."}
    edges = np.unique(np.quantile(distances, np.linspace(0.0, 1.0, int(config.SPACING_DIAGNOSTIC_BINS) + 1)))
    if len(edges) < 3:
        edges = np.unique(np.linspace(float(np.min(distances)), float(np.max(distances)), int(config.SPACING_DIAGNOSTIC_BINS) + 1))
    bin_id = np.digitize(distances, edges[1:-1], right=False)
    records: List[Dict[str, Any]] = []; pc_ranges: Dict[str, Any] = {}
    for pc in range(sample_scores.shape[1]):
        semivar = 0.5 * (sample_scores[left, pc] - sample_scores[right, pc]) ** 2
        far_cutoff = float(np.quantile(distances, 0.80)); sill = float(np.mean(semivar[distances >= far_cutoff])) if np.any(distances >= far_cutoff) else float("nan")
        threshold = float(config.SPACING_TARGET_SEMIVARIANCE_FRACTION * sill) if np.isfinite(sill) else float("nan")
        medians = []; gammas = []
        for b_id in range(len(edges) - 1):
            mask = bin_id == b_id
            if not np.any(mask):
                continue
            gamma = float(np.mean(semivar[mask])); med = float(np.median(distances[mask]))
            records.append({"PC": f"PC{pc + 1}", "bin": int(b_id + 1), "pair_count": int(np.sum(mask)), "distance_min": float(np.min(distances[mask])), "distance_median": med, "distance_max": float(np.max(distances[mask])), "mean_proxy_semivariance": gamma, "far_distance_semivariance_reference": sill, "fraction_of_far_distance_semivariance": float(gamma / sill) if np.isfinite(sill) and sill > 0 else np.nan})
            medians.append(med); gammas.append(gamma)
        proxy_range = None; reliable = False; reason = "not_estimated"
        if np.isfinite(threshold) and threshold > 0 and len(gammas) >= 3:
            g = np.asarray(gammas, dtype=float); m = np.asarray(medians, dtype=float); reached = np.flatnonzero(g >= threshold); tail_increasing = bool(g[-1] > g[-2] and g[-2] > g[-3])
            if len(reached) and not tail_increasing:
                proxy_range = float(m[int(reached[0])]); reliable = True; reason = "threshold_reached_without_increasing_tail"
            elif len(reached):
                reason = "semivariance_tail_still_increasing"
            else:
                reason = "threshold_not_reached"
        pc_ranges[f"PC{pc + 1}"] = {"proxy_range": proxy_range, "reliable": reliable, "reason": reason}
    reliable_ranges = [v["proxy_range"] for v in pc_ranges.values() if v["reliable"] and v["proxy_range"] is not None]
    summary = {"status": "reliable_proxy_range_estimated" if reliable_ranges else "proxy_range_unreliable", "support_sequence_used": True, "simple_random_raw_row_sample_used": False, "sampled_support_points": int(sample_n), "sampled_pairs": int(len(distances)), "per_pc_range_reliability": pc_ranges, "proxy_decorrelation_distance": float(np.nanmax(reliable_ranges)) if reliable_ranges else None, "selected_absolute_minimum_spacing": float(np.min(selected_nn)), "selected_geoMSD": float(selected_geo_msd), "selected_median_nearest_neighbor_spacing": float(np.median(selected_nn)), "interpretation": "This is a pre-calibration sensor-PC proxy spatial-scale diagnostic. It is not the future regression residual correlation range and never changes the hybrid design objective."}
    return pd.DataFrame(records), summary


def export_selected_sites_kmz(selected_sites: pd.DataFrame, config: RSSDConfig, source_epsg: int, output_path: str = "ESAP_RSSD_selected_sites.kmz") -> Dict[str, Any]:
    import html, zipfile
    try:
        from pyproj import CRS, Transformer
    except ImportError:
        return {"created": False, "reason": "pyproj is not installed"}
    if not source_epsg or int(source_epsg) <= 0:
        raise ValueError("A valid projected source EPSG code is required for KMZ export.")
    source_crs = CRS.from_epsg(int(source_epsg))
    if not source_crs.is_projected:
        raise ValueError("KMZ source EPSG must describe a projected CRS.")
    transformer = Transformer.from_crs(source_crs, CRS.from_epsg(4326), always_xy=True)
    longitude, latitude = transformer.transform(pd.to_numeric(selected_sites[config.X_COLUMN]).to_numpy(float), pd.to_numeric(selected_sites[config.Y_COLUMN]).to_numpy(float))
    placemarks: List[str] = []
    for position, (_, row) in enumerate(selected_sites.iterrows(), start=1):
        site_id = str(row.get(config.ID_COLUMN, position)); role = str(row.get("sample_role", "response_surface")); role_label = "Spatial support site" if role == "spatial_support" else "Response-surface site"
        label = f"{position:02d} - {site_id}"
        details = {"Sample order": position, "Sample ID": site_id, "Sample role": role_label, "Target instance": row.get("target_instance_id", ""), "Target type": row.get("target_type", ""), "PCA target distance": row.get("pca_target_distance", ""), "Nearest selected neighbor": row.get("nearest_selected_geographic_neighbor_distance", ""), "Projected X": row.get(config.X_COLUMN, ""), "Projected Y": row.get(config.Y_COLUMN, ""), "Source EPSG": int(source_epsg)}
        description_rows = "".join(f"<tr><th align='left'>{html.escape(str(k))}</th><td>{html.escape(str(v))}</td></tr>" for k, v in details.items())
        placemarks.append("<Placemark>" f"<name>{html.escape(label)}</name><styleUrl>#rssdSample</styleUrl>" f"<description><![CDATA[<table>{description_rows}</table>]]></description>" f"<ExtendedData><Data name='sample_id'><value>{html.escape(site_id)}</value></Data><Data name='sample_role'><value>{html.escape(role_label)}</value></Data></ExtendedData>" f"<Point><coordinates>{float(longitude[position - 1]):.10f},{float(latitude[position - 1]):.10f},0</coordinates></Point></Placemark>")
    kml = "<?xml version='1.0' encoding='UTF-8'?><kml xmlns='http://www.opengis.net/kml/2.2'><Document><name>ESAP RSSD v2.9 selected sites</name><Style id='rssdSample'><IconStyle><color>ff0000ff</color><scale>1.1</scale><Icon><href>http://maps.google.com/mapfiles/kml/pushpin/red-pushpin.png</href></Icon></IconStyle><LabelStyle><color>ff000000</color><scale>0.9</scale></LabelStyle></Style>" + "".join(placemarks) + "</Document></kml>"
    destination = Path(output_path)
    with zipfile.ZipFile(destination, mode="w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("doc.kml", kml.encode("utf-8"))
    return {"created": True, "file": str(destination), "source_epsg": int(source_epsg), "output_crs": "EPSG:4326", "placemark_count": int(len(selected_sites))}



def run_esap_rssd(config: RSSDConfig, source_df: Optional[pd.DataFrame] = None, workflow_state: Optional[Dict[str, Any]] = None) -> RSSDRunResult:
    run_started = time.perf_counter(); validate_config(config); workflow_state = workflow_state or {}; stage_times: Dict[str, float] = {}
    print("1/9 Validating survey data")
    t0 = time.perf_counter()
    if source_df is None:
        source_df_raw, input_label = load_survey(config)
    else:
        source_df_raw = source_df.copy(); input_label = str(workflow_state.get("input_label", "provided_dataframe"))
    source_df, valid_mask, coordinate_eligible_mask, validation_report = validate_and_index_data(source_df_raw, config)
    valid_source_positions = np.flatnonzero(valid_mask); analysis_to_source = valid_source_positions.copy()
    analysis_coordinate_eligible = coordinate_eligible_mask[analysis_to_source]
    coords = source_df.loc[valid_mask, [config.X_COLUMN, config.Y_COLUMN]].to_numpy(dtype=np.float64)
    original_features = source_df.loc[valid_mask, config.FEATURE_COLUMNS].to_numpy(dtype=np.float64)
    transformed_features, transformation_details = transform_features(original_features, config.FEATURE_COLUMNS, config.FEATURE_TRANSFORMS)
    skewness_summary = pd.DataFrame({"configured_transform": [config.FEATURE_TRANSFORMS.get(c, "none") for c in config.FEATURE_COLUMNS], "raw_skewness": skew(original_features, axis=0, bias=False), "transformed_skewness": skew(transformed_features, axis=0, bias=False)}, index=config.FEATURE_COLUMNS)
    stage_times["validate_transform"] = time.perf_counter() - t0

    print("2/9 Standardizing sensor variables and fitting PCA")
    t0 = time.perf_counter()
    design_scores, feature_scaler, pca_estimator, pca_mode, memory_report = fit_standardized_pca(transformed_features, config)
    pca_validation, pca_correlation = pca_validation_table(design_scores)
    stage_times["pca"] = time.perf_counter() - t0

    print("3/9 Constructing the response-surface design")
    t0 = time.perf_counter(); p = config.N_DESIGN_COMPONENTS
    pc_radius = np.linalg.norm(design_scores, axis=1); design_radius = float(np.quantile(pc_radius, config.DESIGN_COVERAGE))
    pca_d2 = np.einsum("ij,ij->i", design_scores, design_scores); outlier_threshold_d2 = float(chi2.ppf(config.OUTLIER_COVERAGE, df=p)); pca_outlier = pca_d2 > outlier_threshold_d2
    candidate_eligible = analysis_coordinate_eligible & ~pca_outlier
    if int(candidate_eligible.sum()) < min(config.N_SAMPLES, len(candidate_eligible)):
        raise ValueError("Outlier and duplicate-coordinate masks leave too few candidate-eligible observations.")
    source_df["pca_space_outlier_flag"] = False; source_df.loc[analysis_to_source, "pca_space_outlier_flag"] = pca_outlier
    base_targets = make_base_ccd(p, design_radius, config.CENTER_REPLICATES)
    target_instances, target_replication_counts, sample_budget_report = allocate_sample_budget(base_targets, config.N_SAMPLES, config, p)
    target_cols = [f"target_PC{j + 1}" for j in range(p)]; target_array = target_instances[target_cols].to_numpy(dtype=np.float64)
    stage_times["response_surface_design"] = time.perf_counter() - t0

    print("4/9 Building spatially balanced geographic support")
    t0 = time.perf_counter()
    support_sequence, support_report = build_spatially_balanced_support_sequence(coords, np.arange(len(coords), dtype=np.int64), config.AD_SUPPORT_MAX_SIZE)
    support_report["candidate_eligible_rows"] = int(candidate_eligible.sum())
    stage_times["support_sequence"] = time.perf_counter() - t0

    print("5/9 Exploring candidate space")
    t0 = time.perf_counter()
    eligible_indices = np.flatnonzero(candidate_eligible)
    search_indices, approximate_prefilter_used, prefilter_report = approximate_pca_prefilter(eligible_indices, design_scores, target_array, config)
    candidate_pools, candidate_pool_distances, candidate_associations, target_tolerances, candidate_saturation, assignment_attempts, final_k = discover_candidate_pools_saturated(target_instances, search_indices, design_scores, coords, config)
    minimum_assignment = assignment_from_costs(candidate_pools, candidate_pool_distances)
    if minimum_assignment is None:
        raise RuntimeError("No unique target-to-site assignment was possible after candidate saturation.")
    candidate_pool_report = candidate_pool_summary_table(target_instances, candidate_pools, candidate_pool_distances, target_tolerances, config)
    stage_times["candidate_exploration"] = time.perf_counter() - t0

    print("6/9 Stabilizing the SBAD support resolution")
    print("7/9 Optimizing the hybrid RSSD response-surface core")
    t0 = time.perf_counter()
    core_selected, ad_reference_selected, ad_support_resolution, optimizer_stability, support_opt_report = adaptive_support_resolution_optimization(minimum_assignment, candidate_pools, candidate_pool_distances, target_array, design_scores, coords, support_sequence, config)
    stage_times["adaptive_support_and_hybrid_optimization"] = time.perf_counter() - t0

    print("8/9 Adding spatial support sites")
    t0 = time.perf_counter(); final_support_size = int(support_opt_report["final_support_size"]); final_support_coords = support_sequence[["X", "Y"]].to_numpy(dtype=np.float64)[:final_support_size]
    support_sequence = support_sequence.copy(); support_sequence["used_in_final_support_prefix"] = support_sequence["support_rank"] <= final_support_size
    final_selected, spatial_support_sites, field_coverage_distances = add_spatial_support_sites(core_selected, int(sample_budget_report["spatial_support_sites"]), final_support_coords, eligible_indices, coords, config)
    stage_times["spatial_support_site_addition"] = time.perf_counter() - t0

    print("9/9 Calculating diagnostics and writing outputs")
    t0 = time.perf_counter()
    selected_coords = coords[final_selected].astype(np.float64); selected_scores = design_scores[final_selected].astype(np.float64)
    nearest_selected = selected_nearest_neighbor_distances(selected_coords); final_geo_msd = exact_geo_msd(selected_coords); final_sbad = exact_sbad(final_support_coords, selected_coords)
    response_surface_count = len(core_selected); core_sbad = exact_sbad(final_support_coords, coords[core_selected]); core_geo_msd = exact_geo_msd(coords[core_selected]); core_min_sep = minimum_selected_separation(coords[core_selected])
    mean_target_distance, max_target_distance = matching_metrics(core_selected, target_array, design_scores)
    regression_diagnostics, leverage = regression_design_diagnostics(selected_scores, design_scores[eligible_indices], config.DIAGNOSTIC_CHUNK_SIZE)
    support_analysis_idx = support_sequence.loc[support_sequence["used_in_final_support_prefix"], "analysis_index"].to_numpy(dtype=int)
    support_scores = design_scores[support_analysis_idx]
    proxy_spatial_scale, spacing_diagnostic = proxy_spacing_diagnostic(coords[eligible_indices], design_scores[eligible_indices], nearest_selected, final_geo_msd, config, support_coords=final_support_coords, support_scores=support_scores)

    selected_source_rows = analysis_to_source[final_selected]
    base_columns = [config.ID_COLUMN, config.X_COLUMN, config.Y_COLUMN, *config.FEATURE_COLUMNS]
    selected_export = source_df.loc[selected_source_rows, base_columns].reset_index(drop=True).copy()
    selected_export.insert(0, "sample_order", np.arange(1, len(selected_export) + 1, dtype=int)); selected_export["analysis_index"] = final_selected
    selected_export["sample_role"] = ["response_surface"] * response_surface_count + ["spatial_support"] * (len(final_selected) - response_surface_count)
    for j in range(p):
        selected_export[f"selected_PC{j + 1}"] = selected_scores[:, j]; selected_export[f"target_PC{j + 1}"] = None
    for column in ["target_instance_id", "base_target_id", "target_type", "target_replicate_number", "pca_target_distance"]:
        selected_export[column] = None
    for i in range(response_surface_count):
        tr = target_instances.iloc[i]
        selected_export.loc[i, "target_instance_id"] = tr["target_instance_id"]; selected_export.loc[i, "base_target_id"] = tr["base_target_id"]; selected_export.loc[i, "target_type"] = tr["target_type"]; selected_export.loc[i, "target_replicate_number"] = tr["target_replicate_number"]
        for j in range(p):
            selected_export.loc[i, f"target_PC{j + 1}"] = float(tr[f"target_PC{j + 1}"])
        selected_export.loc[i, "pca_target_distance"] = float(np.linalg.norm(selected_scores[i] - target_array[i]))
    selected_export["nearest_selected_geographic_neighbor_distance"] = nearest_selected
    if len(leverage) == len(selected_export):
        selected_export["second_order_model_leverage"] = leverage

    candidate_export = candidate_associations.copy()
    candidate_export[config.ID_COLUMN] = source_df.iloc[analysis_to_source[candidate_export["candidate_analysis_index"].to_numpy(dtype=int)]][config.ID_COLUMN].to_numpy()
    candidate_export[config.X_COLUMN] = coords[candidate_export["candidate_analysis_index"].to_numpy(dtype=int), 0]
    candidate_export[config.Y_COLUMN] = coords[candidate_export["candidate_analysis_index"].to_numpy(dtype=int), 1]

    output_tables = {"ESAP_RSSD_spatial_support_sequence.csv": support_sequence, "ESAP_RSSD_ad_support_resolution.csv": ad_support_resolution, "ESAP_RSSD_candidate_saturation.csv": candidate_saturation, "ESAP_RSSD_optimizer_stability.csv": optimizer_stability, "ESAP_RSSD_spatial_support_sites.csv": spatial_support_sites, "ESAP_RSSD_field_coverage_distances.csv": field_coverage_distances, "ESAP_RSSD_proxy_spatial_scale.csv": proxy_spatial_scale, "ESAP_RSSD_selected_sites.csv": selected_export, "ESAP_RSSD_candidate_sites.csv": candidate_export, "ESAP_RSSD_candidate_pool_report.csv": candidate_pool_report, "ESAP_RSSD_target_instances.csv": target_instances, "ESAP_RSSD_pca_standardization_validation.csv": pca_validation, "ESAP_RSSD_feature_skewness.csv": skewness_summary}
    for filename, table in output_tables.items():
        table.to_csv(filename, index=False)

    kmz_export_metadata = {"created": False, "reason": "source projected EPSG was not supplied"}
    source_epsg = workflow_state.get("coordinate_epsg")
    if source_epsg:
        try:
            kmz_export_metadata = export_selected_sites_kmz(selected_export, config, int(source_epsg))
        except Exception as exc:
            kmz_export_metadata = {"created": False, "reason": str(exc)}
    total_runtime = time.perf_counter() - run_started
    support_stable = bool(support_opt_report.get("ad_support_resolution_stable", False))
    run_summary = f"""# ESAP RSSD v2.9 run summary

{int(config.N_SAMPLES)} calibration locations were requested. The selected design used {response_surface_count} PCA response-surface sites and {len(final_selected) - response_surface_count} spatial support sites.

Spatial coverage was measured using {final_support_size:,} nested spatially balanced survey support coordinates. {'Support resolution was judged stable.' if support_stable else 'Support resolution reached the configured maximum without meeting stability criteria; review the support-resolution table.'}

The best attainable response-surface-core SBAD was {support_opt_report['SBAD_star']:.6g}. The hybrid design was constrained to remain within {100 * config.AD_COVERAGE_ENVELOPE_REL_TOL:.3g}% of this value and achieved core SBAD = {core_sbad:.6g}, or {core_sbad / support_opt_report['SBAD_star']:.6g} times the best coverage reference.

Within this coverage envelope, exact geoMSD for the response-surface core was {core_geo_msd:.6g} and the closest selected response-surface pair was {core_min_sep:.6g} projected coordinate units apart.

{len(final_selected) - response_surface_count} spatial support sites were then added sequentially, reducing final SBAD to {final_sbad:.6g}.

The proxy spatial-scale diagnostic status was `{spacing_diagnostic.get('status')}`. This is not the future regression residual correlation range.

ESAP 2 uses spatially balanced geographic support rather than raw observation counts for the AD-style coverage objective. This prevents unequal sensor logging density from implicitly weighting geographic coverage.

Residual independence has not been proven. Residual spatial diagnostics remain a post-calibration requirement after response measurements and an ordinary regression model are fitted.
"""
    Path("ESAP_RSSD_run_summary.md").write_text(run_summary, encoding="utf-8")
    metadata = {"notebook_version": NOTEBOOK_VERSION, "input_label": input_label, "configuration": asdict(config), "validation_report": validation_report, "transformation_details": transformation_details, "pca_mode": pca_mode, "pca_explained_variance_ratio": np.asarray(pca_estimator.explained_variance_ratio_).tolist(), "memory_report": memory_report, "sample_budget": sample_budget_report, "support_sequence": support_report, "support_optimization": support_opt_report, "candidate_prefilter": {"used": approximate_prefilter_used, **prefilter_report}, "candidate_final_K": int(final_k), "regression_design_diagnostics": regression_diagnostics, "spacing_diagnostic": spacing_diagnostic, "kmz_export": kmz_export_metadata, "stage_runtime_seconds": stage_times, "total_runtime_seconds": total_runtime, "package_versions": package_versions()}
    diagnostic_data = {
        "coords": coords,
        "design_scores": design_scores,
        "eligible_indices": eligible_indices,
        "pca_outlier": pca_outlier,
        "target_array": target_array,
        "selected_indices": final_selected,
        "core_selected_indices": core_selected,
        "original_features": original_features,
        "feature_columns": list(config.FEATURE_COLUMNS),
        "candidate_indices": np.unique(candidate_export["candidate_analysis_index"].to_numpy(dtype=int)) if len(candidate_export) else np.array([], dtype=int),
    }
    Path("ESAP_RSSD_run_metadata.json").write_text(json.dumps(json_ready(metadata), indent=2, allow_nan=False), encoding="utf-8")
    stage_times["diagnostics_and_exports"] = time.perf_counter() - t0
    print(f"Completed ESAP RSSD v2.9 in {total_runtime:.1f} seconds.")
    print(run_summary)
    return RSSDRunResult(config, selected_export, candidate_export, target_instances, support_sequence, ad_support_resolution, candidate_saturation, optimizer_stability, spatial_support_sites, field_coverage_distances, proxy_spatial_scale, pca_validation, skewness_summary, metadata, run_summary, diagnostic_data)




def _save_or_show_figure(fig: Any, name: str, output_dir: Optional[Path], show: bool, saved: List[str]) -> None:
    fig.tight_layout()
    if output_dir is not None:
        output_dir.mkdir(parents=True, exist_ok=True)
        path = output_dir / name
        fig.savefig(path, dpi=220, bbox_inches="tight", facecolor="white")
        saved.append(str(path))
    if show:
        plt.show()
    else:
        plt.close(fig)


def create_esap_rssd_figures(result: RSSDRunResult, output_dir: Optional[Any] = None, show: bool = True) -> List[str]:
    """Create live and bundle figures for v2.9 maps, legacy diagnostics, and SBAD diagnostics."""
    cfg_local = result.config
    output_path = Path(output_dir) if output_dir is not None else None
    saved: List[str] = []
    selected = result.selected_sites.copy()
    candidates = result.candidate_sites.copy()
    support = result.support_sequence.copy()
    coverage = result.field_coverage_distances.copy()
    diag = result.diagnostic_data or {}
    coords_all = diag.get("coords")
    scores_all = diag.get("design_scores")
    eligible = diag.get("eligible_indices")
    original_features = diag.get("original_features")
    xcol, ycol = cfg_local.X_COLUMN, cfg_local.Y_COLUMN
    p = int(cfg_local.N_DESIGN_COMPONENTS)

    # 01: full survey/support footprint and final selected sites.
    fig, ax = plt.subplots(figsize=(9, 7))
    if coords_all is not None:
        ax.scatter(coords_all[:, 0], coords_all[:, 1], s=2, alpha=0.10, color=cfg_local.FOOTPRINT_COLOR, rasterized=True, label="Survey footprint")
    if len(support):
        final_support = support[support.get("used_in_final_support_prefix", False).astype(bool)] if "used_in_final_support_prefix" in support else support
        ax.scatter(final_support["X"], final_support["Y"], s=8, alpha=0.20, color=cfg_local.SUPPORT_COLOR, label="SBAD support prefix")
    for role, marker, color, label in [("response_surface", "x", cfg_local.SELECTED_COLOR, "Response-surface sites"), ("spatial_support", "^", cfg_local.SUPPORT_COLOR, "Spatial support sites")]:
        part = selected[selected["sample_role"] == role]
        if len(part):
            ax.scatter(part[xcol], part[ycol], marker=marker, s=90 if marker == "x" else 70, color=color, edgecolor="black" if marker != "x" else None, label=label)
    if len(selected) <= 60:
        for _, row in selected.iterrows():
            ax.annotate(str(row["sample_order"]), (row[xcol], row[ycol]), xytext=(3, 3), textcoords="offset points", fontsize=7)
    ax.set_xlabel(f"{xcol} (projected units)"); ax.set_ylabel(f"{ycol} (projected units)"); ax.set_title("Final ESAP RSSD v2.9 field map")
    ax.set_aspect("equal", adjustable="box"); ax.legend(loc="best")
    _save_or_show_figure(fig, "01_final_field_map.png", output_path, show, saved)

    # 02: candidates and selected sites.
    fig, ax = plt.subplots(figsize=(9, 7))
    if len(candidates):
        unique_candidates = candidates.drop_duplicates("candidate_analysis_index")
        ax.scatter(unique_candidates[xcol], unique_candidates[ycol], s=12, alpha=0.35, color=cfg_local.CANDIDATE_COLOR, label="Candidate observations")
    ax.scatter(selected[xcol], selected[ycol], marker="x", s=80, color=cfg_local.SELECTED_COLOR, label="Final selected")
    ax.set_xlabel(f"{xcol} (projected units)"); ax.set_ylabel(f"{ycol} (projected units)"); ax.set_title("Candidate pool and final selected locations")
    ax.set_aspect("equal", adjustable="box"); ax.legend(loc="best")
    _save_or_show_figure(fig, "02_candidates_and_selected_map.png", output_path, show, saved)

    # 03: response-surface target matches in PC space.
    response = selected[selected["sample_role"] == "response_surface"].copy()
    target_cols = [f"target_PC{j + 1}" for j in range(p)]
    selected_pc_cols = [f"selected_PC{j + 1}" for j in range(p)]
    for col in target_cols + selected_pc_cols:
        if col in response:
            response[col] = pd.to_numeric(response[col], errors="coerce")
    fig, ax = plt.subplots(figsize=(8, 6 if p > 1 else 4.5))
    if p == 1:
        ax.scatter(response["target_PC1"], np.zeros(len(response)), marker="x", s=80, color=cfg_local.TARGET_COLOR, label="Targets")
        ax.scatter(response["selected_PC1"], np.zeros(len(response)), s=40, color=cfg_local.SELECTED_COLOR, label="Selected")
        for _, row in response.iterrows():
            ax.plot([row["target_PC1"], row["selected_PC1"]], [0, 0], linewidth=0.8, alpha=0.55)
        ax.set_yticks([]); ax.set_xlabel("Standardized PC1")
    else:
        ax.scatter(response["target_PC1"], response["target_PC2"], marker="x", s=80, color=cfg_local.TARGET_COLOR, label="Targets")
        ax.scatter(response["selected_PC1"], response["selected_PC2"], s=42, color=cfg_local.SELECTED_COLOR, label="Selected response-surface sites")
        for _, row in response.iterrows():
            ax.plot([row["target_PC1"], row["selected_PC1"]], [row["target_PC2"], row["selected_PC2"]], linewidth=0.8, alpha=0.55)
        ax.set_xlabel("Standardized PC1"); ax.set_ylabel("Standardized PC2"); ax.set_aspect("equal", adjustable="box")
    ax.set_title("Response-surface target matches"); ax.legend(loc="best")
    _save_or_show_figure(fig, "03_response_surface_target_matches.png", output_path, show, saved)

    # 04: selected versus whole eligible PC boxplots, restored from v2.8 style.
    if scores_all is not None and eligible is not None:
        fig, axes = plt.subplots(1, p, figsize=(4.8 * p, 5), squeeze=False)
        selected_indices = np.asarray(diag.get("selected_indices"), dtype=int)
        for j in range(p):
            axes[0, j].boxplot([scores_all[eligible, j], scores_all[selected_indices, j]], patch_artist=True)
            axes[0, j].set_xticks([1, 2], ["Whole eligible", "Selected"])
            axes[0, j].set_title(f"Standardized PC{j + 1}")
        fig.suptitle("PC distributions: whole eligible survey vs selected")
        _save_or_show_figure(fig, "04_pc_boxplots_whole_vs_selected.png", output_path, show, saved)

    # 05: original feature boxplots, restored from v2.8 style.
    if original_features is not None and eligible is not None:
        feature_names = list(diag.get("feature_columns", cfg_local.FEATURE_COLUMNS))
        n_features = len(feature_names); ncols = min(3, n_features); nrows = math.ceil(n_features / ncols)
        selected_indices = np.asarray(diag.get("selected_indices"), dtype=int)
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.2 * nrows), squeeze=False)
        for j, feature in enumerate(feature_names):
            axes.flat[j].boxplot([original_features[eligible, j], original_features[selected_indices, j]], patch_artist=True)
            axes.flat[j].set_xticks([1, 2], ["Whole eligible", "Selected"])
            axes.flat[j].set_title(str(feature))
        for ax in axes.flat[n_features:]:
            ax.set_visible(False)
        fig.suptitle("Sensor features: whole eligible survey vs selected")
        _save_or_show_figure(fig, "05_feature_boxplots_whole_vs_selected.png", output_path, show, saved)

    # 06: SBAD support-resolution diagnostics.
    ad = result.ad_support_resolution.copy()
    if len(ad):
        fig, ax1 = plt.subplots(figsize=(8.5, 5.2))
        ax1.plot(ad["support_size"], ad["SBAD_star"], marker="o", label="AD-reference SBAD*")
        ax1.plot(ad["support_size"], ad["hybrid_core_SBAD"], marker="o", label="Hybrid core SBAD")
        ax1.set_xlabel("Nested support prefix size"); ax1.set_ylabel("SBAD (projected units)")
        ax2 = ax1.twinx()
        ax2.plot(ad["support_size"], ad["core_coverage_ratio"], marker="s", linestyle="--", color="black", label="Core coverage ratio")
        ax2.axhline(1.0 + cfg_local.AD_COVERAGE_ENVELOPE_REL_TOL, linestyle=":", color="black", linewidth=1)
        ax2.set_ylabel("SBAD / SBAD*")
        lines1, labels1 = ax1.get_legend_handles_labels(); lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")
        ax1.set_title("Adaptive SBAD support-resolution diagnostics")
        _save_or_show_figure(fig, "06_sbad_support_resolution.png", output_path, show, saved)

    # 07: final field coverage distance distribution.
    if len(coverage):
        d = pd.to_numeric(coverage["nearest_final_site_distance"], errors="coerce").dropna().to_numpy(float)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.8))
        ax1.hist(d, bins=30, color=cfg_local.SUPPORT_COLOR, alpha=0.8)
        ax1.axvline(np.mean(d), linestyle="--", color="black", label=f"SBAD={np.mean(d):.3g}")
        ax1.set_xlabel("Nearest final-site distance"); ax1.set_ylabel("Support coordinates"); ax1.set_title("Field coverage distances"); ax1.legend()
        sorted_d = np.sort(d); ax2.plot(sorted_d, np.linspace(0, 1, len(sorted_d), endpoint=True))
        ax2.set_xlabel("Nearest final-site distance"); ax2.set_ylabel("Cumulative fraction"); ax2.set_title("Coverage-distance CDF")
        _save_or_show_figure(fig, "07_field_coverage_distance_distribution.png", output_path, show, saved)

    # 08: spatial support additions.
    additions = result.spatial_support_sites.copy()
    if len(additions):
        fig, ax = plt.subplots(figsize=(7.5, 4.8))
        x = additions["support_addition_order"].to_numpy(int)
        ax.plot(x - 1, additions["SBAD_before_addition"], marker="o", label="Before addition")
        ax.plot(x, additions["SBAD_after_addition"], marker="o", label="After addition")
        ax.set_xlabel("Spatial support addition step"); ax.set_ylabel("Final-design SBAD")
        ax.set_title("Sequential spatial support-site SBAD reduction"); ax.legend()
        _save_or_show_figure(fig, "08_spatial_support_sbad_reduction.png", output_path, show, saved)

    # 09: nearest selected-neighbor distances.
    if "nearest_selected_geographic_neighbor_distance" in selected:
        nn = pd.to_numeric(selected["nearest_selected_geographic_neighbor_distance"], errors="coerce").dropna().to_numpy(float)
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.hist(nn, bins=min(15, max(5, len(nn) // 2)), color=cfg_local.SELECTED_COLOR, alpha=0.85)
        ax.axvline(float(np.exp(np.mean(np.log(nn[nn > 0])))), linestyle="--", color="black", label="geoMSD")
        ax.set_xlabel("Nearest selected-neighbor distance"); ax.set_ylabel("Selected sites"); ax.set_title("Selected-site nearest-neighbor distances"); ax.legend()
        _save_or_show_figure(fig, "09_nearest_neighbor_distances.png", output_path, show, saved)

    # 10: optimizer stability traces.
    opt = result.optimizer_stability.copy()
    if len(opt):
        final_support = opt["support_size"].max()
        opt_final = opt[opt["support_size"] == final_support]
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))
        for objective, part in opt_final.groupby("objective_type"):
            ax1.plot(part["start_number"], part["final_SBAD"], marker=".", label=objective)
            ax2.plot(part["start_number"], part["final_geoMSD"], marker=".", label=objective)
        ax1.set_xlabel("Optimizer start"); ax1.set_ylabel("Final SBAD"); ax1.set_title("Optimizer SBAD trace")
        ax2.set_xlabel("Optimizer start"); ax2.set_ylabel("Final geoMSD"); ax2.set_title("Optimizer geoMSD trace")
        ax1.legend(); ax2.legend()
        _save_or_show_figure(fig, "10_optimizer_stability_traces.png", output_path, show, saved)

    # 11: proxy spatial-scale diagnostic.
    proxy = result.proxy_spatial_scale.copy()
    if len(proxy):
        fig, ax = plt.subplots(figsize=(8.5, 5.2))
        for pc_name, part in proxy.groupby("PC"):
            ax.plot(part["distance_median"], part["fraction_of_far_distance_semivariance"], marker="o", label=pc_name)
        ax.axhline(cfg_local.SPACING_TARGET_SEMIVARIANCE_FRACTION, linestyle="--", color="black", linewidth=1, label="Target fraction")
        ax.set_xlabel("Pair distance (projected units)"); ax.set_ylabel("Fraction of far-distance semivariance")
        ax.set_title("Per-PC proxy spatial-scale diagnostic")
        ax.legend(loc="best")
        _save_or_show_figure(fig, "11_proxy_spatial_scale.png", output_path, show, saved)

    return saved


def run_unit_validations(seed: int) -> None:
    coords_small = np.array([[0.0, 0.0], [3.0, 0.0], [0.0, 4.0], [3.0, 4.0], [8.0, 1.0]])
    kd_value = exact_geo_msd(coords_small)
    dense = squareform(pdist(coords_small)); np.fill_diagonal(dense, np.inf)
    brute_value = float(np.exp(np.mean(np.log(np.min(dense, axis=1)))))
    np.testing.assert_allclose(kd_value, brute_value, rtol=1e-13, atol=1e-13)
    rng = np.random.default_rng(seed)
    latent = rng.normal(size=(4000, 3)); mixed = latent @ np.array([[1.0, 0.8, -0.4], [0.2, 1.4, 0.7], [-0.5, 0.1, 1.1]])
    xs = StandardScaler().fit_transform(mixed); estimator = PCA(n_components=3, svd_solver="full").fit(xs)
    whitened = estimator.transform(xs) / np.sqrt(estimator.explained_variance_)
    np.testing.assert_allclose(np.mean(whitened, axis=0), 0.0, atol=1e-12)
    np.testing.assert_allclose(np.std(whitened, axis=0, ddof=1), 1.0, atol=1e-12)
    np.testing.assert_allclose(np.corrcoef(whitened, rowvar=False), np.eye(3), atol=1e-12)
    support = np.array([[0.0, 0.0], [5.0, 0.0], [0.0, 5.0], [5.0, 5.0], [2.5, 2.5]])
    selected = np.array([[0.0, 0.0], [5.0, 5.0]])
    brute_sbad = np.mean(np.min(np.sqrt(((support[:, None, :] - selected[None, :, :]) ** 2).sum(axis=2)), axis=1))
    np.testing.assert_allclose(exact_sbad(support, selected), brute_sbad, rtol=1e-13, atol=1e-13)
    coords_candidates = np.array([[0.,0.],[5.,0.],[0.,5.],[5.,5.],[2.,2.],[8.,8.]])
    selected_idx = np.array([0, 3, 4]); cache = SupportDistanceCache(support, coords_candidates, 4); state = SBADNearestState(support, coords_candidates, selected_idx, cache)
    for pos in range(len(selected_idx)):
        for candidate in range(len(coords_candidates)):
            if candidate in set(np.delete(selected_idx, pos)):
                continue
            proposal = selected_idx.copy(); proposal[pos] = candidate
            np.testing.assert_allclose(state.evaluate_swap(pos, candidate), exact_sbad(support, coords_candidates[proposal]), rtol=1e-7, atol=1e-7)
    base = make_base_ccd(2, 1.0, 2); tmp = RSSDConfig(N_SAMPLES=12, N_DESIGN_COMPONENTS=2, SAMPLE_BUDGET_MODE="rssd_with_support")
    inst, _, rep = allocate_sample_budget(base, 12, tmp, 2); assert len(inst) == 10 and rep["spatial_support_sites"] == 2
    tmp.N_SAMPLES = 20; inst, _, rep = allocate_sample_budget(base, 20, tmp, 2); assert len(inst) == 18 and rep["spatial_support_sites"] == 2
    tmp.N_SAMPLES = 10; inst, _, rep = allocate_sample_budget(base, 10, tmp, 2); assert len(inst) == 10 and rep["spatial_support_sites"] == 0
    support_seq, report = build_spatially_balanced_support_sequence(rng.uniform(size=(250, 2)), np.arange(250), 64)
    assert len(support_seq) <= 64 and support_seq["support_rank"].is_monotonic_increasing and report["uses_only_xy"] and not report["uses_pc_scores"]
    print("Unit validations passed: exact geoMSD, standardized PCA, exact SBAD, cached swap SBAD, support allocation, and nested support sequence.")


validate_config(cfg)
run_unit_validations(cfg.RANDOM_SEED)
print("ESAP RSSD v2.9 engine initialized successfully.")



## 1. Guided Google Colab workflow

Run the following cells in order. Each stage presents buttons, selectors, sliders, previews, or text fields. No code editing is required. After the final **Apply optimization settings** button reports that the workflow is ready, continue to **Run the scalable RSSD analysis**.


### 1.1 Upload the survey file

Choose one complete CSV or Excel survey file. Running the next cell opens Colab's native uploader, including its upload-progress and Cancel Upload controls. The same cell can instead create the synthetic demonstration or read an existing `/content/...` path.


In [ ]:
# @title 1. Load one survey file or use the synthetic demonstration { display-mode: "form" }
DATA_SOURCE = "Upload one CSV or Excel file"  # @param ["Upload one CSV or Excel file", "Use synthetic demonstration", "Read an existing Colab file path"]
EXISTING_COLAB_FILE_PATH = ""  # @param {type:"string"}

import io

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError("ipywidgets is required for the later interactive controls and is included in Google Colab.") from exc

try:
    from google.colab import output as colab_output  # type: ignore

    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def responsive_widget_grid(
    children: Sequence[Any], min_width: int = 250, gap: str = "8px 12px"
) -> widgets.GridBox:
    """Wrap controls automatically and preserve complete description labels in narrow Colab outputs."""
    prepared = []
    for child in children:
        try:
            child.style.description_width = "initial"
        except (AttributeError, TypeError):
            pass
        prepared.append(child)
    return widgets.GridBox(
        children=tuple(prepared),
        layout=widgets.Layout(
            width="100%",
            grid_template_columns=f"repeat(auto-fit, minmax({min_width}px, 1fr))",
            grid_gap=gap,
            align_items="center",
        ),
    )


def use_full_description(widget: Any, width: Optional[str] = None) -> Any:
    """Apply natural label width and an optional full control width."""
    try:
        widget.style.description_width = "initial"
    except (AttributeError, TypeError):
        pass
    if width is not None:
        widget.layout.width = width
    return widget


display(
    widgets.HTML(
        "<style>.widget-label{max-width:none !important; white-space:normal !important;}"
        ".widget-inline-hbox{align-items:center !important;}</style>"
    )
)

SURVEY_DF: Optional[pd.DataFrame] = None
SURVEY_INPUT_LABEL: Optional[str] = None
WORKFLOW_CONFIG_APPLIED = False
RANGE_FILTER_AUDIT: List[Dict[str, Any]] = []
ROW_RANGE_FILTER_APPLIED = False


def _read_survey_bytes(name: str, content: bytes) -> pd.DataFrame:
    suffix = Path(name).suffix.lower()
    stream = io.BytesIO(content)
    if suffix == ".csv":
        return pd.read_csv(stream)
    if suffix in {".xls", ".xlsx", ".xlsm"}:
        return pd.read_excel(stream)
    raise ValueError(f"Unsupported file extension {suffix!r}; use CSV, XLS, XLSX, or XLSM.")


def _read_survey_path(file_path: str) -> pd.DataFrame:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"No file exists at {path}.")
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".xls", ".xlsx", ".xlsm"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported file extension {suffix!r}; use CSV, XLS, XLSX, or XLSM.")


if DATA_SOURCE == "Use synthetic demonstration":
    print("Creating the deterministic synthetic demonstration...")
    SURVEY_DF = make_synthetic_survey(cfg.SYNTHETIC_ROWS, cfg.RANDOM_SEED)
    SURVEY_INPUT_LABEL = "synthetic_demo"
    print(
        f"SYNTHETIC DATA READY: {len(SURVEY_DF):,} rows × "
        f"{SURVEY_DF.shape[1]} columns."
    )

elif DATA_SOURCE == "Read an existing Colab file path":
    if not EXISTING_COLAB_FILE_PATH.strip():
        raise ValueError("Enter an existing Colab file path, such as /content/my_survey.csv.")
    print(f"[1/3] Reading {EXISTING_COLAB_FILE_PATH}...")
    SURVEY_DF = _read_survey_path(EXISTING_COLAB_FILE_PATH.strip())
    SURVEY_INPUT_LABEL = str(Path(EXISTING_COLAB_FILE_PATH.strip()))
    print("[2/3] File parsed. Validating rows and columns...")
    if SURVEY_DF.empty or SURVEY_DF.shape[1] == 0:
        raise ValueError("The parsed file contains no data rows or columns.")
    print(
        f"[3/3] FILE LOAD SUCCESSFUL: {SURVEY_INPUT_LABEL}; "
        f"{len(SURVEY_DF):,} rows × {SURVEY_DF.shape[1]} columns."
    )

else:
    try:
        from google.colab import files as colab_files  # type: ignore
    except ImportError as exc:
        raise RuntimeError(
            "Native upload is available in Google Colab. Outside Colab, choose "
            "'Read an existing Colab file path' and provide a local path."
        ) from exc

    print("[1/3] Choose exactly one CSV or Excel file in the native Colab dialog below.")
    print("Colab displays its own byte-level upload indicator and Cancel Upload button.")
    native_uploaded = colab_files.upload()
    if not native_uploaded:
        raise RuntimeError("Upload was cancelled or no file was received. Rerun this cell to try again.")
    if len(native_uploaded) != 1:
        raise ValueError("Upload exactly one complete survey file, then rerun this cell.")
    uploaded_name = next(iter(native_uploaded))
    uploaded_content = bytes(native_uploaded[uploaded_name])
    print(
        f"[2/3] TRANSFER COMPLETE: {uploaded_name} "
        f"({len(uploaded_content) / 1024**2:,.2f} MiB). Parsing the dataframe..."
    )
    SURVEY_DF = _read_survey_bytes(uploaded_name, uploaded_content)
    if SURVEY_DF.empty or SURVEY_DF.shape[1] == 0:
        raise ValueError("The uploaded file parsed successfully but contains no data rows or columns.")
    SURVEY_INPUT_LABEL = f"uploaded:{uploaded_name}"
    print(
        f"[3/3] UPLOAD AND FILE LOAD SUCCESSFUL: {uploaded_name}; "
        f"{len(SURVEY_DF):,} rows × {SURVEY_DF.shape[1]} columns."
    )

if SURVEY_DF is not None:
    WORKFLOW_CONFIG_APPLIED = False
    display(SURVEY_DF.head(8))
    display(
        pd.DataFrame(
            {
                "dtype": SURVEY_DF.dtypes.astype(str),
                "missing": SURVEY_DF.isna().sum(),
                "unique": SURVEY_DF.nunique(dropna=True),
            }
        )
    )


### 1.2 Enter existing or required target locations (optional)

Use **+ Add location** for sites already known or previously sampled. Enter X/longitude and Y/latitude in the same coordinate system as the survey input. Later, choose whether these locations are only evaluated/overlaid or forced into the final design. Forced locations are matched to unique eligible survey observations and locked to the statistically closest response-surface targets.


In [ ]:
# @title 2. Add or remove existing target locations { display-mode: "form" }
preferred_rows: List[Tuple[widgets.Text, widgets.FloatText, widgets.FloatText, widgets.HBox]] = []
preferred_box = widgets.VBox()
add_preferred_button = widgets.Button(description="+ Add location", button_style="info")
clear_preferred_button = widgets.Button(description="Clear all", button_style="warning")


def _refresh_preferred_box() -> None:
    preferred_box.children = tuple(row[3] for row in preferred_rows)


def _remove_preferred(row: Tuple[Any, Any, Any, Any]) -> None:
    if row in preferred_rows:
        preferred_rows.remove(row)
    _refresh_preferred_box()


def _add_preferred(_: Optional[widgets.Button] = None) -> None:
    identifier = widgets.Text(value=f"existing_{len(preferred_rows) + 1}", description="Location ID:")
    x_value = widgets.FloatText(description="X / longitude:", style={"description_width": "initial"})
    y_value = widgets.FloatText(description="Y / latitude:", style={"description_width": "initial"})
    remove = widgets.Button(description="X", layout=widgets.Layout(width="38px"))
    container = widgets.VBox([
        responsive_widget_grid([identifier, x_value, y_value], min_width=220),
        remove,
    ])
    row = (identifier, x_value, y_value, container)
    remove.on_click(lambda _, current=row: _remove_preferred(current))
    preferred_rows.append(row)
    _refresh_preferred_box()


def collect_preferred_raw() -> pd.DataFrame:
    records = []
    for identifier, x_value, y_value, _ in preferred_rows:
        records.append(
            {
                "preferred_location_id": identifier.value or f"existing_{len(records) + 1}",
                "input_x": float(x_value.value),
                "input_y": float(y_value.value),
            }
        )
    return pd.DataFrame(records, columns=["preferred_location_id", "input_x", "input_y"])


add_preferred_button.on_click(_add_preferred)
clear_preferred_button.on_click(lambda _: (preferred_rows.clear(), _refresh_preferred_box()))
display(preferred_box, widgets.HBox([add_preferred_button, clear_preferred_button]))
print("No existing locations are required. Leave this list empty to skip them.")


### 1.3 Preview the first data file

This preview shows the first rows, data types, missing counts, and unique counts for the single RSSD survey file loaded in step 1.


In [ ]:
# @title 3. View the first data file and its columns { display-mode: "form" }
preview_rows = use_full_description(widgets.IntSlider(value=8, min=3, max=30, description="Preview rows:"))
preview_button = widgets.Button(description="Refresh preview", button_style="info")
preview_output = widgets.Output()


def _preview_files(_: Optional[widgets.Button] = None) -> None:
    with preview_output:
        preview_output.clear_output(wait=True)
        if SURVEY_DF is None:
            print("Upload/load the first survey file in step 1.")
            return
        print("FIRST / RSSD SURVEY FILE")
        display(SURVEY_DF.head(preview_rows.value))
        display(
            pd.DataFrame(
                {
                    "dtype": SURVEY_DF.dtypes.astype(str),
                    "missing": SURVEY_DF.isna().sum(),
                    "unique": SURVEY_DF.nunique(dropna=True),
                }
            )
        )


preview_button.on_click(_preview_files)
display(widgets.HBox([preview_rows, preview_button]), preview_output)
_preview_files()


### 1.4 Apply scientific validity ranges before outlier screening (optional)

Use this stage to remove values that are physically invalid or scientifically irrelevant before PCA and statistical outlier screening—for example NDVI below 0.1, negative EMI readings caused by metal, or sensor saturation above a known maximum. Add one or more rules; all rules are combined with AND logic. Preview the effect first, then explicitly apply it.

Applying a filter permanently drops rows from the working dataframe for this run. The uploaded source file is never modified. To undo an applied filter, rerun the upload cell. Every rule and row count is stored in the run metadata and ZIP archive.


In [ ]:
# @title 4. Define, preview, and apply raw-data range filters { display-mode: "form" }
range_filter_rows: List[Dict[str, Any]] = []
range_filter_box = widgets.VBox()
add_range_filter_button = widgets.Button(description="+ Add range rule", button_style="info")
clear_range_filters_button = widgets.Button(description="Clear rules", button_style="warning")
preview_range_filter_button = widgets.Button(description="Preview rows to drop", button_style="info")
apply_range_filter_button = widgets.Button(description="Apply filters and drop rows", button_style="danger")
drop_nonfinite_control = widgets.Checkbox(
    value=True,
    description="Drop missing/nonfinite values in filtered columns",
    indent=False,
)
filter_plot_max_control = widgets.IntSlider(
    value=50000,
    min=1000,
    max=100000,
    step=1000,
    description="Plot sample:",
    style={"description_width": "initial"},
)
range_filter_output = widgets.Output()


def _numeric_filter_columns() -> List[str]:
    if SURVEY_DF is None:
        return []
    numeric = [str(column) for column in SURVEY_DF.select_dtypes(include=[np.number]).columns]
    # Also allow columns that are numeric strings throughout a small validation sample.
    for column in SURVEY_DF.columns:
        name = str(column)
        if name in numeric:
            continue
        sample = SURVEY_DF[column].dropna().head(1000)
        if len(sample) and pd.to_numeric(sample, errors="coerce").notna().all():
            numeric.append(name)
    return numeric


def _refresh_range_filter_box() -> None:
    range_filter_box.children = tuple(rule["container"] for rule in range_filter_rows)


def _remove_range_rule(rule: Dict[str, Any]) -> None:
    if rule in range_filter_rows:
        range_filter_rows.remove(rule)
    _refresh_range_filter_box()


def _add_range_rule(_: Optional[widgets.Button] = None) -> None:
    columns = _numeric_filter_columns()
    if not columns:
        with range_filter_output:
            range_filter_output.clear_output(wait=True)
            print("Load the survey dataframe before adding a range rule.")
        return
    preferred = next(
        (column for column in columns if column.lower() in {"ndvi", "ndre", "eca", "ec", "emi"}),
        columns[0],
    )
    column_control = widgets.Dropdown(
        options=columns,
        value=preferred,
        description="Column:",
        layout=widgets.Layout(width="260px"),
    )
    lower_control = widgets.Text(
        value="",
        placeholder="No lower bound",
        description="Lower:",
        layout=widgets.Layout(width="210px"),
    )
    upper_control = widgets.Text(
        value="",
        placeholder="No upper bound",
        description="Upper:",
        layout=widgets.Layout(width="210px"),
    )
    inclusive_control = widgets.Checkbox(value=True, description="Inclusive", indent=False)
    remove_control = widgets.Button(description="X", layout=widgets.Layout(width="38px"))
    container = widgets.VBox(
        [
            responsive_widget_grid(
                [column_control, lower_control, upper_control, inclusive_control], min_width=190
            ),
            remove_control,
        ]
    )
    rule = {
        "column": column_control,
        "lower": lower_control,
        "upper": upper_control,
        "inclusive": inclusive_control,
        "container": container,
    }
    remove_control.on_click(lambda _, current=rule: _remove_range_rule(current))
    range_filter_rows.append(rule)
    _refresh_range_filter_box()


def _parse_optional_bound(text_value: str, label: str) -> Optional[float]:
    stripped = text_value.strip()
    if stripped == "":
        return None
    value = float(stripped)
    if not np.isfinite(value):
        raise ValueError(f"{label} bound must be finite or blank.")
    return value


def _calculate_range_filter() -> Tuple[np.ndarray, List[Dict[str, Any]]]:
    if SURVEY_DF is None:
        raise ValueError("Load the survey dataframe first.")
    if not range_filter_rows:
        return np.ones(len(SURVEY_DF), dtype=bool), []
    cumulative_mask = np.ones(len(SURVEY_DF), dtype=bool)
    records: List[Dict[str, Any]] = []
    for position, rule in enumerate(range_filter_rows, start=1):
        column = str(rule["column"].value)
        lower = _parse_optional_bound(rule["lower"].value, "Lower")
        upper = _parse_optional_bound(rule["upper"].value, "Upper")
        inclusive = bool(rule["inclusive"].value)
        if lower is None and upper is None and not drop_nonfinite_control.value:
            raise ValueError(f"Rule {position} has no bound and does not filter nonfinite values.")
        if lower is not None and upper is not None and lower > upper:
            raise ValueError(f"Rule {position}: lower bound exceeds upper bound.")

        numeric = pd.to_numeric(SURVEY_DF[column], errors="coerce").to_numpy(dtype=float)
        finite = np.isfinite(numeric)
        rule_mask = finite.copy() if drop_nonfinite_control.value else np.ones(len(numeric), dtype=bool)
        if lower is not None:
            comparison = numeric >= lower if inclusive else numeric > lower
            rule_mask &= comparison if drop_nonfinite_control.value else (~finite | comparison)
        if upper is not None:
            comparison = numeric <= upper if inclusive else numeric < upper
            rule_mask &= comparison if drop_nonfinite_control.value else (~finite | comparison)
        cumulative_before = int(cumulative_mask.sum())
        cumulative_mask &= rule_mask
        records.append(
            {
                "rule_number": position,
                "column": column,
                "lower_bound": lower,
                "upper_bound": upper,
                "inclusive": inclusive,
                "drop_nonfinite": bool(drop_nonfinite_control.value),
                "finite_values": int(finite.sum()),
                "rows_passing_rule_alone": int(rule_mask.sum()),
                "cumulative_rows_before_rule": cumulative_before,
                "cumulative_rows_after_rule": int(cumulative_mask.sum()),
            }
        )
    return cumulative_mask, records


def _preview_range_filters(_: Optional[widgets.Button] = None) -> None:
    with range_filter_output:
        range_filter_output.clear_output(wait=True)
        try:
            mask, records = _calculate_range_filter()
            total = len(mask)
            kept = int(mask.sum())
            dropped = total - kept
            display(
                pd.DataFrame(
                    {
                        "rows_before": [total],
                        "rows_retained": [kept],
                        "rows_dropped": [dropped],
                        "percent_retained": [100.0 * kept / total if total else np.nan],
                    }
                )
            )
            if records:
                display(pd.DataFrame(records))
                n_rules = len(records)
                ncols = min(2, n_rules)
                nrows = math.ceil(n_rules / ncols)
                fig, axes = plt.subplots(
                    nrows, ncols, figsize=(6 * ncols, 3.8 * nrows), squeeze=False
                )
                rng = np.random.default_rng(cfg.RANDOM_SEED)
                sample_n = min(filter_plot_max_control.value, total)
                sample_index = np.sort(rng.choice(total, sample_n, replace=False))
                for j, record in enumerate(records):
                    values = pd.to_numeric(
                        SURVEY_DF[record["column"]], errors="coerce"
                    ).to_numpy(dtype=float)
                    sampled = values[sample_index]
                    axes.flat[j].hist(sampled[np.isfinite(sampled)], bins=50, color=cfg.SURVEY_COLOR)
                    if record["lower_bound"] is not None:
                        axes.flat[j].axvline(
                            record["lower_bound"], color="C3", linestyle="--", label="Lower bound"
                        )
                    if record["upper_bound"] is not None:
                        axes.flat[j].axvline(
                            record["upper_bound"], color="C1", linestyle="--", label="Upper bound"
                        )
                    axes.flat[j].set_title(f"Rule {j + 1}: {record['column']}")
                    axes.flat[j].legend()
                for ax in axes.flat[n_rules:]:
                    ax.set_visible(False)
                fig.suptitle("Raw-data range-filter preview")
                fig.tight_layout()
                plt.show()
            if kept == 0:
                print("WARNING: current rules would remove every row. Adjust bounds before applying.")
            else:
                print("Preview only—no rows have been removed yet.")
        except Exception as exc:
            print(f"Range-filter preview error: {type(exc).__name__}: {exc}")


def _apply_range_filters(_: widgets.Button) -> None:
    global SURVEY_DF, RANGE_FILTER_AUDIT, ROW_RANGE_FILTER_APPLIED, WORKFLOW_CONFIG_APPLIED
    with range_filter_output:
        range_filter_output.clear_output(wait=True)
        try:
            mask, records = _calculate_range_filter()
            total = len(mask)
            kept = int(mask.sum())
            dropped = total - kept
            if kept == 0:
                raise ValueError("Current rules would remove every row; filters were not applied.")
            application = {
                "application_number": len(RANGE_FILTER_AUDIT) + 1,
                "rows_before": total,
                "rows_retained": kept,
                "rows_dropped": dropped,
                "percent_retained": 100.0 * kept / total,
                "rules": records,
            }
            # Create only the retained working dataframe; the source file remains unchanged on disk.
            SURVEY_DF = SURVEY_DF.loc[mask].reset_index(drop=True)
            RANGE_FILTER_AUDIT.append(application)
            ROW_RANGE_FILTER_APPLIED = True
            WORKFLOW_CONFIG_APPLIED = False
            gc.collect()
            print(
                f"RANGE FILTER APPLIED: retained {kept:,} of {total:,} rows; "
                f"dropped {dropped:,} ({100.0 * dropped / total:.2f}%)."
            )
            print("This working-data change is recorded. Rerun the upload cell to undo it.")
            display(pd.DataFrame(records))
            display(SURVEY_DF.head(8))
        except Exception as exc:
            print(f"Range-filter apply error: {type(exc).__name__}: {exc}")


def _clear_range_rules(_: widgets.Button) -> None:
    range_filter_rows.clear()
    _refresh_range_filter_box()
    with range_filter_output:
        range_filter_output.clear_output(wait=True)
        print("Filter rules cleared. Previously applied row drops are not undone; rerun upload to reset.")


add_range_filter_button.on_click(_add_range_rule)
clear_range_filters_button.on_click(_clear_range_rules)
preview_range_filter_button.on_click(_preview_range_filters)
apply_range_filter_button.on_click(_apply_range_filters)

display(
    widgets.VBox(
        [
            range_filter_box,
            widgets.HBox([add_range_filter_button, clear_range_filters_button]),
            responsive_widget_grid([drop_nonfinite_control, filter_plot_max_control]),
            responsive_widget_grid([preview_range_filter_button, apply_range_filter_button]),
            range_filter_output,
        ]
    )
)
print("No range filter is applied unless you click 'Apply filters and drop rows'.")


### 1.5 Coordinate system and decimal-degree to UTM conversion

Select the source X/longitude and Y/latitude columns. If they are decimal degrees, choose automatic UTM detection or a manual zone and hemisphere, then convert. Original longitude/latitude columns are retained and new `_rssd_easting` and `_rssd_northing` columns are added. No zone is hard-coded. If the input is already projected, enter its EPSG code so the final selected sites can be transformed correctly for the Google Earth KMZ export.


In [ ]:
# @title 5. Select projected coordinates or convert longitude/latitude to UTM { display-mode: "form" }
coordinate_mode = widgets.RadioButtons(
    options=[
        ("Already projected in linear units", "projected"),
        ("Decimal degrees (longitude / latitude)", "decimal_degrees"),
    ],
    value="projected",
    description="Input coordinates:",
    style={"description_width": "initial"},
)
coordinate_x_source = use_full_description(widgets.Dropdown(description="X / longitude:"))
coordinate_y_source = use_full_description(widgets.Dropdown(description="Y / latitude:"))
utm_detection = widgets.RadioButtons(
    options=[("Automatically detect from data centroid", "auto"), ("Choose manually", "manual")],
    value="auto",
    description="UTM selection:",
    style={"description_width": "initial"},
)
utm_zone = use_full_description(widgets.Dropdown(options=list(range(1, 61)), value=13, description="UTM zone:"))
utm_hemisphere = use_full_description(widgets.Dropdown(options=["N", "S"], value="N", description="Hemisphere:"))
projected_epsg_control = use_full_description(
    widgets.IntText(
        value=0,
        description="Projected EPSG for Google Earth/KMZ (0 = unknown):",
        style={"description_width": "initial"},
    ),
    width="100%",
)
apply_coordinate_system = widgets.Button(description="Apply / convert coordinates", button_style="success")
coordinate_output = widgets.Output()
COORDINATE_MODE_APPLIED: Optional[str] = None
COORDINATE_EPSG: Optional[int] = None


def _guess_column(columns: Sequence[str], candidates: Sequence[str], fallback: str) -> str:
    lookup = {str(column).lower(): str(column) for column in columns}
    for candidate in candidates:
        if candidate in lookup:
            return lookup[candidate]
    return fallback


def refresh_coordinate_options() -> None:
    if SURVEY_DF is None:
        coordinate_x_source.options = []
        coordinate_y_source.options = []
        return
    numeric = [str(c) for c in SURVEY_DF.select_dtypes(include=[np.number]).columns]
    if not numeric:
        raise ValueError("No numeric coordinate columns were found.")
    coordinate_x_source.options = numeric
    coordinate_y_source.options = numeric
    coordinate_x_source.value = _guess_column(
        numeric, ["longitude", "lon", "long", "x", "easting", "utm_x"], numeric[0]
    )
    coordinate_y_source.value = _guess_column(
        numeric,
        ["latitude", "lat", "y", "northing", "utm_y"],
        numeric[min(1, len(numeric) - 1)],
    )


def _epsg_from_zone(zone: int, hemisphere: str) -> int:
    return (32600 if hemisphere == "N" else 32700) + int(zone)


def _auto_utm(lon: np.ndarray, lat: np.ndarray) -> Tuple[int, str, int]:
    lon_center = float(np.nanmedian(lon))
    lat_center = float(np.nanmedian(lat))
    zone = int(np.floor((lon_center + 180.0) / 6.0) + 1)
    zone = min(60, max(1, zone))
    hemisphere = "N" if lat_center >= 0 else "S"
    return zone, hemisphere, _epsg_from_zone(zone, hemisphere)


def transform_lonlat_arrays(lon: np.ndarray, lat: np.ndarray, epsg: int) -> Tuple[np.ndarray, np.ndarray]:
    try:
        from pyproj import Transformer
    except ImportError:
        import subprocess

        print("pyproj is not present; installing the coordinate-conversion dependency once...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyproj"])
        from pyproj import Transformer
    transformer = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
    easting, northing = transformer.transform(lon, lat)
    return np.asarray(easting), np.asarray(northing)


def _apply_coordinates(_: widgets.Button) -> None:
    global SURVEY_DF, COORDINATE_MODE_APPLIED, COORDINATE_EPSG, WORKFLOW_CONFIG_APPLIED
    with coordinate_output:
        coordinate_output.clear_output(wait=True)
        try:
            if SURVEY_DF is None:
                raise ValueError("Load the survey file first.")
            x_col, y_col = coordinate_x_source.value, coordinate_y_source.value
            if not x_col or not y_col or x_col == y_col:
                raise ValueError("Choose two distinct numeric coordinate columns.")
            x = pd.to_numeric(SURVEY_DF[x_col], errors="coerce").to_numpy(dtype=float)
            y = pd.to_numeric(SURVEY_DF[y_col], errors="coerce").to_numpy(dtype=float)
            if coordinate_mode.value == "decimal_degrees":
                if not np.all(np.isfinite(x) & np.isfinite(y)):
                    raise ValueError("Longitude/latitude columns contain missing or nonfinite values.")
                if np.any((x < -180) | (x > 180) | (y < -90) | (y > 90)):
                    raise ValueError("Selected decimal-degree values fall outside longitude/latitude bounds.")
                if utm_detection.value == "auto":
                    zone, hemisphere, epsg = _auto_utm(x, y)
                    zone_values = np.floor((x + 180.0) / 6.0).astype(int) + 1
                    if np.unique(zone_values).size > 1:
                        warnings.warn("Survey crosses more than one UTM zone; inspect projection distortion.")
                else:
                    zone, hemisphere = int(utm_zone.value), str(utm_hemisphere.value)
                    epsg = _epsg_from_zone(zone, hemisphere)
                easting, northing = transform_lonlat_arrays(x, y, epsg)
                SURVEY_DF = SURVEY_DF.copy()
                SURVEY_DF["_rssd_easting"] = easting
                SURVEY_DF["_rssd_northing"] = northing
                cfg.X_COLUMN, cfg.Y_COLUMN = "_rssd_easting", "_rssd_northing"
                COORDINATE_EPSG = epsg
                print(f"Converted WGS84 decimal degrees to UTM zone {zone}{hemisphere} (EPSG:{epsg}).")
                print("Added columns: _rssd_easting and _rssd_northing; original coordinates retained.")
            else:
                cfg.X_COLUMN, cfg.Y_COLUMN = str(x_col), str(y_col)
                entered_epsg = int(projected_epsg_control.value)
                if entered_epsg > 0:
                    try:
                        from pyproj import CRS
                    except ImportError:
                        import subprocess

                        print("pyproj is not present; installing the CRS validation dependency once...")
                        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyproj"])
                        from pyproj import CRS
                    projected_crs = CRS.from_epsg(entered_epsg)
                    if not projected_crs.is_projected:
                        raise ValueError("Enter a projected CRS EPSG code, not a geographic longitude/latitude CRS.")
                    COORDINATE_EPSG = entered_epsg
                    print(f"Using existing projected coordinates with EPSG:{entered_epsg}.")
                    print("This CRS will be used to create the Google Earth KMZ export.")
                else:
                    COORDINATE_EPSG = None
                    print("Using existing projected coordinates, but their EPSG code is unknown.")
                    print("RSSD can run, but KMZ export will be skipped until a valid EPSG code is supplied.")
            COORDINATE_MODE_APPLIED = coordinate_mode.value
            WORKFLOW_CONFIG_APPLIED = False
            display(SURVEY_DF[[cfg.X_COLUMN, cfg.Y_COLUMN]].head())
        except Exception as exc:
            print(f"Coordinate error: {type(exc).__name__}: {exc}")


apply_coordinate_system.on_click(_apply_coordinates)
refresh_coordinate_options()
display(
    widgets.VBox(
        [
            coordinate_mode,
            responsive_widget_grid([coordinate_x_source, coordinate_y_source]),
            utm_detection,
            responsive_widget_grid([utm_zone, utm_hemisphere]),
            projected_epsg_control,
            widgets.HTML(
                "<i>For projected input, enter its EPSG code so selected sites can be transformed "
                "to WGS84 for Google Earth. Leave 0 only if KMZ is not needed.</i>"
            ),
            apply_coordinate_system,
            coordinate_output,
        ]
    )
)


### 1.6 Select the sample ID column

Choose an existing unique identifier or create a sequential ID. Duplicate IDs are reported immediately and remain a fatal ambiguity for analysis.


In [ ]:
# @title 6. Select or create the ID column { display-mode: "form" }
id_column_selector = widgets.Dropdown(description="Sample ID:", style={"description_width": "initial"})
apply_id_button = widgets.Button(description="Apply ID column", button_style="success")
id_output = widgets.Output()


def refresh_id_options() -> None:
    options = ["Create sequential ID"]
    if SURVEY_DF is not None:
        options.extend([str(c) for c in SURVEY_DF.columns])
    id_column_selector.options = options


def _apply_id(_: widgets.Button) -> None:
    global SURVEY_DF, WORKFLOW_CONFIG_APPLIED
    with id_output:
        id_output.clear_output(wait=True)
        try:
            if SURVEY_DF is None:
                raise ValueError("Load the survey file first.")
            if id_column_selector.value == "Create sequential ID":
                SURVEY_DF = SURVEY_DF.copy()
                SURVEY_DF["_rssd_id"] = np.arange(1, len(SURVEY_DF) + 1, dtype=np.int64)
                cfg.ID_COLUMN = "_rssd_id"
            else:
                cfg.ID_COLUMN = str(id_column_selector.value)
            duplicate_rows = int(SURVEY_DF[cfg.ID_COLUMN].duplicated(keep=False).sum())
            if duplicate_rows:
                print(f"ERROR: {duplicate_rows} rows have duplicate IDs. Choose/create a unique ID before analysis.")
            else:
                print(f"ID column set to {cfg.ID_COLUMN!r}; all {len(SURVEY_DF):,} IDs are unique.")
            WORKFLOW_CONFIG_APPLIED = False
        except Exception as exc:
            print(f"ID error: {type(exc).__name__}: {exc}")


apply_id_button.on_click(_apply_id)
refresh_id_options()
display(id_column_selector, apply_id_button, id_output)


### 1.7 Select X/easting and Y/northing; preview geographic space

Select the coordinates used by geoMSD. The preview is interactive and does not calculate a survey-wide distance matrix. If longitude/latitude were converted, choose `_rssd_easting` and `_rssd_northing`.


In [ ]:
# @title 7. Select X/Y columns and preview geographic space { display-mode: "form" }
x_column_selector = widgets.Dropdown(description="X / easting:", style={"description_width": "initial"})
y_column_selector = widgets.Dropdown(description="Y / northing:", style={"description_width": "initial"})
geo_color = widgets.Dropdown(
    options=[("Blue", "C0"), ("Green", "C2"), ("Gray", "0.5"), ("Black", "black")],
    value=cfg.SURVEY_COLOR,
    description="Point color:",
)
geo_point_size = widgets.IntSlider(value=4, min=1, max=30, description="Point size:")
apply_xy_button = widgets.Button(description="Apply X/Y", button_style="success")
geo_output = widgets.Output()


def refresh_xy_options() -> None:
    if SURVEY_DF is None:
        x_column_selector.options = []
        y_column_selector.options = []
        return
    numeric = [str(c) for c in SURVEY_DF.select_dtypes(include=[np.number]).columns]
    x_column_selector.options = numeric
    y_column_selector.options = numeric
    if cfg.X_COLUMN in numeric:
        x_column_selector.value = cfg.X_COLUMN
    if cfg.Y_COLUMN in numeric:
        y_column_selector.value = cfg.Y_COLUMN


def _plot_geospace(*_: Any) -> None:
    with geo_output:
        geo_output.clear_output(wait=True)
        if SURVEY_DF is None or not x_column_selector.value or not y_column_selector.value:
            print("Load data and choose coordinate columns.")
            return
        x = pd.to_numeric(SURVEY_DF[x_column_selector.value], errors="coerce").to_numpy(float)
        y = pd.to_numeric(SURVEY_DF[y_column_selector.value], errors="coerce").to_numpy(float)
        finite = np.isfinite(x) & np.isfinite(y)
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(x[finite], y[finite], s=geo_point_size.value, alpha=0.35, color=geo_color.value, rasterized=True)
        raw_preferred = collect_preferred_raw()
        if len(raw_preferred):
            preferred_x = raw_preferred.input_x.to_numpy(float)
            preferred_y = raw_preferred.input_y.to_numpy(float)
            if COORDINATE_MODE_APPLIED == "decimal_degrees" and COORDINATE_EPSG is not None:
                preferred_x, preferred_y = transform_lonlat_arrays(
                    preferred_x, preferred_y, COORDINATE_EPSG
                )
            ax.scatter(preferred_x, preferred_y, marker="*", s=100, color="C3", label="Entered locations")
            ax.legend()
        ax.set_xlabel(str(x_column_selector.value))
        ax.set_ylabel(str(y_column_selector.value))
        ax.set_title("Geographic-space preview")
        ax.set_aspect("equal", adjustable="box")
        fig.tight_layout()
        plt.show()
        print(f"Finite coordinate rows shown: {int(finite.sum()):,} / {len(finite):,}")


def _apply_xy(_: widgets.Button) -> None:
    global WORKFLOW_CONFIG_APPLIED
    if x_column_selector.value == y_column_selector.value:
        with geo_output:
            print("Choose distinct X and Y columns.")
        return
    cfg.X_COLUMN = str(x_column_selector.value)
    cfg.Y_COLUMN = str(y_column_selector.value)
    cfg.SURVEY_COLOR = str(geo_color.value)
    WORKFLOW_CONFIG_APPLIED = False
    _plot_geospace()


apply_xy_button.on_click(_apply_xy)
x_column_selector.observe(_plot_geospace, names="value")
y_column_selector.observe(_plot_geospace, names="value")
geo_color.observe(_plot_geospace, names="value")
geo_point_size.observe(_plot_geospace, names="value")
refresh_xy_options()
display(
    responsive_widget_grid([x_column_selector, y_column_selector]),
    responsive_widget_grid([geo_color, geo_point_size, apply_xy_button], min_width=200),
    geo_output,
)
_plot_geospace()


### 1.8 Select sensor features for PCA

Select NDVI alone, NDRE alone, both, or any other numeric proxy columns. Use Ctrl/Cmd-click for multiple selections. Coordinate and ID columns are excluded automatically.


In [ ]:
# @title 8. Select PCA features { display-mode: "form" }
feature_selector = widgets.SelectMultiple(
    description="PCA features:", rows=12, layout=widgets.Layout(width="650px"), style={"description_width": "initial"}
)
select_all_features = widgets.Button(description="Select all numeric")
clear_features = widgets.Button(description="Clear selection")
apply_features = widgets.Button(description="Apply feature selection", button_style="success")
feature_output = widgets.Output()


def refresh_feature_options() -> None:
    if SURVEY_DF is None:
        feature_selector.options = []
        return
    excluded = {cfg.ID_COLUMN, cfg.X_COLUMN, cfg.Y_COLUMN}
    numeric = [str(c) for c in SURVEY_DF.select_dtypes(include=[np.number]).columns if str(c) not in excluded]
    feature_selector.options = numeric
    preferred = [c for c in numeric if c.lower() in {"ndvi", "ndre"}]
    feature_selector.value = tuple(preferred if preferred else numeric[: min(2, len(numeric))])


def _apply_features(_: widgets.Button) -> None:
    global WORKFLOW_CONFIG_APPLIED
    with feature_output:
        feature_output.clear_output(wait=True)
        features = [str(c) for c in feature_selector.value]
        if not features:
            print("Select at least one numeric sensor feature.")
            return
        cfg.FEATURE_COLUMNS = features
        cfg.N_DESIGN_COMPONENTS = min(cfg.N_DESIGN_COMPONENTS, len(features), 4)
        WORKFLOW_CONFIG_APPLIED = False
        display(SURVEY_DF[features].describe().T)
        print(f"Selected {len(features)} PCA feature(s): {features}")


select_all_features.on_click(lambda _: setattr(feature_selector, "value", tuple(feature_selector.options)))
clear_features.on_click(lambda _: setattr(feature_selector, "value", ()))
apply_features.on_click(_apply_features)
refresh_feature_options()
display(
    feature_selector,
    widgets.HBox([select_all_features, clear_features, apply_features]),
    feature_output,
)


### 1.9 Choose transformation/scaling and inspect feature distributions

Lesch's method requires centered, standardized sensor variables, so `StandardScaler` is always applied. The selector controls the explicit transformation before standardization. Robust or min-max scaling is not substituted because that would change the published coding step.


In [ ]:
# @title 9. Select transformation/scaling and view distributions/correlation { display-mode: "form" }
scaling_scheme = widgets.RadioButtons(
    options=[
        ("StandardScaler only (recommended)", "none"),
        ("Yeo-Johnson, then StandardScaler", "yeo-johnson"),
        ("Natural log, then StandardScaler", "log"),
        ("Choose separately for each feature", "custom"),
    ],
    value="none",
    description="Scaling scheme:",
    style={"description_width": "initial"},
)
custom_transform_box = widgets.VBox()
custom_transform_controls: Dict[str, widgets.Dropdown] = {}
apply_scaling_button = widgets.Button(description="Apply and preview", button_style="success")
scaling_output = widgets.Output()


def _refresh_custom_transforms(*_: Any) -> None:
    global custom_transform_controls
    previous = {name: control.value for name, control in custom_transform_controls.items()}
    custom_transform_controls = {}
    controls = []
    for feature in cfg.FEATURE_COLUMNS:
        control = widgets.Dropdown(
            options=["none", "log", "yeo-johnson"],
            value=previous.get(feature, cfg.FEATURE_TRANSFORMS.get(feature, "none")),
            description=feature,
            style={"description_width": "initial"},
        )
        custom_transform_controls[feature] = control
        controls.append(control)
    custom_transform_box.children = tuple(controls) if scaling_scheme.value == "custom" else ()


def _apply_scaling(_: widgets.Button) -> None:
    global WORKFLOW_CONFIG_APPLIED
    with scaling_output:
        scaling_output.clear_output(wait=True)
        try:
            if SURVEY_DF is None or not cfg.FEATURE_COLUMNS:
                raise ValueError("Load data and apply the feature selection first.")
            if scaling_scheme.value == "custom":
                transforms = {name: control.value for name, control in custom_transform_controls.items()}
            else:
                transforms = {name: scaling_scheme.value for name in cfg.FEATURE_COLUMNS}
            values = SURVEY_DF[cfg.FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")
            finite = np.isfinite(values.to_numpy(float)).all(axis=1)
            transformed, details = transform_features(
                values.loc[finite].to_numpy(float), cfg.FEATURE_COLUMNS, transforms
            )
            cfg.FEATURE_TRANSFORMS = transforms
            WORKFLOW_CONFIG_APPLIED = False
            n_features = len(cfg.FEATURE_COLUMNS)
            ncols = min(3, n_features)
            nrows = math.ceil(n_features / ncols)
            fig, axes = plt.subplots(nrows, ncols, figsize=(4.8 * ncols, 3.4 * nrows), squeeze=False)
            for j, feature in enumerate(cfg.FEATURE_COLUMNS):
                axes.flat[j].hist(transformed[:, j], bins=40, color=cfg.SURVEY_COLOR)
                axes.flat[j].set_title(f"{feature} ({transforms[feature]})")
            for ax in axes.flat[n_features:]:
                ax.set_visible(False)
            fig.suptitle("Transformed feature distributions")
            fig.tight_layout()
            plt.show()

            corr = np.corrcoef(transformed, rowvar=False)
            corr = np.atleast_2d(corr)
            fig, ax = plt.subplots(figsize=(max(5, n_features), max(4, n_features * 0.8)))
            image = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
            ax.set_xticks(range(n_features), cfg.FEATURE_COLUMNS, rotation=45, ha="right")
            ax.set_yticks(range(n_features), cfg.FEATURE_COLUMNS)
            for i in range(n_features):
                for j in range(n_features):
                    ax.text(j, i, f"{corr[i, j]:.2f}", ha="center", va="center")
            ax.set_title("Selected-feature correlation matrix")
            fig.colorbar(image, ax=ax, shrink=0.8)
            fig.tight_layout()
            plt.show()
            display(pd.DataFrame(details).T)
        except Exception as exc:
            print(f"Scaling/transform error: {type(exc).__name__}: {exc}")


scaling_scheme.observe(_refresh_custom_transforms, names="value")
apply_scaling_button.on_click(_apply_scaling)
_refresh_custom_transforms()
display(scaling_scheme, custom_transform_box, apply_scaling_button, scaling_output)


### 1.10 Select the number of PCA components

Calculate the PCA preview, inspect individual and cumulative explained variance, then choose one to four standardized design PCs. For NDVI-only or NDRE-only, choose one. For a joint NDVI/NDRE design, choose one or two depending on whether both independent directions are needed.


In [ ]:
# @title 10. Calculate PCA preview and select components to retain { display-mode: "form" }
calculate_pca_button = widgets.Button(description="Calculate PCA preview", button_style="info")
pca_selector = use_full_description(widgets.IntSlider(value=1, min=1, max=1, step=1, description="Design PCs:"))
pca_color = widgets.Dropdown(
    options=[("Blue", "C0"), ("Green", "C2"), ("Purple", "C4"), ("Gray", "0.5")],
    value=cfg.SURVEY_COLOR,
    description="Point color:",
)
apply_pca_button = widgets.Button(description="Apply PCA count", button_style="success")
pca_output = widgets.Output()
pca_dimension_note = widgets.HTML(
    value="Select two design PCs for the familiar two-dimensional PC1-vs-PC2 response-surface plot."
)
PCA_PREVIEW_SCORES: Optional[np.ndarray] = None
PCA_PREVIEW_ESTIMATOR: Any = None


def _calculate_pca_preview(_: Optional[widgets.Button] = None) -> None:
    global PCA_PREVIEW_SCORES, PCA_PREVIEW_ESTIMATOR, WORKFLOW_CONFIG_APPLIED
    with pca_output:
        pca_output.clear_output(wait=True)
        try:
            if SURVEY_DF is None or not cfg.FEATURE_COLUMNS:
                raise ValueError("Load data and apply features/transforms first.")
            values = SURVEY_DF[cfg.FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce").to_numpy(float)
            finite = np.isfinite(values).all(axis=1)
            transformed, _ = transform_features(values[finite], cfg.FEATURE_COLUMNS, cfg.FEATURE_TRANSFORMS)
            standardized = StandardScaler().fit_transform(transformed).astype(np.float32, copy=False)
            estimator = PCA(n_components=standardized.shape[1], svd_solver="full", random_state=cfg.RANDOM_SEED)
            raw_scores = estimator.fit_transform(standardized)
            whitened = raw_scores / np.sqrt(estimator.explained_variance_)
            PCA_PREVIEW_ESTIMATOR = estimator
            PCA_PREVIEW_SCORES = whitened[:, : min(4, whitened.shape[1])].astype(np.float32)
            pca_selector.max = PCA_PREVIEW_SCORES.shape[1]
            pca_selector.value = 2 if pca_selector.max >= 2 else 1
            explained = estimator.explained_variance_ratio_
            cumulative = np.cumsum(explained)
            display(
                pd.DataFrame(
                    {
                        "component": np.arange(1, len(explained) + 1),
                        "explained_variance_ratio": explained,
                        "cumulative_explained_variance": cumulative,
                    }
                )
            )
            fig, ax1 = plt.subplots(figsize=(8, 4.5))
            numbers = np.arange(1, len(explained) + 1)
            ax1.bar(numbers, explained, color=pca_color.value, alpha=0.7)
            ax1.set_xlabel("Principal component")
            ax1.set_ylabel("Explained variance ratio")
            ax2 = ax1.twinx()
            ax2.plot(numbers, cumulative, marker="o", color=cfg.TARGET_COLOR)
            ax2.set_ylabel("Cumulative explained variance")
            ax2.set_ylim(0, 1.05)
            ax1.set_title("PCA explained variance")
            fig.tight_layout()
            plt.show()
            WORKFLOW_CONFIG_APPLIED = False
        except Exception as exc:
            print(f"PCA preview error: {type(exc).__name__}: {exc}")


def _apply_pca(_: widgets.Button) -> None:
    global WORKFLOW_CONFIG_APPLIED
    with pca_output:
        if PCA_PREVIEW_SCORES is None:
            print("Calculate the PCA preview first.")
            return
        cfg.N_DESIGN_COMPONENTS = int(pca_selector.value)
        cfg.SURVEY_COLOR = str(pca_color.value)
        WORKFLOW_CONFIG_APPLIED = False
        print(f"Retained design PCs set to {cfg.N_DESIGN_COMPONENTS}.")


def _update_pca_dimension_note(change: Optional[Mapping[str, Any]] = None) -> None:
    if pca_selector.value == 1:
        extra = (
            " Only a one-dimensional PC1 line can be drawn. Choose 2 above for the "
            "traditional PC1-vs-PC2 rifle-scope view."
            if pca_selector.max >= 2
            else " Only one PCA direction exists because one feature was selected."
        )
        pca_dimension_note.value = f"<b>One design PC selected.</b>{extra}"
    else:
        pca_dimension_note.value = (
            "<b>Two-dimensional design enabled:</b> the next step plots standardized PC1 "
            "against PC2 with CCD targets and design/tolerance rings."
        )


calculate_pca_button.on_click(_calculate_pca_preview)
apply_pca_button.on_click(_apply_pca)
pca_selector.observe(_update_pca_dimension_note, names="value")
_update_pca_dimension_note()
display(
    responsive_widget_grid([calculate_pca_button, pca_selector, pca_color, apply_pca_button], min_width=190),
    pca_dimension_note,
    pca_output,
)


### 1.11 Select outlier threshold, design scale, and sample count

The legacy IQR slider is replaced by a statistically coherent chi-square coverage threshold in standardized PCA space. The **Design coverage** slider is the scale factor: it changes the empirical radius containing the selected proportion of survey observations. Sample count is editable, subject to the full-CCD minimum displayed by the widget.


In [ ]:
# @title 11. Adjust PCA outlier coverage, design radius, sample count, and preview layers { display-mode: "form" }
outlier_coverage_control = widgets.FloatSlider(
    value=cfg.OUTLIER_COVERAGE,
    min=0.95,
    max=0.9999,
    step=0.0001,
    readout_format=".4f",
    description="Outlier coverage:",
    style={"description_width": "initial"},
)
design_coverage_control = widgets.FloatSlider(
    value=cfg.DESIGN_COVERAGE,
    min=0.50,
    max=0.99,
    step=0.01,
    description="Design coverage / scale:",
    style={"description_width": "initial"},
)
center_replicates_control = widgets.IntSlider(value=cfg.CENTER_REPLICATES, min=1, max=10, description="Center replicates:", style={"description_width": "initial"})
budget_mode_control = widgets.Dropdown(
    options=[("RSSD with spatial support", "rssd_with_support"), ("Balanced target replication", "balanced_target_replication"), ("Exact base CCD", "ccd_exact")],
    value=cfg.SAMPLE_BUDGET_MODE,
    description="Budget mode:",
)
sample_count_control = use_full_description(widgets.BoundedIntText(value=cfg.N_SAMPLES, min=1, max=1000, description="Sample count:"))
design_layers = widgets.SelectMultiple(
    options=["All observations", "PCA outliers", "CCD targets", "Design radius", "Target tolerance rings"],
    value=("All observations", "PCA outliers", "CCD targets", "Design radius", "Target tolerance rings"),
    description="Plot layers:",
    rows=3,
    style={"description_width": "initial"},
)
target_color_control = widgets.Dropdown(
    options=[("Red", "C3"), ("Orange", "C1"), ("Black", "black"), ("Purple", "C4")],
    value=cfg.TARGET_COLOR,
    description="Target color:",
)
outlier_color_control = widgets.Dropdown(
    options=[("Orange", "C1"), ("Red", "C3"), ("Purple", "C4"), ("Black", "black")],
    value=cfg.OUTLIER_COLOR,
    description="Outlier color:",
)
design_point_size = use_full_description(widgets.IntSlider(value=5, min=1, max=30, description="Point size:"))
preview_tolerance_control = widgets.FloatSlider(
    value=cfg.PC_CANDIDATE_TOLERANCE,
    min=0.02,
    max=1.0,
    step=0.01,
    description="Target-ring radius:",
    style={"description_width": "initial"},
)
apply_design_button = widgets.Button(description="Apply design settings", button_style="success")
design_preview_output = widgets.Output()
design_minimum_label = widgets.HTML()


def _update_design_minimum(*_: Any) -> None:
    p_value = int(pca_selector.value)
    base_count = 2**p_value + 2 * p_value + int(center_replicates_control.value)
    sample_count_control.min = base_count
    if budget_mode_control.value == "ccd_exact":
        sample_count_control.value = base_count
        sample_count_control.disabled = True
    else:
        sample_count_control.disabled = False
        if sample_count_control.value < base_count:
            sample_count_control.value = base_count
    design_minimum_label.value = (
        f"<b>Full {p_value}-PC CCD minimum: {base_count} samples.</b> "
        f"Current requested count: {sample_count_control.value}."
    )


def _design_preview(*_: Any) -> None:
    with design_preview_output:
        design_preview_output.clear_output(wait=True)
        if PCA_PREVIEW_SCORES is None:
            print("Calculate the PCA preview in step 9 first.")
            return
        p_value = int(pca_selector.value)
        scores = PCA_PREVIEW_SCORES[:, :p_value]
        d2 = np.einsum("ij,ij->i", scores, scores)
        threshold_value = chi2.ppf(outlier_coverage_control.value, df=p_value)
        outliers = d2 > threshold_value
        radius_values = np.linalg.norm(scores, axis=1)
        radius = float(np.quantile(radius_values, design_coverage_control.value))
        base = make_base_ccd(p_value, radius, int(center_replicates_control.value))
        targets, replication = allocate_target_instances(
            base, int(sample_count_control.value), str(budget_mode_control.value), p_value
        )
        target_columns = [f"target_PC{i + 1}" for i in range(p_value)]
        target_values = targets[target_columns].to_numpy(float)
        rng = np.random.default_rng(cfg.RANDOM_SEED)
        plot_n = min(cfg.PLOT_MAX_POINTS, len(scores))
        plot_indices = np.sort(rng.choice(len(scores), plot_n, replace=False))
        fig, ax = plt.subplots(figsize=(8, 5.5 if p_value > 1 else 4.5))
        if p_value == 1:
            if "All observations" in design_layers.value:
                ax.scatter(scores[plot_indices, 0], np.zeros(plot_n), s=design_point_size.value, alpha=0.18, color=pca_color.value, label="Observations")
            if "PCA outliers" in design_layers.value and outliers.any():
                idx = np.flatnonzero(outliers)
                ax.scatter(scores[idx, 0], np.zeros(len(idx)), s=design_point_size.value + 5, color=outlier_color_control.value, label="Outliers")
            if "CCD targets" in design_layers.value:
                ax.scatter(target_values[:, 0], np.zeros(len(target_values)), marker="x", s=75, color=target_color_control.value, label="CCD targets")
            if "Design radius" in design_layers.value:
                ax.axvline(-radius, linestyle="--", linewidth=1, color=target_color_control.value)
                ax.axvline(radius, linestyle="--", linewidth=1, color=target_color_control.value)
            ax.set_yticks([])
            ax.set_xlabel("Standardized PC1")
            print(
                "This is a one-dimensional design because one PC is selected. "
                "Return to step 9 and select two PCs for the PC1-vs-PC2 rifle-scope view."
            )
        else:
            if "All observations" in design_layers.value:
                ax.scatter(scores[plot_indices, 0], scores[plot_indices, 1], s=design_point_size.value, alpha=0.18, color=pca_color.value, label="Observations")
            if "PCA outliers" in design_layers.value and outliers.any():
                idx = np.flatnonzero(outliers)
                ax.scatter(scores[idx, 0], scores[idx, 1], s=design_point_size.value + 5, color=outlier_color_control.value, label="Outliers")
            if "CCD targets" in design_layers.value:
                ax.scatter(target_values[:, 0], target_values[:, 1], marker="x", s=75, color=target_color_control.value, label="CCD targets")
            if "Design radius" in design_layers.value:
                ax.add_patch(
                    plt.Circle((0, 0), radius, fill=False, linestyle="--", linewidth=1.3, color=target_color_control.value, label="Design radius R")
                )
            if "Target tolerance rings" in design_layers.value:
                for target in np.unique(target_values[:, :2], axis=0):
                    ax.add_patch(
                        plt.Circle(
                            tuple(target),
                            preview_tolerance_control.value,
                            fill=False,
                            linestyle=":",
                            linewidth=0.9,
                            color=target_color_control.value,
                            alpha=0.75,
                        )
                    )
            ax.set_xlabel("Standardized PC1")
            ax.set_ylabel("Standardized PC2")
            ax.set_aspect("equal", adjustable="box")
        ax.set_title("PCA response-surface design preview" + (" (first two PCs)" if p_value > 2 else ""))
        ax.legend()
        fig.tight_layout()
        plt.show()
        display(
            pd.DataFrame(
                {
                    "design_radius_R": [radius],
                    "realized_coverage": [float(np.mean(radius_values <= radius))],
                    "masked_outliers": [int(outliers.sum())],
                    "base_targets": [len(base)],
                    "target_instances": [len(targets)],
                }
            )
        )
        display(replication.to_frame())


def _apply_design(_: widgets.Button) -> None:
    global WORKFLOW_CONFIG_APPLIED
    cfg.N_DESIGN_COMPONENTS = int(pca_selector.value)
    cfg.OUTLIER_COVERAGE = float(outlier_coverage_control.value)
    cfg.DESIGN_COVERAGE = float(design_coverage_control.value)
    cfg.CENTER_REPLICATES = int(center_replicates_control.value)
    cfg.SAMPLE_BUDGET_MODE = str(budget_mode_control.value)
    cfg.N_SAMPLES = int(sample_count_control.value)
    cfg.TARGET_COLOR = str(target_color_control.value)
    cfg.PC_CANDIDATE_TOLERANCE = float(preview_tolerance_control.value)
    cfg.OUTLIER_COLOR = str(outlier_color_control.value)
    WORKFLOW_CONFIG_APPLIED = False
    with design_preview_output:
        print("Design settings applied. Continue to optimization settings.")


for control in [outlier_coverage_control, design_coverage_control, center_replicates_control, budget_mode_control, sample_count_control, design_layers, target_color_control, outlier_color_control, design_point_size, preview_tolerance_control]:
    control.observe(_design_preview, names="value")
center_replicates_control.observe(_update_design_minimum, names="value")
budget_mode_control.observe(_update_design_minimum, names="value")
pca_selector.observe(_update_design_minimum, names="value")
apply_design_button.on_click(_apply_design)
_update_design_minimum()
display(
    widgets.VBox(
        [
            responsive_widget_grid([outlier_coverage_control, design_coverage_control]),
            responsive_widget_grid([center_replicates_control, budget_mode_control, sample_count_control], min_width=220),
            design_minimum_label,
            responsive_widget_grid([design_layers, target_color_control, outlier_color_control, design_point_size], min_width=190),
            preview_tolerance_control,
            apply_design_button,
            design_preview_output,
        ]
    )
)
_design_preview()


### 1.12 Candidate, optimization, preferred-location, and output controls

These controls replace the legacy weighted-score slider. Candidate tolerance constrains response-surface fidelity; exact geoMSD remains the primary objective. Mean and maximum PCA mismatch are tie-breakers. Existing target locations can be overlaid/evaluated or forced into the design.


In [ ]:
# @title 12. Apply candidate and optimization settings; finalize workflow { display-mode: "form" }
support_mode_control = widgets.Dropdown(
    options=[("Legacy-inspired automatic", "legacy_inspired_auto"), ("Manual", "manual"), ("None", "none")],
    value=cfg.SUPPORT_SITE_MODE,
    description="Support-site mode:",
    style={"description_width": "initial"},
)
manual_support_control = widgets.IntSlider(value=cfg.N_SUPPORT_SITES, min=0, max=50, description="Manual support sites:", style={"description_width": "initial"})
candidate_mode_control = widgets.Dropdown(options=[("Automatic saturation", "adaptive"), ("Fixed", "fixed")], value=cfg.CANDIDATE_EXPLORATION_MODE, description="Candidate exploration:", style={"description_width": "initial"})
candidates_control = widgets.IntSlider(value=cfg.CANDIDATES_PER_TARGET, min=2, max=25, description="Initial candidates/target:", style={"description_width": "initial"})
tolerance_control = widgets.FloatSlider(value=cfg.PC_CANDIDATE_TOLERANCE, min=0.02, max=1.0, step=0.01, description="Initial PC tolerance:", style={"description_width": "initial"})
max_tolerance_control = widgets.FloatSlider(value=cfg.MAX_PC_CANDIDATE_TOLERANCE, min=0.15, max=3.0, step=0.05, description="Maximum PC tolerance:", style={"description_width": "initial"})
starts_control = widgets.IntSlider(value=cfg.N_OPTIMIZER_STARTS, min=1, max=150, description="Max starts:", style={"description_width": "initial"})
min_starts_control = widgets.IntSlider(value=cfg.MIN_OPTIMIZER_STARTS, min=1, max=150, description="Minimum starts:", style={"description_width": "initial"})
patience_control = widgets.IntSlider(value=cfg.OPTIMIZER_NO_IMPROVEMENT_PATIENCE, min=1, max=100, description="No-improvement patience:", style={"description_width": "initial"})
early_stop_control = widgets.Checkbox(value=cfg.EARLY_STOP_ON_STABLE_SEARCH, description="Stop early when stable", indent=False)
verify_rerun_control = widgets.Checkbox(value=cfg.VERIFY_REPRODUCIBILITY_BY_RERUN, description="Rerun optimizer reproducibility check", indent=False)
spacing_diagnostic_control = widgets.Checkbox(value=cfg.RUN_SPACING_DIAGNOSTIC, description="Proxy spatial scale diagnostic", indent=False)
sbad_mode_control = widgets.Dropdown(options=[("Automatic decision-stable", "adaptive"), ("Fixed", "fixed")], value=cfg.AD_SUPPORT_MODE, description="SBAD accuracy:", style={"description_width": "initial"})
coverage_envelope_control = widgets.FloatSlider(value=cfg.AD_COVERAGE_ENVELOPE_REL_TOL, min=0.0, max=0.25, step=0.01, readout_format=".2f", description="SBAD envelope:", style={"description_width": "initial"})
optimizer_mode_control = widgets.Dropdown(options=[("Automatic", "adaptive"), ("Fixed starts", "fixed")], value=cfg.OPTIMIZER_INITIALIZATION_MODE, description="Optimizer starts:", style={"description_width": "initial"})
memory_control = widgets.Dropdown(options=["auto", "full", "incremental"], value=cfg.MEMORY_MODE, description="Memory mode:")
approximate_control = widgets.Checkbox(value=cfg.ALLOW_APPROXIMATE_PREFILTER, description="Allow approximate extreme-data prefilter", indent=False)
preferred_mode_control = widgets.Dropdown(
    options=[
        ("Ignore entered locations", "none"),
        ("Evaluate and overlay only", "evaluate_only"),
        ("Force nearest eligible survey sites into final design", "force"),
    ],
    value="evaluate_only" if preferred_rows else "none",
    description="Existing locations:",
    style={"description_width": "initial"},
)
candidate_color_control = widgets.Dropdown(options=[("Green", "C2"), ("Blue", "C0"), ("Purple", "C4"), ("Black", "black")], value=cfg.CANDIDATE_COLOR, description="Candidate color:")
selected_color_control = widgets.Dropdown(options=[("Red", "C3"), ("Orange", "C1"), ("Purple", "C4"), ("Black", "black")], value=cfg.SELECTED_COLOR, description="Selected color:")
apply_optimizer_button = widgets.Button(description="Apply optimization settings", button_style="success", icon="check")
optimizer_output = widgets.Output()
PREFERRED_LOCATIONS = pd.DataFrame(columns=["preferred_location_id", "x", "y"])
PREFERRED_MODE = "none"


def _prepare_preferred_locations() -> pd.DataFrame:
    raw = collect_preferred_raw()
    if len(raw) == 0:
        return pd.DataFrame(columns=["preferred_location_id", "x", "y"])
    if COORDINATE_MODE_APPLIED == "decimal_degrees":
        if COORDINATE_EPSG is None:
            raise ValueError("Apply decimal-degree to UTM conversion before finalizing locations.")
        x, y = transform_lonlat_arrays(
            raw.input_x.to_numpy(float), raw.input_y.to_numpy(float), COORDINATE_EPSG
        )
    else:
        x, y = raw.input_x.to_numpy(float), raw.input_y.to_numpy(float)
    return pd.DataFrame(
        {"preferred_location_id": raw.preferred_location_id.astype(str), "x": x, "y": y}
    )


def _finalize_workflow(_: widgets.Button) -> None:
    global WORKFLOW_CONFIG_APPLIED, PREFERRED_LOCATIONS, PREFERRED_MODE
    with optimizer_output:
        optimizer_output.clear_output(wait=True)
        try:
            if SURVEY_DF is None:
                raise ValueError("No survey dataframe is loaded.")
            required = [cfg.ID_COLUMN, cfg.X_COLUMN, cfg.Y_COLUMN, *cfg.FEATURE_COLUMNS]
            missing = [column for column in required if column not in SURVEY_DF.columns]
            if missing:
                raise KeyError(f"Required columns not applied or missing: {missing}")
            if COORDINATE_MODE_APPLIED == "decimal_degrees" and cfg.X_COLUMN not in {"_rssd_easting"}:
                raise ValueError("Select the generated UTM easting/northing columns in step 6.")
            cfg.SUPPORT_SITE_MODE = str(support_mode_control.value)
            cfg.N_SUPPORT_SITES = int(manual_support_control.value)
            cfg.CANDIDATE_EXPLORATION_MODE = str(candidate_mode_control.value)
            cfg.CANDIDATES_PER_TARGET = int(candidates_control.value)
            cfg.PC_CANDIDATE_TOLERANCE = float(tolerance_control.value)
            cfg.MAX_PC_CANDIDATE_TOLERANCE = max(float(max_tolerance_control.value), cfg.PC_CANDIDATE_TOLERANCE)
            cfg.N_OPTIMIZER_STARTS = int(starts_control.value)
            cfg.MIN_OPTIMIZER_STARTS = min(int(min_starts_control.value), cfg.N_OPTIMIZER_STARTS)
            cfg.OPTIMIZER_NO_IMPROVEMENT_PATIENCE = int(patience_control.value)
            cfg.EARLY_STOP_ON_STABLE_SEARCH = bool(early_stop_control.value)
            cfg.VERIFY_REPRODUCIBILITY_BY_RERUN = bool(verify_rerun_control.value)
            cfg.RUN_SPACING_DIAGNOSTIC = bool(spacing_diagnostic_control.value)
            cfg.AD_SUPPORT_MODE = str(sbad_mode_control.value)
            cfg.AD_COVERAGE_ENVELOPE_REL_TOL = float(coverage_envelope_control.value)
            cfg.OPTIMIZER_INITIALIZATION_MODE = str(optimizer_mode_control.value)
            cfg.MEMORY_MODE = str(memory_control.value)
            cfg.ALLOW_APPROXIMATE_PREFILTER = bool(approximate_control.value)
            cfg.CANDIDATE_COLOR = str(candidate_color_control.value)
            cfg.SELECTED_COLOR = str(selected_color_control.value)
            cfg.USE_SYNTHETIC_DEMO = False
            PREFERRED_LOCATIONS = _prepare_preferred_locations()
            PREFERRED_MODE = str(preferred_mode_control.value) if len(PREFERRED_LOCATIONS) else "none"
            if len(PREFERRED_LOCATIONS) > cfg.N_SAMPLES:
                raise ValueError("Existing/forced locations cannot exceed the requested sample count.")
            validate_config(cfg)
            WORKFLOW_CONFIG_APPLIED = True
            display(
                pd.DataFrame(
                    {
                        "setting": ["input", "range-filter applications", "current working rows", "ID", "X", "Y", "features", "transforms", "design PCs", "samples", "outlier coverage", "design coverage", "candidates/target", "PC tolerance", "max starts", "minimum starts", "no-improvement patience", "early stop", "spacing diagnostic", "existing-location mode"],
                        "value": [SURVEY_INPUT_LABEL, len(RANGE_FILTER_AUDIT), len(SURVEY_DF), cfg.ID_COLUMN, cfg.X_COLUMN, cfg.Y_COLUMN, cfg.FEATURE_COLUMNS, cfg.FEATURE_TRANSFORMS, cfg.N_DESIGN_COMPONENTS, cfg.N_SAMPLES, cfg.OUTLIER_COVERAGE, cfg.DESIGN_COVERAGE, cfg.CANDIDATES_PER_TARGET, cfg.PC_CANDIDATE_TOLERANCE, cfg.N_OPTIMIZER_STARTS, cfg.MIN_OPTIMIZER_STARTS, cfg.OPTIMIZER_NO_IMPROVEMENT_PATIENCE, cfg.EARLY_STOP_ON_STABLE_SEARCH, cfg.RUN_SPACING_DIAGNOSTIC, PREFERRED_MODE],
                    }
                )
            )
            if len(PREFERRED_LOCATIONS):
                display(PREFERRED_LOCATIONS)
            print("WORKFLOW READY. Continue to 'Run the scalable RSSD analysis' below.")
        except Exception as exc:
            WORKFLOW_CONFIG_APPLIED = False
            print(f"Finalization error: {type(exc).__name__}: {exc}")


apply_optimizer_button.on_click(_finalize_workflow)
display(
    widgets.VBox(
        [
            widgets.HTML("<b>Spatial support sites</b><br><span>What this controls: reserves a small portion of the sample budget for calibration locations selected specifically to reduce geographic coverage gaps after the response-surface regression design is built.<br>Recommended default: legacy-inspired automatic allocation.<br>When to change it: use manual or none primarily for validation against classical designs or when every sample must be tied to a theoretical response-surface target.</span>"),
            responsive_widget_grid([support_mode_control, manual_support_control], min_width=260),
            widgets.HTML("<b>Candidate exploration</b><br><span>What this controls: how many geographically distinct survey observations are made available as alternatives for each PCA response-surface condition.<br>Recommended default: automatic candidate-space saturation.<br>When to change it: use fixed mode for methodological comparisons or exact reruns with a fixed candidate pool.</span>"),
            responsive_widget_grid([candidate_mode_control, candidates_control, tolerance_control, max_tolerance_control], min_width=230),
            widgets.HTML("<b>SBAD accuracy</b><br><span>What this controls: the geographic support resolution used to measure field-wide coverage independently of raw sensor observation density.<br>Recommended default: automatic decision-stable support resolution.<br>When to change it: use fixed support size only for benchmarks or methodological experiments.</span>"),
            responsive_widget_grid([sbad_mode_control, coverage_envelope_control], min_width=260),
            widgets.HTML("<b>Optimizer starts</b><br><span>What this controls: how many distinct valid starting designs are explored before the result is judged stable to initialization.<br>Recommended default: automatic.</span>"),
            responsive_widget_grid([optimizer_mode_control, starts_control, min_starts_control, patience_control], min_width=230),
            widgets.HTML("<b>Proxy spatial scale</b><br><span>What this controls: a pre-calibration diagnostic describing the distance over which standardized sensor predictor patterns appear spatially structured.<br>Recommended default: estimate automatically. This is not the future regression residual correlation range.</span>"),
            responsive_widget_grid([early_stop_control, verify_rerun_control, spacing_diagnostic_control], min_width=230),
            responsive_widget_grid([memory_control, approximate_control], min_width=230),
            preferred_mode_control,
            responsive_widget_grid([candidate_color_control, selected_color_control]),
            widgets.HTML("<i>Primary objective: maintain near-optimal SBAD field coverage, then maximize exact geoMSD inside that coverage envelope.</i>"),
            apply_optimizer_button,
            optimizer_output,
        ]
    )
)


## 2. Run ESAP RSSD v2.9
Run the single analysis cell below after the guided workflow reports **WORKFLOW READY**. Internal numerical stages are wrapped by `run_esap_rssd(...)`.


In [ ]:
# @title Run ESAP RSSD analysis { display-mode: "form" }
workflow_state = {}
if globals().get("WORKFLOW_CONFIG_APPLIED", False):
    source_df_for_run = SURVEY_DF.copy()
    workflow_state = {
        "input_label": str(SURVEY_INPUT_LABEL),
        "preferred_locations": globals().get("PREFERRED_LOCATIONS", pd.DataFrame(columns=["preferred_location_id", "x", "y"])),
        "preferred_mode": globals().get("PREFERRED_MODE", "none"),
        "coordinate_epsg": globals().get("SOURCE_PROJECTED_EPSG", globals().get("COORDINATE_EPSG", None)),
    }
    print("Using the dataframe and settings finalized in the sequential guided workflow.")
else:
    source_df_for_run = None
    print("Workflow was not finalized; using the configuration/default synthetic input.")

rssd_result = run_esap_rssd(cfg, source_df_for_run, workflow_state)
selected_export = rssd_result.selected_sites
candidate_export = rssd_result.candidate_sites
target_instances = rssd_result.target_instances
support_sequence = rssd_result.support_sequence
ad_support_resolution = rssd_result.ad_support_resolution
candidate_saturation = rssd_result.candidate_saturation
optimizer_stability = rssd_result.optimizer_stability
spatial_support_sites = rssd_result.spatial_support_sites
field_coverage_distances = rssd_result.field_coverage_distances
proxy_spatial_scale = rssd_result.proxy_spatial_scale
pca_validation = rssd_result.pca_validation
skewness_summary = rssd_result.feature_skewness
metadata = rssd_result.metadata
run_summary = rssd_result.run_summary

display(selected_export)
display(ad_support_resolution)
display(candidate_saturation)
if len(optimizer_stability):
    display(optimizer_stability.groupby(["support_size", "objective_type"], as_index=False).agg(starts=("start_number", "max"), best_SBAD=("best_so_far_SBAD", "last"), best_geoMSD=("best_so_far_geoMSD", "last")))

create_esap_rssd_figures(rssd_result, show=True)



## Why Moran's I Is Not Used Here

Lesch (2005) uses the Moran residual statistic to assess spatial autocorrelation after a calibration response has been measured and a regression model has been fitted. At the RSSD site-selection stage, no target response or fitted calibration residuals exist. Standardized PCA scores are coded sensor covariates, not regression residuals. Consequently, Moran's I is neither calculated nor optimized here. It belongs in the later calibration and model-validation workflow, alongside other residual diagnostics.


## 3. Download a complete RSSD v2.9 run bundle
Create a timestamped ZIP containing the v2.9 selected sites, support sequence, SBAD support-resolution table, candidate-saturation table, optimizer-stability table, metadata, summary, and optional KMZ.


In [ ]:
# @title Create and download complete RSSD v2.9 run bundle { display-mode: "form" }
CREATE_RUN_BUNDLE = False  # @param {type:"boolean"}

if CREATE_RUN_BUNDLE:
    import re, shutil
    from datetime import datetime, timezone
    if "rssd_result" not in globals():
        raise RuntimeError("Run the ESAP RSSD analysis successfully before creating the bundle.")
    bundle_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    input_label = str(rssd_result.metadata.get("input_label", "rssd_input"))
    input_stem = re.sub(r"[^A-Za-z0-9._-]+", "_", Path(str(input_label).replace("uploaded:", "")).stem or "rssd_input")
    bundle_dir = Path(f"ESAP_RSSD_v2_9_{input_stem}_{bundle_timestamp}"); bundle_dir.mkdir(parents=False, exist_ok=False)
    figures_dir = bundle_dir / "figures"
    saved_figures = create_esap_rssd_figures(rssd_result, output_dir=figures_dir, show=False)
    for filename, table in {
        "ESAP_RSSD_selected_sites.csv": rssd_result.selected_sites,
        "ESAP_RSSD_candidate_sites.csv": rssd_result.candidate_sites,
        "ESAP_RSSD_target_instances.csv": rssd_result.target_instances,
        "ESAP_RSSD_spatial_support_sequence.csv": rssd_result.support_sequence,
        "ESAP_RSSD_ad_support_resolution.csv": rssd_result.ad_support_resolution,
        "ESAP_RSSD_candidate_saturation.csv": rssd_result.candidate_saturation,
        "ESAP_RSSD_optimizer_stability.csv": rssd_result.optimizer_stability,
        "ESAP_RSSD_spatial_support_sites.csv": rssd_result.spatial_support_sites,
        "ESAP_RSSD_field_coverage_distances.csv": rssd_result.field_coverage_distances,
        "ESAP_RSSD_proxy_spatial_scale.csv": rssd_result.proxy_spatial_scale,
        "ESAP_RSSD_pca_standardization_validation.csv": rssd_result.pca_validation,
        "ESAP_RSSD_feature_skewness.csv": rssd_result.feature_skewness,
    }.items():
        table.to_csv(bundle_dir / filename, index=False)
    (bundle_dir / "ESAP_RSSD_run_metadata.json").write_text(json.dumps(json_ready(rssd_result.metadata), indent=2, allow_nan=False), encoding="utf-8")
    (bundle_dir / "ESAP_RSSD_run_summary.md").write_text(rssd_result.run_summary, encoding="utf-8")
    (bundle_dir / "ESAP_RSSD_configuration.json").write_text(json.dumps(json_ready(asdict(rssd_result.config)), indent=2, allow_nan=False), encoding="utf-8")
    kmz_meta = rssd_result.metadata.get("kmz_export", {})
    if kmz_meta.get("created") and Path(str(kmz_meta.get("file"))).exists():
        shutil.copy2(Path(str(kmz_meta["file"])), bundle_dir / "ESAP_RSSD_selected_sites.kmz")
    manifest = f"""# ESAP RSSD v2.9 complete run bundle\n\n- Notebook version: {NOTEBOOK_VERSION}\n- Bundle created (UTC): {bundle_timestamp}\n- Selected samples: {len(rssd_result.selected_sites):,}\n- Response-surface sites: {(rssd_result.selected_sites.sample_role == 'response_surface').sum():,}\n- Spatial support sites: {(rssd_result.selected_sites.sample_role == 'spatial_support').sum():,}\n- Final SBAD support size: {rssd_result.metadata.get('support_optimization', {}).get('final_support_size')}\n- Random seed: {rssd_result.config.RANDOM_SEED}\n- Figures: {len(saved_figures)} PNG files\n- Raw input dataset is not copied into this bundle.\n"""
    (bundle_dir / "README.md").write_text(manifest, encoding="utf-8")
    zip_path = Path(shutil.make_archive(str(bundle_dir), "zip", root_dir=bundle_dir))
    print(f"Complete run bundle created: {zip_path.resolve()}")
    try:
        from google.colab import files as colab_files  # type: ignore
        colab_files.download(str(zip_path))
    except ImportError:
        print("Outside Colab: use the ZIP path printed above.")
else:
    print("Run bundle not created. Check CREATE_RUN_BUNDLE and rerun this final cell after analysis.")

